# Machine Learning — 90-Day Repeat Purchase Propensity

## Notebook Objective

The preceding exploratory and diagnostic analysis identified repeat purchasing as one of the most important commercial opportunities in the transaction data.

The analysis showed that a substantial proportion of newly observed customers do not make a second purchase within the first 90 days, while repeat purchasing continues to develop materially over longer customer lifecycles.

This notebook converts that finding into a predictive machine-learning problem.

### Primary Business Question

> **At the moment a customer's first observable order is completed, what is the probability that the customer will make a second purchase within the next 90 days?**

The model is intended to support post-purchase customer management.

Rather than treating every first-time customer identically, the business could use predicted repeat probability to identify:

- customers likely to repeat naturally;
- customers who may require additional engagement;
- customers whose first-order characteristics resemble historically low-repeat customers;
- and customer groups where different retention strategies should be tested.

## Prediction Point

The prediction is made immediately after the customer's first observable completed order.

This means that every model feature must represent information that would genuinely have been available at that moment.

No subsequent customer activity may be used as an input feature.

## Target

The modelling target is:

**1 — the customer makes a second purchase within 90 days of the first observable order**

**0 — the customer does not make a second purchase within 90 days**

Customers without at least 90 days of future observation are excluded from the binary modelling population rather than incorrectly labelled as non-repeaters.

Customers whose first observed transaction occurs during the opening month of the dataset are also excluded because their earlier purchasing history may pre-date the available records.

## Modelling Strategy

The notebook will:

1. load and validate the cleaned canonical transaction dataset;
2. reconstruct a leakage-safe first-purchase modelling population;
3. engineer features observable at the first order;
4. create the 90-day repeat-purchase target;
5. inspect class balance and feature quality;
6. split the data chronologically into train, validation and test periods;
7. establish simple predictive baselines;
8. train an interpretable logistic-regression benchmark;
9. train non-linear tree-based models;
10. compare model discrimination, lift and probability calibration;
11. select the strongest model;
12. interpret the drivers of repeat probability;
13. translate model scores into commercially usable customer segments.

## Evaluation Philosophy

Model performance will not be judged using accuracy alone.

The principal evaluation measures will include:

- PR-AUC;
- ROC-AUC;
- precision and recall;
- lift within the highest-scored customer groups;
- Brier score;
- and probability calibration.

The final question is not simply whether the model predicts correctly.

It is whether the model can rank customers sufficiently well to support better post-purchase decisions.

## Important Methodological Principle

All validation will be temporal.

Earlier customers will be used to train the model and later customers will be held out for validation and testing.

This mirrors the real production problem:

> **learn from historical customers and predict future customers.**


## STEP 24 — MODEL 1 FEATURE ENGINEERING & TEMPORAL DATA SPLIT

In [ ]:
# ================================================================
# 24.1 — IMPORT LIBRARIES
# ================================================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path


pd.set_option(
    "display.max_columns",
    100
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.4f}"
)

In [ ]:
# ================================================================
# 24.2 — LOAD CLEANED CANONICAL DATA
# ================================================================

DATA_PATH = Path(
    "cleaned_online_retail.parquet"
)


if not DATA_PATH.exists():

    raise FileNotFoundError(
        "Cleaned dataset not found. "
        "Export the final canonical df from Notebook 1 "
        "as cleaned_online_retail.parquet."
    )


df = pd.read_parquet(
    DATA_PATH
)


print(
    f"Rows loaded: {len(df):,}"
)

print(
    f"Columns loaded: {df.shape[1]:,}"
)

print(
    f"Observed dates: "
    f"{df['date'].min()} to {df['date'].max()}"
)

In [ ]:
# ================================================================
# 24.3 — VALIDATE REQUIRED COLUMNS
# ================================================================

required_columns = [
    "date",
    "invoice",
    "customer",
    "product",
    "product_description",
    "region",
    "quantity",
    "unit_price",
    "customer_is_identified",
    "is_merchandise",
    "is_cancellation",
    "net_merchandise_revenue",
]


missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]


if missing_columns:

    raise KeyError(
        "Required modelling columns are missing: "
        f"{missing_columns}"
    )


print(
    "✓ Required canonical modelling fields are available."
)

In [ ]:
# ================================================================
# 24.4 — STANDARDISE MODELLING DATATYPES
# ================================================================

df[
    "date"
] = pd.to_datetime(
    df[
        "date"
    ],
    errors="coerce"
)


df[
    "customer"
] = (
    df[
        "customer"
    ]
    .astype("string")
)


df[
    "invoice"
] = (
    df[
        "invoice"
    ]
    .astype("string")
)


df[
    "product"
] = (
    df[
        "product"
    ]
    .astype("string")
)


df[
    "region"
] = (
    df[
        "region"
    ]
    .astype("string")
)


assert df["date"].notna().all()


print(
    "✓ Modelling datatypes standardised."
)

In [ ]:
# ================================================================
# 24.5 — MODELLING OBSERVATION BOUNDARY
# ================================================================

modeling_end_date = pd.Timestamp(
    "2011-11-30 23:59:59"
)


modeling_data = (
    df.loc[
        df[
            "date"
        ]
        <= modeling_end_date
    ]
    .copy()
)


print(
    f"Modelling observation end: "
    f"{modeling_end_date.date()}"
)

print(
    f"Rows retained: "
    f"{len(modeling_data):,}"
)

In [ ]:
# ================================================================
# 24.6 — ELIGIBLE CUSTOMER PURCHASE TRANSACTIONS
# ================================================================

positive_customer_purchases = (
    modeling_data.loc[
        modeling_data[
            "customer_is_identified"
        ]
        &
        modeling_data[
            "is_merchandise"
        ]
        &
        modeling_data[
            "quantity"
        ].gt(0)
        &
        modeling_data[
            "unit_price"
        ].gt(0)
        &
        ~modeling_data[
            "is_cancellation"
        ]
    ]
    .copy()
)


print(
    f"Positive identified merchandise rows: "
    f"{len(positive_customer_purchases):,}"
)

print(
    f"Identified purchasing customers: "
    f"{positive_customer_purchases['customer'].nunique():,}"
)

In [ ]:
# ================================================================
# 24.7 — VALIDATE INVOICE → CUSTOMER RELATIONSHIP
# ================================================================

invoice_customer_counts = (
    positive_customer_purchases
    .groupby(
        "invoice",
        dropna=False
    )[
        "customer"
    ]
    .nunique()
)


multi_customer_invoices = (
    invoice_customer_counts
    .gt(1)
    .sum()
)


print(
    f"Invoices linked to multiple customers: "
    f"{multi_customer_invoices:,}"
)

In [ ]:
# ================================================================
# 24.8 — BUILD CUSTOMER ORDER HISTORY
# ================================================================

customer_orders = (
    positive_customer_purchases
    .groupby(
        [
            "customer",
            "invoice"
        ],
        as_index=False
    )
    .agg(
        order_date=(
            "date",
            "min"
        ),

        order_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),

        order_units=(
            "quantity",
            "sum"
        ),

        distinct_products=(
            "product",
            "nunique"
        ),

        average_unit_price=(
            "unit_price",
            "mean"
        ),

        region=(
            "region",
            "first"
        ),
    )
)


customer_orders = (
    customer_orders
    .sort_values(
        [
            "customer",
            "order_date",
            "invoice"
        ]
    )
    .reset_index(
        drop=True
    )
)


customer_orders[
    "order_number"
] = (
    customer_orders
    .groupby(
        "customer"
    )
    .cumcount()
    + 1
)


print(
    f"Orders created: "
    f"{len(customer_orders):,}"
)

display(
    customer_orders.head()
)

In [ ]:
# ================================================================
# 24.9 — FIRST OBSERVABLE ORDER
# ================================================================

first_orders = (
    customer_orders.loc[
        customer_orders[
            "order_number"
        ] == 1
    ]
    .copy()
)


first_orders = (
    first_orders
    .rename(
        columns={
            "invoice":
                "first_invoice",

            "order_date":
                "first_order_date",

            "order_revenue":
                "first_order_value",

            "order_units":
                "first_order_units",

            "distinct_products":
                "first_order_product_count",

            "average_unit_price":
                "first_order_average_unit_price",

            "region":
                "first_order_region",
        }
    )
)


print(
    f"Customers with first observable order: "
    f"{len(first_orders):,}"
)

In [ ]:
# ================================================================
# 24.10 — SECOND OBSERVABLE ORDER
# ================================================================

second_orders = (
    customer_orders.loc[
        customer_orders[
            "order_number"
        ] == 2
    ][
        [
            "customer",
            "order_date"
        ]
    ]
    .rename(
        columns={
            "order_date":
                "second_order_date"
        }
    )
)


first_orders = (
    first_orders
    .merge(
        second_orders,
        on="customer",
        how="left"
    )
)


first_orders[
    "days_to_second_purchase"
] = (
    first_orders[
        "second_order_date"
    ]
    -
    first_orders[
        "first_order_date"
    ]
).dt.days

In [ ]:
# ================================================================
# 24.11 — REMOVE LEFT-TRUNCATED CUSTOMERS
# ================================================================

dataset_start_month = (
    modeling_data[
        "date"
    ]
    .min()
    .to_period("M")
)


first_orders[
    "first_order_month"
] = (
    first_orders[
        "first_order_date"
    ]
    .dt.to_period("M")
)


first_orders[
    "is_left_truncated"
] = (
    first_orders[
        "first_order_month"
    ]
    ==
    dataset_start_month
)


print(
    f"Opening-month first observations: "
    f"{first_orders['is_left_truncated'].sum():,}"
)


first_orders = (
    first_orders.loc[
        ~first_orders[
            "is_left_truncated"
        ]
    ]
    .copy()
)

In [ ]:
# ================================================================
# 24.12 — 90-DAY FOLLOW-UP ELIGIBILITY
# ================================================================

first_orders[
    "followup_days_available"
] = (
    modeling_end_date.normalize()
    -
    first_orders[
        "first_order_date"
    ]
    .dt.normalize()
).dt.days


first_orders[
    "eligible_90d"
] = (
    first_orders[
        "followup_days_available"
    ]
    >= 90
)


eligible_first_orders = (
    first_orders.loc[
        first_orders[
            "eligible_90d"
        ]
    ]
    .copy()
)


print(
    f"Eligible customers: "
    f"{len(eligible_first_orders):,}"
)

print(
    f"Excluded due to <90 days follow-up: "
    f"{(~first_orders['eligible_90d']).sum():,}"
)

In [ ]:
# ================================================================
# 24.13 — CREATE 90-DAY REPEAT TARGET
# ================================================================

eligible_first_orders[
    "repeat_within_90d"
] = (
    eligible_first_orders[
        "days_to_second_purchase"
    ]
    .between(
        0,
        90,
        inclusive="both"
    )
    .fillna(False)
    .astype(int)
)


target_summary = (
    eligible_first_orders[
        "repeat_within_90d"
    ]
    .value_counts()
    .rename_axis(
        "repeat_within_90d"
    )
    .reset_index(
        name="customers"
    )
)


target_summary[
    "share_pct"
] = (
    target_summary[
        "customers"
    ]
    /
    target_summary[
        "customers"
    ].sum()
    * 100
)


display(
    target_summary
)

In [ ]:
# ================================================================
# 24.14 — FIRST-ORDER TEMPORAL FEATURES
# ================================================================

eligible_first_orders[
    "first_order_month_number"
] = (
    eligible_first_orders[
        "first_order_date"
    ]
    .dt.month
)


eligible_first_orders[
    "first_order_weekday"
] = (
    eligible_first_orders[
        "first_order_date"
    ]
    .dt.dayofweek
)


eligible_first_orders[
    "first_order_hour"
] = (
    eligible_first_orders[
        "first_order_date"
    ]
    .dt.hour
)


eligible_first_orders[
    "first_order_is_weekend"
] = (
    eligible_first_orders[
        "first_order_weekday"
    ]
    .isin(
        [5, 6]
    )
    .astype(int)
)

In [ ]:
# ================================================================
# 24.15 — FIRST-ORDER BASKET FEATURES
# ================================================================

first_invoice_lookup = (
    eligible_first_orders[
        [
            "customer",
            "first_invoice"
        ]
    ]
)


first_order_lines = (
    positive_customer_purchases
    .merge(
        first_invoice_lookup,
        left_on=[
            "customer",
            "invoice"
        ],
        right_on=[
            "customer",
            "first_invoice"
        ],
        how="inner"
    )
)


first_order_extra_features = (
    first_order_lines
    .groupby(
        "customer",
        as_index=False
    )
    .agg(
        first_order_line_count=(
            "product",
            "size"
        ),

        first_order_max_unit_price=(
            "unit_price",
            "max"
        ),

        first_order_min_unit_price=(
            "unit_price",
            "min"
        ),

        first_order_price_std=(
            "unit_price",
            "std"
        ),

        first_order_max_line_quantity=(
            "quantity",
            "max"
        ),

        first_order_mean_line_quantity=(
            "quantity",
            "mean"
        ),
    )
)


eligible_first_orders = (
    eligible_first_orders
    .merge(
        first_order_extra_features,
        on="customer",
        how="left"
    )
)

In [ ]:
# ================================================================
# 24.16 — BASKET COMPOSITION FEATURES
# ================================================================

eligible_first_orders[
    "units_per_product"
] = (
    eligible_first_orders[
        "first_order_units"
    ]
    /
    eligible_first_orders[
        "first_order_product_count"
    ]
)


eligible_first_orders[
    "revenue_per_product"
] = (
    eligible_first_orders[
        "first_order_value"
    ]
    /
    eligible_first_orders[
        "first_order_product_count"
    ]
)


eligible_first_orders[
    "revenue_per_unit"
] = (
    eligible_first_orders[
        "first_order_value"
    ]
    /
    eligible_first_orders[
        "first_order_units"
    ]
)

In [ ]:
# ================================================================
# 24.17 — INITIAL FEATURE REGISTER
# ================================================================

numeric_features = [
    "first_order_value",
    "first_order_units",
    "first_order_product_count",
    "first_order_average_unit_price",
    "first_order_line_count",
    "first_order_max_unit_price",
    "first_order_min_unit_price",
    "first_order_price_std",
    "first_order_max_line_quantity",
    "first_order_mean_line_quantity",
    "units_per_product",
    "revenue_per_product",
    "revenue_per_unit",
    "first_order_month_number",
    "first_order_weekday",
    "first_order_hour",
    "first_order_is_weekend",
]


categorical_features = [
    "first_order_region",
]


target_column = (
    "repeat_within_90d"
)


model_features = (
    numeric_features
    +
    categorical_features
)


print(
    f"Numeric features: "
    f"{len(numeric_features)}"
)

print(
    f"Categorical features: "
    f"{len(categorical_features)}"
)

print(
    f"Total predictors: "
    f"{len(model_features)}"
)

In [ ]:
# ================================================================
# 24.18 — FINAL MODEL 1 ANALYTICAL TABLE
# ================================================================

model_dataset = (
    eligible_first_orders[
        [
            "customer",
            "first_order_date",
            *model_features,
            target_column,
        ]
    ]
    .copy()
)


model_dataset = (
    model_dataset
    .sort_values(
        "first_order_date"
    )
    .reset_index(
        drop=True
    )
)


print(
    f"Model rows: "
    f"{len(model_dataset):,}"
)

print(
    f"Predictors: "
    f"{len(model_features):,}"
)


display(
    model_dataset.head()
)

In [ ]:
# ================================================================
# 24.19 — FEATURE MISSINGNESS
# ================================================================

feature_missingness = (
    model_dataset[
        model_features
    ]
    .isna()
    .mean()
    .mul(100)
    .sort_values(
        ascending=False
    )
    .rename(
        "missing_pct"
    )
    .reset_index()
    .rename(
        columns={
            "index":
                "feature"
        }
    )
)


display(
    feature_missingness
)

In [ ]:
# ================================================================
# 24.20 — MODEL POPULATION BY ACQUISITION MONTH
# ================================================================

model_dataset[
    "acquisition_month"
] = (
    model_dataset[
        "first_order_date"
    ]
    .dt.to_period("M")
)


monthly_model_population = (
    model_dataset
    .groupby(
        "acquisition_month",
        as_index=False
    )
    .agg(
        customers=(
            "customer",
            "size"
        ),

        repeat_rate=(
            target_column,
            "mean"
        ),
    )
)


monthly_model_population[
    "repeat_rate_pct"
] = (
    monthly_model_population[
        "repeat_rate"
    ]
    * 100
)


display(
    monthly_model_population
)

In [ ]:
# ================================================================
# VISUAL 24.1 — ELIGIBLE CUSTOMERS BY ACQUISITION MONTH
# ================================================================

monthly_model_population[
    "month_label"
] = (
    monthly_model_population[
        "acquisition_month"
    ]
    .astype(str)
)


plt.figure(
    figsize=(13, 6)
)


sns.barplot(
    data=monthly_model_population,
    x="month_label",
    y="customers"
)


plt.title(
    "Model-Eligible Customers by First Purchase Month"
)

plt.xlabel(
    "First Purchase Month"
)

plt.ylabel(
    "Customers"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 24.2 — 90-DAY REPEAT RATE OVER TIME
# ================================================================

plt.figure(
    figsize=(13, 6)
)


sns.lineplot(
    data=monthly_model_population,
    x="month_label",
    y="repeat_rate_pct",
    marker="o"
)


plt.title(
    "90-Day Repeat Rate by Acquisition Month"
)

plt.xlabel(
    "First Purchase Month"
)

plt.ylabel(
    "Repeat Within 90 Days (%)"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 24.21 — DETERMINE TEMPORAL SPLIT MONTHS
# ================================================================

monthly_split_table = (
    monthly_model_population[
        [
            "acquisition_month",
            "customers"
        ]
    ]
    .copy()
)


monthly_split_table[
    "cumulative_customers"
] = (
    monthly_split_table[
        "customers"
    ]
    .cumsum()
)


total_customers = (
    monthly_split_table[
        "customers"
    ]
    .sum()
)


monthly_split_table[
    "cumulative_share"
] = (
    monthly_split_table[
        "cumulative_customers"
    ]
    /
    total_customers
)


display(
    monthly_split_table
)

In [ ]:
# ================================================================
# 24.22 — TEMPORAL SPLIT BOUNDARIES
# ================================================================

validation_start_month = (
    monthly_split_table.loc[
        monthly_split_table[
            "cumulative_share"
        ] >= 0.70,
        "acquisition_month"
    ]
    .iloc[0]
)


test_start_month = (
    monthly_split_table.loc[
        monthly_split_table[
            "cumulative_share"
        ] >= 0.85,
        "acquisition_month"
    ]
    .iloc[0]
)


print(
    f"Validation starts: "
    f"{validation_start_month}"
)

print(
    f"Test starts: "
    f"{test_start_month}"
)

In [ ]:
# ================================================================
# 24.23 — ASSIGN TRAIN / VALIDATION / TEST
# ================================================================

model_dataset[
    "dataset_split"
] = np.select(
    [
        model_dataset[
            "acquisition_month"
        ]
        <
        validation_start_month,

        (
            model_dataset[
                "acquisition_month"
            ]
            >=
            validation_start_month
        )
        &
        (
            model_dataset[
                "acquisition_month"
            ]
            <
            test_start_month
        ),
    ],

    [
        "Train",
        "Validation",
    ],

    default="Test"
)

In [ ]:
# ================================================================
# 24.24 — TEMPORAL SPLIT SUMMARY
# ================================================================

split_summary = (
    model_dataset
    .groupby(
        "dataset_split",
        as_index=False
    )
    .agg(
        customers=(
            "customer",
            "size"
        ),

        first_date=(
            "first_order_date",
            "min"
        ),

        last_date=(
            "first_order_date",
            "max"
        ),

        repeat_rate=(
            target_column,
            "mean"
        ),
    )
)


split_summary[
    "repeat_rate_pct"
] = (
    split_summary[
        "repeat_rate"
    ]
    * 100
)


split_order = pd.CategoricalDtype(
    [
        "Train",
        "Validation",
        "Test"
    ],
    ordered=True
)


split_summary[
    "dataset_split"
] = (
    split_summary[
        "dataset_split"
    ]
    .astype(
        split_order
    )
)


split_summary = (
    split_summary
    .sort_values(
        "dataset_split"
    )
)


display(
    split_summary
)

In [ ]:
# ================================================================
# 24.25 — CREATE TRAIN / VALIDATION / TEST MATRICES
# ================================================================

train_data = (
    model_dataset.loc[
        model_dataset[
            "dataset_split"
        ] == "Train"
    ]
    .copy()
)


validation_data = (
    model_dataset.loc[
        model_dataset[
            "dataset_split"
        ] == "Validation"
    ]
    .copy()
)


test_data = (
    model_dataset.loc[
        model_dataset[
            "dataset_split"
        ] == "Test"
    ]
    .copy()
)


X_train = (
    train_data[
        model_features
    ]
    .copy()
)

y_train = (
    train_data[
        target_column
    ]
    .copy()
)


X_validation = (
    validation_data[
        model_features
    ]
    .copy()
)

y_validation = (
    validation_data[
        target_column
    ]
    .copy()
)


X_test = (
    test_data[
        model_features
    ]
    .copy()
)

y_test = (
    test_data[
        target_column
    ]
    .copy()
)

In [ ]:
# ================================================================
# 24.26 — VALIDATE TEMPORAL SEPARATION
# ================================================================

assert (
    train_data[
        "first_order_date"
    ].max()
    <
    validation_data[
        "first_order_date"
    ].min()
), (
    "Training and validation periods overlap."
)


assert (
    validation_data[
        "first_order_date"
    ].max()
    <
    test_data[
        "first_order_date"
    ].min()
), (
    "Validation and test periods overlap."
)


assert (
    len(
        train_data
    )
    +
    len(
        validation_data
    )
    +
    len(
        test_data
    )
    ==
    len(
        model_dataset
    )
)


print(
    "✓ Temporal split validated successfully."
)

In [ ]:
# ================================================================
# 24.27 — TARGET PREVALENCE BY SPLIT
# ================================================================

target_prevalence = pd.DataFrame(
    {
        "Dataset": [
            "Train",
            "Validation",
            "Test"
        ],

        "Customers": [
            len(y_train),
            len(y_validation),
            len(y_test),
        ],

        "Repeaters": [
            int(y_train.sum()),
            int(y_validation.sum()),
            int(y_test.sum()),
        ],

        "Repeat Rate (%)": [
            y_train.mean() * 100,
            y_validation.mean() * 100,
            y_test.mean() * 100,
        ],
    }
)


display(
    target_prevalence
)

In [ ]:
# ================================================================
# VISUAL 24.3 — TARGET RATE BY TEMPORAL SPLIT
# ================================================================

plt.figure(
    figsize=(8, 5)
)


sns.barplot(
    data=target_prevalence,
    x="Dataset",
    y="Repeat Rate (%)"
)


plt.title(
    "90-Day Repeat Rate Across Temporal Splits"
)

plt.xlabel(
    ""
)

plt.ylabel(
    "Repeat Within 90 Days (%)"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 24.28 — TARGET LEAKAGE AUDIT
# ================================================================

forbidden_features = {
    "second_order_date",
    "days_to_second_purchase",
    "followup_days_available",
    "eligible_90d",
    "repeat_within_90d",
    "future_revenue",
    "future_order_count",
    "last_purchase_date",
}


leaked_features = (
    set(
        model_features
    )
    &
    forbidden_features
)


assert not leaked_features, (
    f"Target leakage detected: "
    f"{leaked_features}"
)


print(
    "✓ No explicitly forbidden future features "
    "are present in the predictor set."
)

In [ ]:
# ================================================================
# 24.29 — FINAL STEP 24 VALIDATION
# ================================================================

assert (
    model_dataset[
        "customer"
    ]
    .nunique()
    ==
    len(
        model_dataset
    )
)


assert (
    model_dataset[
        target_column
    ]
    .isin(
        [
            0,
            1
        ]
    )
    .all()
)


assert (
    (
        modeling_end_date.normalize()
        -
        model_dataset[
            "first_order_date"
        ]
        .dt.normalize()
    )
    .dt.days
    .ge(90)
    .all()
)


assert len(train_data) > 0
assert len(validation_data) > 0
assert len(test_data) > 0


print(
    "✓ Step 24 complete."
)

print(
    "✓ Model 1 population is leakage-safe."
)

print(
    "✓ 90-day outcomes are fully observable."
)

print(
    "✓ Train, validation and test sets are "
    "chronologically separated."
)

### Step 24 Summary — Feature Engineering and Temporal Validation Architecture

Step 24 created the analytical dataset for the 90-day repeat-purchase propensity model.

The modelling population contains one row per observable newly acquired identified customer.

Customers first observed during the opening month of the dataset were excluded because their true purchase history may pre-date the available records.

Customers without at least 90 days of future observation were also excluded rather than incorrectly treated as non-repeaters.

The resulting target therefore distinguishes between customers who genuinely did and did not make a second purchase within a fully observable 90-day period.

### Leakage-Safe Features

All predictors represent information available immediately after the first observable order.

The initial feature set captures:

- first-order monetary value;
- units purchased;
- number of distinct products;
- number of invoice lines;
- average, minimum and maximum unit price;
- variation in item prices;
- quantity characteristics;
- revenue and quantity per product;
- order month;
- weekday;
- time of day;
- weekend behaviour;
- and first-order geographic market.

No information from the customer's second or subsequent purchase is included as a predictor.

### Temporal Validation

The dataset was divided chronologically rather than randomly.

Earlier customer acquisitions form the training population, subsequent acquisitions form the validation population, and the most recent eligible acquisitions form the final test population.

This design more closely represents how the model would operate in practice: historical customers are used to learn patterns that are then applied to future customers.

The test set will remain untouched during model development and model selection.

### Modelling Objective

The next stage will establish predictive benchmarks before introducing more complex algorithms.

The modelling process will begin with simple baselines and an interpretable logistic-regression model.

More complex models will only be retained if they demonstrate meaningful improvement in ranking, lift and probability quality on future customers.

The central modelling question remains:

> **Can information available from a customer's first order meaningfully predict whether that customer will purchase again within 90 days?**

## Step 25 — Preprocessing, Baselines and Logistic Regression

Step 25 begins the predictive modelling phase for the 90-day repeat-purchase problem.

The objective is to determine whether information available immediately after a customer's first observable order contains useful predictive signal about whether that customer will make a second purchase within 90 days.

The modelling process begins with simple benchmarks before introducing more complex algorithms.

Three levels of prediction will be evaluated:

1. a naive prevalence baseline;
2. a simple first-order behavioural benchmark; and
3. a multivariable logistic-regression model.

All preprocessing will be learned from the training data only.

The validation set will be used for model comparison and diagnostic analysis, while the final temporal test set will remain untouched until model selection is complete.

The principal evaluation criteria are:

- PR-AUC;
- ROC-AUC;
- Brier score;
- precision;
- recall;
- F1 score;
- lift among the highest-scored customers;
- and probability calibration.

The purpose is not merely to maximise statistical accuracy, but to establish whether first-order behaviour can rank future repeat customers in a commercially useful way.

In [ ]:
# ================================================================
# 25.2 — IMPORT MODELLING LIBRARIES
# ================================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve,
)

from sklearn.calibration import calibration_curve

In [ ]:
# ================================================================
# 25.3 — CONFIRM DATASET SPLIT INTEGRITY
# ================================================================

print("TRAIN")
print(
    f"Rows: {len(X_train):,}"
)

print(
    f"Repeat rate: "
    f"{y_train.mean() * 100:.2f}%"
)


print("\nVALIDATION")
print(
    f"Rows: {len(X_validation):,}"
)

print(
    f"Repeat rate: "
    f"{y_validation.mean() * 100:.2f}%"
)


print("\nTEST — RESERVED")
print(
    f"Rows: {len(X_test):,}"
)

print(
    f"Repeat rate: "
    f"{y_test.mean() * 100:.2f}%"
)

In [ ]:
assert len(X_train) == len(y_train)
assert len(X_validation) == len(y_validation)
assert len(X_test) == len(y_test)

assert len(X_train) > 0
assert len(X_validation) > 0
assert len(X_test) > 0

print(
    "\n✓ Train, validation and test datasets are available."
)

print(
    "✓ Test dataset will remain untouched during model development."
)

In [ ]:
# ================================================================
# 25.4 — NUMERIC FEATURE SUMMARY
# ================================================================

numeric_feature_summary = (
    X_train[
        numeric_features
    ]
    .describe()
    .T
)


display(
    numeric_feature_summary
)

In [ ]:
# ================================================================
# 25.5 — CATEGORICAL FEATURE SUMMARY
# ================================================================

for feature in categorical_features:

    print(
        f"\n{feature}"
    )

    display(
        X_train[
            feature
        ]
        .value_counts(
            dropna=False
        )
        .head(20)
        .to_frame(
            "customers"
        )
    )

In [ ]:
# ================================================================
# 25.6 — PREPROCESSING PIPELINE
# ================================================================

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "scaler",
            StandardScaler()
        ),
    ]
)


categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        ),
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features,
        ),

        (
            "categorical",
            categorical_transformer,
            categorical_features,
        ),
    ]
)


print(
    "✓ Preprocessing pipeline defined."
)

In [ ]:
# ================================================================
# 25.7 — MODEL EVALUATION FUNCTIONS
# ================================================================

def lift_at_fraction(
    y_true,
    y_probability,
    fraction=0.10
):
    """
    Calculate lift among the highest-scored fraction.

    If all model probabilities are identical, the model
    has no ranking ability and theoretical lift = 1.0.
    """

    y_true = np.asarray(y_true)
    y_probability = np.asarray(y_probability)


    # Constant-score model has no ranking ability
    if np.allclose(
        y_probability,
        y_probability[0]
    ):
        return 1.0


    evaluation = pd.DataFrame(
        {
            "actual": y_true,
            "probability": y_probability,
        }
    )


    evaluation = (
        evaluation
        .sort_values(
            "probability",
            ascending=False
        )
        .reset_index(drop=True)
    )


    n_top = max(
        1,
        int(
            np.ceil(
                len(evaluation)
                * fraction
            )
        )
    )


    overall_rate = (
        evaluation[
            "actual"
        ]
        .mean()
    )


    top_rate = (
        evaluation
        .head(n_top)[
            "actual"
        ]
        .mean()
    )


    if overall_rate == 0:
        return np.nan


    return (
        top_rate
        / overall_rate
    )

In [ ]:
def capture_at_fraction(
    y_true,
    y_probability,
    fraction=0.10
):
    """
    Share of actual positive outcomes captured within
    the highest-scored customer fraction.

    Constant scores have no ranking ability, so expected
    capture equals the selected population fraction.
    """

    y_true = np.asarray(y_true)
    y_probability = np.asarray(y_probability)


    if np.allclose(
        y_probability,
        y_probability[0]
    ):
        return fraction


    evaluation = pd.DataFrame(
        {
            "actual": y_true,
            "probability": y_probability,
        }
    )


    evaluation = (
        evaluation
        .sort_values(
            "probability",
            ascending=False
        )
        .reset_index(drop=True)
    )


    n_top = max(
        1,
        int(
            np.ceil(
                len(evaluation)
                * fraction
            )
        )
    )


    total_positives = (
        evaluation[
            "actual"
        ]
        .sum()
    )


    if total_positives == 0:
        return np.nan


    captured_positives = (
        evaluation
        .head(n_top)[
            "actual"
        ]
        .sum()
    )


    return (
        captured_positives
        / total_positives
    )

In [ ]:
def evaluate_binary_model(
    model_name,
    y_true,
    probabilities,
    threshold=0.50
):
    """
    Standard evaluation framework used across
    repeat-purchase classification models.
    """

    y_true = np.asarray(
        y_true
    )

    probabilities = np.asarray(
        probabilities
    )

    predictions = (
        probabilities
        >= threshold
    ).astype(int)


    results = {
        "Model":
            model_name,

        "PR-AUC":
            average_precision_score(
                y_true,
                probabilities
            ),

        "ROC-AUC":
            roc_auc_score(
                y_true,
                probabilities
            ),

        "Brier Score":
            brier_score_loss(
                y_true,
                probabilities
            ),

        "Precision":
            precision_score(
                y_true,
                predictions,
                zero_division=0
            ),

        "Recall":
            recall_score(
                y_true,
                predictions,
                zero_division=0
            ),

        "F1":
            f1_score(
                y_true,
                predictions,
                zero_division=0
            ),

        "Lift @ 10%":
            lift_at_fraction(
                y_true,
                probabilities,
                0.10
            ),

        "Lift @ 20%":
            lift_at_fraction(
                y_true,
                probabilities,
                0.20
            ),

        "Capture @ 10%":
            capture_at_fraction(
                y_true,
                probabilities,
                0.10
            ),

        "Capture @ 20%":
            capture_at_fraction(
                y_true,
                probabilities,
                0.20
            ),
    }


    return pd.DataFrame(
        [results]
    )

### 25.8 — Baseline 1: prevalence model

In [ ]:
# ================================================================
# 25.8 — BASELINE 1: TRAINING PREVALENCE
# ================================================================

training_repeat_rate = (
    y_train.mean()
)


validation_prevalence_probabilities = (
    np.full(
        shape=len(y_validation),
        fill_value=training_repeat_rate,
        dtype=float
    )
)


print(
    f"Training repeat rate: "
    f"{training_repeat_rate * 100:.2f}%"
)

In [ ]:
prevalence_results = (
    evaluate_binary_model(
        model_name="Training Prevalence Baseline",
        y_true=y_validation,
        probabilities=validation_prevalence_probabilities,
    )
)


display(
    prevalence_results
)

### 25.9 — Baseline 2: single-feature behavioural benchmark

In [ ]:
# ================================================================
# 25.9 — BASELINE 2: SINGLE-FEATURE LOGISTIC MODEL
# ================================================================

single_feature = [
    "first_order_product_count"
]


single_feature_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            ColumnTransformer(
                transformers=[
                    (
                        "numeric",
                        Pipeline(
                            steps=[
                                (
                                    "imputer",
                                    SimpleImputer(
                                        strategy="median"
                                    )
                                ),

                                (
                                    "scaler",
                                    StandardScaler()
                                ),
                            ]
                        ),
                        single_feature,
                    )
                ]
            )
        ),

        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        ),
    ]
)

In [ ]:
single_feature_pipeline.fit(
    X_train,
    y_train
)

In [ ]:
validation_single_feature_probabilities = (
    single_feature_pipeline
    .predict_proba(
        X_validation
    )[:, 1]
)

In [ ]:
single_feature_results = (
    evaluate_binary_model(
        model_name="Single-Feature Basket Breadth",
        y_true=y_validation,
        probabilities=validation_single_feature_probabilities,
    )
)


display(
    single_feature_results
)

### 25.10 — Train full logistic regression model

In [ ]:
# ================================================================
# 25.10 — FULL LOGISTIC REGRESSION PIPELINE
# ================================================================

logistic_model = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=3000,
                random_state=42
            )
        ),
    ]
)

In [ ]:
logistic_model.fit(
    X_train,
    y_train
)


print(
    "✓ Logistic regression trained."
)

In [ ]:
# ================================================================
# 25.11 — LOGISTIC VALIDATION PREDICTIONS
# ================================================================

validation_logistic_probabilities = (
    logistic_model
    .predict_proba(
        X_validation
    )[:, 1]
)


print(
    f"Minimum probability: "
    f"{validation_logistic_probabilities.min():.4f}"
)

print(
    f"Maximum probability: "
    f"{validation_logistic_probabilities.max():.4f}"
)

print(
    f"Mean probability: "
    f"{validation_logistic_probabilities.mean():.4f}"
)

In [ ]:
# ================================================================
# 25.12 — LOGISTIC REGRESSION PERFORMANCE
# ================================================================

logistic_results = (
    evaluate_binary_model(
        model_name="Logistic Regression",
        y_true=y_validation,
        probabilities=validation_logistic_probabilities,
    )
)


display(
    logistic_results
)

In [ ]:
# ================================================================
# 25.13 — BASELINE MODEL COMPARISON
# ================================================================

step25_model_comparison = (
    pd.concat(
        [
            prevalence_results,
            single_feature_results,
            logistic_results,
        ],
        ignore_index=True
    )
)


display(
    step25_model_comparison
)

In [ ]:
# ================================================================
# VISUAL 25.1 — PRECISION-RECALL CURVES
# ================================================================

precision_logistic, recall_logistic, _ = (
    precision_recall_curve(
        y_validation,
        validation_logistic_probabilities
    )
)


precision_single, recall_single, _ = (
    precision_recall_curve(
        y_validation,
        validation_single_feature_probabilities
    )
)


plt.figure(
    figsize=(9, 6)
)


plt.plot(
    recall_logistic,
    precision_logistic,
    label="Logistic Regression"
)


plt.plot(
    recall_single,
    precision_single,
    label="Single-Feature Baseline"
)


plt.axhline(
    y=y_validation.mean(),
    linestyle="--",
    label="Validation Prevalence"
)


plt.title(
    "Precision-Recall Curve — 90-Day Repeat Purchase"
)

plt.xlabel(
    "Recall"
)

plt.ylabel(
    "Precision"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 25.2 — ROC CURVE
# ================================================================

fpr_logistic, tpr_logistic, _ = (
    roc_curve(
        y_validation,
        validation_logistic_probabilities
    )
)


fpr_single, tpr_single, _ = (
    roc_curve(
        y_validation,
        validation_single_feature_probabilities
    )
)


plt.figure(
    figsize=(8, 6)
)


plt.plot(
    fpr_logistic,
    tpr_logistic,
    label="Logistic Regression"
)


plt.plot(
    fpr_single,
    tpr_single,
    label="Single-Feature Baseline"
)


plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Ranking"
)


plt.title(
    "ROC Curve — 90-Day Repeat Purchase"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 25.16 — VALIDATION DECILE ANALYSIS
# ================================================================

validation_scored = pd.DataFrame(
    {
        "actual":
            np.asarray(
                y_validation
            ),

        "repeat_probability":
            validation_logistic_probabilities,
    }
)


validation_scored = (
    validation_scored
    .sort_values(
        "repeat_probability",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


validation_scored[
    "score_decile"
] = (
    pd.qcut(
        validation_scored.index,
        q=10,
        labels=[
            1, 2, 3, 4, 5,
            6, 7, 8, 9, 10
        ]
    )
)

In [ ]:
decile_performance = (
    validation_scored
    .groupby(
        "score_decile",
        observed=True,
        as_index=False
    )
    .agg(
        customers=(
            "actual",
            "size"
        ),

        actual_repeat_rate=(
            "actual",
            "mean"
        ),

        average_predicted_probability=(
            "repeat_probability",
            "mean"
        ),

        repeaters=(
            "actual",
            "sum"
        ),
    )
)


decile_performance[
    "actual_repeat_rate_pct"
] = (
    decile_performance[
        "actual_repeat_rate"
    ]
    * 100
)


decile_performance[
    "average_predicted_probability_pct"
] = (
    decile_performance[
        "average_predicted_probability"
    ]
    * 100
)


portfolio_repeat_rate = (
    y_validation.mean()
)


decile_performance[
    "lift"
] = (
    decile_performance[
        "actual_repeat_rate"
    ]
    /
    portfolio_repeat_rate
)


display(
    decile_performance
)

In [ ]:
# ================================================================
# VISUAL 25.3 — ACTUAL REPEAT RATE BY SCORE DECILE
# ================================================================

plt.figure(
    figsize=(11, 6)
)


sns.barplot(
    data=decile_performance,
    x="score_decile",
    y="actual_repeat_rate_pct"
)


plt.axhline(
    y=portfolio_repeat_rate * 100,
    linestyle="--",
    label=(
        f"Overall validation rate "
        f"({portfolio_repeat_rate * 100:.1f}%)"
    )
)


plt.title(
    "Actual 90-Day Repeat Rate by Predicted Probability Decile"
)

plt.xlabel(
    "Predicted Probability Decile — 1 = Highest"
)

plt.ylabel(
    "Actual Repeat Rate (%)"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 25.17 — CUMULATIVE GAINS
# ================================================================

validation_scored[
    "cumulative_customers"
] = (
    np.arange(
        1,
        len(validation_scored) + 1
    )
)


validation_scored[
    "cumulative_customer_pct"
] = (
    validation_scored[
        "cumulative_customers"
    ]
    /
    len(validation_scored)
    * 100
)


validation_scored[
    "cumulative_repeaters"
] = (
    validation_scored[
        "actual"
    ]
    .cumsum()
)


total_validation_repeaters = (
    validation_scored[
        "actual"
    ]
    .sum()
)


validation_scored[
    "cumulative_repeater_capture_pct"
] = (
    validation_scored[
        "cumulative_repeaters"
    ]
    /
    total_validation_repeaters
    * 100
)

In [ ]:
# ================================================================
# VISUAL 25.4 — CUMULATIVE GAINS CURVE
# ================================================================

plt.figure(
    figsize=(8, 6)
)


plt.plot(
    validation_scored[
        "cumulative_customer_pct"
    ],
    validation_scored[
        "cumulative_repeater_capture_pct"
    ],
    label="Logistic Regression"
)


plt.plot(
    [0, 100],
    [0, 100],
    linestyle="--",
    label="Random Targeting"
)


plt.title(
    "Cumulative Gains — Repeat Purchasers Captured"
)

plt.xlabel(
    "Customers Targeted (%)"
)

plt.ylabel(
    "Actual Repeat Purchasers Captured (%)"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 25.18 — PROBABILITY CALIBRATION
# ================================================================

fraction_of_positives, mean_predicted_value = (
    calibration_curve(
        y_validation,
        validation_logistic_probabilities,
        n_bins=10,
        strategy="quantile"
    )
)

In [ ]:
# ================================================================
# VISUAL 25.5 — CALIBRATION CURVE
# ================================================================

plt.figure(
    figsize=(8, 6)
)


plt.plot(
    mean_predicted_value,
    fraction_of_positives,
    marker="o",
    label="Logistic Regression"
)


plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect Calibration"
)


plt.title(
    "Probability Calibration — 90-Day Repeat Purchase"
)

plt.xlabel(
    "Mean Predicted Probability"
)

plt.ylabel(
    "Observed Repeat Rate"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 25.19 — CONFUSION MATRIX AT 0.50
# ================================================================

validation_predictions_050 = (
    validation_logistic_probabilities
    >= 0.50
).astype(int)


validation_confusion_matrix = (
    confusion_matrix(
        y_validation,
        validation_predictions_050
    )
)


validation_confusion_df = pd.DataFrame(
    validation_confusion_matrix,
    index=[
        "Actual Non-Repeater",
        "Actual Repeater"
    ],
    columns=[
        "Predicted Non-Repeater",
        "Predicted Repeater"
    ]
)


display(
    validation_confusion_df
)

In [ ]:
# ================================================================
# 25.20 — CLASSIFICATION REPORT
# ================================================================

print(
    classification_report(
        y_validation,
        validation_predictions_050,
        target_names=[
            "Non-Repeater",
            "Repeater"
        ],
        zero_division=0
    )
)

In [ ]:
# ================================================================
# 25.21 — EXTRACT LOGISTIC MODEL COEFFICIENTS
# ================================================================

fitted_preprocessor = (
    logistic_model
    .named_steps[
        "preprocessing"
    ]
)


transformed_feature_names = (
    fitted_preprocessor
    .get_feature_names_out()
)


logistic_coefficients = (
    logistic_model
    .named_steps[
        "model"
    ]
    .coef_[0]
)


coefficient_table = pd.DataFrame(
    {
        "feature":
            transformed_feature_names,

        "coefficient":
            logistic_coefficients,
    }
)


coefficient_table[
    "absolute_coefficient"
] = (
    coefficient_table[
        "coefficient"
    ]
    .abs()
)


coefficient_table = (
    coefficient_table
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    coefficient_table.head(25)
)

In [ ]:
# ================================================================
# VISUAL 25.6 — STRONGEST LOGISTIC COEFFICIENTS
# ================================================================

top_coefficients = (
    coefficient_table
    .head(15)
    .sort_values(
        "coefficient"
    )
)


plt.figure(
    figsize=(10, 7)
)


sns.barplot(
    data=top_coefficients,
    y="feature",
    x="coefficient"
)


plt.axvline(
    0,
    linewidth=1
)


plt.title(
    "Strongest Logistic Associations with 90-Day Repeat Purchase"
)

plt.xlabel(
    "Logistic Regression Coefficient"
)

plt.ylabel(
    ""
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 25.22 — STEP 25 PERFORMANCE REGISTER
# ================================================================

step25_performance_register = (
    step25_model_comparison
    .sort_values(
        "PR-AUC",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    step25_performance_register
)

In [ ]:
# ================================================================
# 25.23 — TEST-SET GOVERNANCE CHECK
# ================================================================

test_set_status = pd.DataFrame(
    {
        "Dataset": [
            "Train",
            "Validation",
            "Test"
        ],

        "Used for Model Fitting": [
            "Yes",
            "No",
            "No"
        ],

        "Used for Model Selection": [
            "Indirectly",
            "Yes",
            "No"
        ],

        "Final Evaluation Status": [
            "Development",
            "Development",
            "RESERVED"
        ],
    }
)


display(
    test_set_status
)

In [ ]:
# ================================================================
# 25.24 — STEP 25 COMPLETION CHECKS
# ================================================================

assert len(
    validation_logistic_probabilities
) == len(
    y_validation
)


assert np.all(
    (
        validation_logistic_probabilities
        >= 0
    )
    &
    (
        validation_logistic_probabilities
        <= 1
    )
)


assert (
    step25_performance_register[
        "PR-AUC"
    ]
    .notna()
    .all()
)


assert (
    step25_performance_register[
        "ROC-AUC"
    ]
    .notna()
    .all()
)


assert (
    step25_performance_register[
        "Brier Score"
    ]
    .notna()
    .all()
)


print(
    "✓ Step 25 complete."
)

print(
    "✓ Preprocessing pipeline validated."
)

print(
    "✓ Baseline models evaluated."
)

print(
    "✓ Logistic regression evaluated on validation data."
)

print(
    "✓ Final test set remains untouched."
)

### Step 25 Summary — Baselines and Logistic Regression

Step 25 established the first predictive benchmarks for the 90-day repeat-purchase problem.

The results show that information available from a customer's first observable order contains genuine, although moderate, predictive signal about whether that customer will purchase again within 90 days.

### Logistic Regression Outperformed the Baselines

The training-prevalence benchmark produced no customer-level discrimination, with a ROC-AUC of **0.50** and a PR-AUC of approximately **0.326**.

Using first-order basket breadth alone improved performance modestly to approximately:

- **PR-AUC: 0.368**
- **ROC-AUC: 0.560**

The multivariable logistic-regression model performed better again, achieving approximately:

- **PR-AUC: 0.407**
- **ROC-AUC: 0.593**
- **Lift at the top 10%: 1.43×**
- **Lift at the top 20%: 1.33×**

This confirms that combining monetary, basket, product, temporal and geographic first-order characteristics provides more predictive information than either the overall repeat rate or basket breadth alone.

### The Model Provides Useful Customer Ranking

The strongest commercial evidence comes from the score-decile analysis.

The highest-scored 10% of validation customers achieved an actual 90-day repeat rate of approximately **46.7%**, compared with an overall validation repeat rate of approximately **32.6%**.

This represents approximately **1.43 times the portfolio repeat rate**.

The highest-scored 10% also captured approximately **14.4% of all actual repeat purchasers**, while the highest-scored 20% captured approximately **26.7%**.

The lowest-scored decile recorded an actual repeat rate of approximately **25%**, materially below the portfolio average.

The model therefore demonstrates an ability to separate customers with relatively high and low repeat propensity.

### Ranking Is Useful but Not Yet Precise

The decile pattern is not perfectly monotonic.

Some neighbouring score groups outperform groups ranked above them, indicating that the model can distinguish broad customer propensity but does not yet produce highly precise ordering across the entire customer population.

The logistic model should therefore be treated as a meaningful benchmark rather than a final production model.

### Probability Calibration Requires Improvement

The model also tends to predict probabilities above the observed repeat rates.

For example, customers in the highest-scored decile received an average predicted probability of approximately **60.3%**, while their actual repeat rate was approximately **46.7%**.

Similar overprediction occurs across several other score groups.

This means the current model is more reliable for **relative ranking** than for interpreting its probability estimates literally.

Probability calibration should therefore be revisited after the strongest predictive model has been selected.

### Overall Conclusion

The central question of Step 25 was whether first-order customer behaviour contains sufficient signal to justify continued predictive modelling.

The answer is **yes**.

Logistic regression materially outperformed both the naive prevalence benchmark and the single-feature benchmark and produced meaningful lift among the highest-ranked customers.

However, discrimination remains moderate and probability calibration is imperfect.

The next stage should therefore test whether non-linear tree-based models can capture relationships that logistic regression cannot, while preserving the temporal validation framework and keeping the final test set untouched.

The logistic-regression model now becomes the benchmark that all subsequent models must outperform.





## Step 26 — Non-Linear Models and Validation Comparison

Step 26 evaluates whether non-linear machine-learning models improve prediction of 90-day repeat purchasing beyond the logistic-regression benchmark established in Step 25.

The models are trained on the same historical training population and evaluated on the same later validation population.

The final temporal test set remains untouched.

Three additional models are evaluated:

- Random Forest;
- Gradient Boosting;
- HistGradientBoosting.

The objective is not to select the most complex algorithm.

A more complex model will only be preferred if it demonstrates a meaningful improvement in:

- PR-AUC;
- ROC-AUC;
- lift among the highest-ranked customers;
- capture of actual repeat purchasers;
- and probability quality.

The logistic-regression model remains the benchmark because it already demonstrates useful ranking performance and provides a comparatively interpretable solution.

In [ ]:
# ================================================================
# 26.2 — IMPORT TREE-BASED MODELS
# ================================================================

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)

In [ ]:
# ================================================================
# 26.3 — TREE-MODEL PREPROCESSING
# ================================================================

tree_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
    ]
)


tree_categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        ),
    ]
)


tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            tree_numeric_transformer,
            numeric_features,
        ),

        (
            "categorical",
            tree_categorical_transformer,
            categorical_features,
        ),
    ]
)


print(
    "✓ Tree-model preprocessing pipeline defined."
)

### 26.4 — RANDOM FOREST

In [ ]:
# ================================================================
# 26.4 — RANDOM FOREST
# ================================================================

random_forest_model = Pipeline(
    steps=[
        (
            "preprocessing",
            tree_preprocessor
        ),

        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=8,
                min_samples_leaf=10,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1
            )
        ),
    ]
)

In [ ]:
random_forest_model.fit(
    X_train,
    y_train
)


print(
    "✓ Random Forest trained."
)

In [ ]:
# ================================================================
# 26.5 — RANDOM FOREST VALIDATION PREDICTIONS
# ================================================================

validation_rf_probabilities = (
    random_forest_model
    .predict_proba(
        X_validation
    )[:, 1]
)

In [ ]:
rf_results = (
    evaluate_binary_model(
        model_name="Random Forest",
        y_true=y_validation,
        probabilities=validation_rf_probabilities,
    )
)


display(
    rf_results
)

### 26.6 — GRADIENT BOOSTING

In [ ]:
# ================================================================
# 26.6 — GRADIENT BOOSTING
# ================================================================

gradient_boosting_model = Pipeline(
    steps=[
        (
            "preprocessing",
            tree_preprocessor
        ),

        (
            "model",
            GradientBoostingClassifier(
                n_estimators=200,
                learning_rate=0.03,
                max_depth=3,
                min_samples_leaf=10,
                random_state=42
            )
        ),
    ]
)

In [ ]:
gradient_boosting_model.fit(
    X_train,
    y_train
)


print(
    "✓ Gradient Boosting trained."
)

In [ ]:
validation_gb_probabilities = (
    gradient_boosting_model
    .predict_proba(
        X_validation
    )[:, 1]
)

In [ ]:
gb_results = (
    evaluate_binary_model(
        model_name="Gradient Boosting",
        y_true=y_validation,
        probabilities=validation_gb_probabilities,
    )
)


display(
    gb_results
)

### 26.7 — HistGradientBoosting

In [ ]:
# ================================================================
# 26.7 — HISTGRADIENTBOOSTING
# ================================================================

hist_gradient_model = Pipeline(
    steps=[
        (
            "preprocessing",
            tree_preprocessor
        ),

        (
            "model",
            HistGradientBoostingClassifier(
                learning_rate=0.05,
                max_iter=300,
                max_leaf_nodes=15,
                min_samples_leaf=20,
                l2_regularization=1.0,
                random_state=42
            )
        ),
    ]
)

In [ ]:
hist_gradient_model.fit(
    X_train,
    y_train
)


print(
    "✓ HistGradientBoosting trained."
)

In [ ]:
validation_hist_probabilities = (
    hist_gradient_model
    .predict_proba(
        X_validation
    )[:, 1]
)

In [ ]:
hist_results = (
    evaluate_binary_model(
        model_name="HistGradientBoosting",
        y_true=y_validation,
        probabilities=validation_hist_probabilities,
    )
)


display(
    hist_results
)

### 26.8 — Compare all models

In [ ]:
# ================================================================
# 26.8 — FULL VALIDATION MODEL COMPARISON
# ================================================================

step26_model_comparison = (
    pd.concat(
        [
            prevalence_results,
            single_feature_results,
            logistic_results,
            rf_results,
            gb_results,
            hist_results,
        ],
        ignore_index=True
    )
)


step26_model_comparison = (
    step26_model_comparison
    .sort_values(
        "PR-AUC",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    step26_model_comparison
)

In [ ]:
# ================================================================
# 26.9 — IMPROVEMENT OVER LOGISTIC BENCHMARK
# ================================================================

logistic_pr_auc = (
    logistic_results[
        "PR-AUC"
    ].iloc[0]
)


logistic_roc_auc = (
    logistic_results[
        "ROC-AUC"
    ].iloc[0]
)


logistic_lift_10 = (
    logistic_results[
        "Lift @ 10%"
    ].iloc[0]
)


model_improvement = (
    step26_model_comparison[
        [
            "Model",
            "PR-AUC",
            "ROC-AUC",
            "Lift @ 10%",
            "Lift @ 20%",
            "Brier Score",
        ]
    ]
    .copy()
)


model_improvement[
    "PR-AUC Improvement vs Logistic"
] = (
    model_improvement[
        "PR-AUC"
    ]
    -
    logistic_pr_auc
)


model_improvement[
    "ROC-AUC Improvement vs Logistic"
] = (
    model_improvement[
        "ROC-AUC"
    ]
    -
    logistic_roc_auc
)


model_improvement[
    "Lift10 Improvement vs Logistic"
] = (
    model_improvement[
        "Lift @ 10%"
    ]
    -
    logistic_lift_10
)


display(
    model_improvement
)

In [ ]:
# ================================================================
# VISUAL 26.1 — PR-AUC COMPARISON
# ================================================================

plt.figure(
    figsize=(11, 6)
)


sns.barplot(
    data=step26_model_comparison,
    y="Model",
    x="PR-AUC"
)


plt.axvline(
    logistic_pr_auc,
    linestyle="--",
    label=(
        f"Logistic benchmark "
        f"({logistic_pr_auc:.3f})"
    )
)


plt.title(
    "Validation PR-AUC by Model"
)

plt.xlabel(
    "PR-AUC"
)

plt.ylabel(
    ""
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 26.2 — TOP-DECILE LIFT
# ================================================================

plt.figure(
    figsize=(11, 6)
)


sns.barplot(
    data=step26_model_comparison,
    y="Model",
    x="Lift @ 10%"
)


plt.axvline(
    1,
    linestyle="--",
    label="No Lift"
)


plt.title(
    "Validation Lift at Top 10% by Model"
)

plt.xlabel(
    "Lift at Top 10%"
)

plt.ylabel(
    ""
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 26.10 — VALIDATION PREDICTION REGISTER
# ================================================================

validation_prediction_register = pd.DataFrame(
    {
        "actual":
            np.asarray(
                y_validation
            ),

        "logistic_probability":
            validation_logistic_probabilities,

        "random_forest_probability":
            validation_rf_probabilities,

        "gradient_boosting_probability":
            validation_gb_probabilities,

        "hist_gradient_probability":
            validation_hist_probabilities,
    }
)


display(
    validation_prediction_register.head()
)

In [ ]:
# ================================================================
# VISUAL 26.3 — PRECISION-RECALL MODEL COMPARISON
# ================================================================

plt.figure(
    figsize=(9, 7)
)


for label, probabilities in [
    (
        "Logistic Regression",
        validation_logistic_probabilities
    ),
    (
        "Random Forest",
        validation_rf_probabilities
    ),
    (
        "Gradient Boosting",
        validation_gb_probabilities
    ),
    (
        "HistGradientBoosting",
        validation_hist_probabilities
    ),
]:

    precision, recall, _ = (
        precision_recall_curve(
            y_validation,
            probabilities
        )
    )

    plt.plot(
        recall,
        precision,
        label=label
    )


plt.axhline(
    y=y_validation.mean(),
    linestyle="--",
    label="Validation Prevalence"
)


plt.title(
    "Precision-Recall Curves — Model Comparison"
)

plt.xlabel(
    "Recall"
)

plt.ylabel(
    "Precision"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 26.4 — ROC MODEL COMPARISON
# ================================================================

plt.figure(
    figsize=(9, 7)
)


for label, probabilities in [
    (
        "Logistic Regression",
        validation_logistic_probabilities
    ),
    (
        "Random Forest",
        validation_rf_probabilities
    ),
    (
        "Gradient Boosting",
        validation_gb_probabilities
    ),
    (
        "HistGradientBoosting",
        validation_hist_probabilities
    ),
]:

    fpr, tpr, _ = (
        roc_curve(
            y_validation,
            probabilities
        )
    )

    plt.plot(
        fpr,
        tpr,
        label=label
    )


plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Ranking"
)


plt.title(
    "ROC Curves — Model Comparison"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 26.13 — CURRENT BEST VALIDATION MODEL
# ================================================================

candidate_models_only = (
    step26_model_comparison.loc[
        step26_model_comparison[
            "Model"
        ].isin(
            [
                "Logistic Regression",
                "Random Forest",
                "Gradient Boosting",
                "HistGradientBoosting",
            ]
        )
    ]
    .copy()
)


best_validation_row = (
    candidate_models_only
    .sort_values(
        [
            "PR-AUC",
            "Lift @ 10%"
        ],
        ascending=False
    )
    .iloc[0]
)


best_validation_model_name = (
    best_validation_row[
        "Model"
    ]
)


print(
    "Current best validation model:"
)

print(
    best_validation_model_name
)

print(
    f"\nPR-AUC: "
    f"{best_validation_row['PR-AUC']:.4f}"
)

print(
    f"ROC-AUC: "
    f"{best_validation_row['ROC-AUC']:.4f}"
)

print(
    f"Lift@10%: "
    f"{best_validation_row['Lift @ 10%']:.4f}"
)

print(
    f"Lift@20%: "
    f"{best_validation_row['Lift @ 20%']:.4f}"
)

In [ ]:
# ================================================================
# 26.14 — REUSABLE DECILE PERFORMANCE FUNCTION
# ================================================================

def create_decile_performance(
    y_true,
    probabilities,
    n_deciles=10
):
    scored = pd.DataFrame(
        {
            "actual":
                np.asarray(
                    y_true
                ),

            "probability":
                np.asarray(
                    probabilities
                ),
        }
    )


    scored = (
        scored
        .sort_values(
            "probability",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


    scored[
        "score_decile"
    ] = (
        pd.qcut(
            scored.index,
            q=n_deciles,
            labels=range(
                1,
                n_deciles + 1
            )
        )
    )


    overall_rate = (
        scored[
            "actual"
        ]
        .mean()
    )


    summary = (
        scored
        .groupby(
            "score_decile",
            observed=True,
            as_index=False
        )
        .agg(
            customers=(
                "actual",
                "size"
            ),

            actual_repeat_rate=(
                "actual",
                "mean"
            ),

            average_predicted_probability=(
                "probability",
                "mean"
            ),

            repeaters=(
                "actual",
                "sum"
            ),
        )
    )


    summary[
        "actual_repeat_rate_pct"
    ] = (
        summary[
            "actual_repeat_rate"
        ]
        * 100
    )


    summary[
        "average_predicted_probability_pct"
    ] = (
        summary[
            "average_predicted_probability"
        ]
        * 100
    )


    summary[
        "lift"
    ] = (
        summary[
            "actual_repeat_rate"
        ]
        / overall_rate
    )


    return summary

In [ ]:
# ================================================================
# 26.15 — SELECT BEST VALIDATION PROBABILITIES
# ================================================================

validation_probability_map = {
    "Logistic Regression":
        validation_logistic_probabilities,

    "Random Forest":
        validation_rf_probabilities,

    "Gradient Boosting":
        validation_gb_probabilities,

    "HistGradientBoosting":
        validation_hist_probabilities,
}


best_validation_probabilities = (
    validation_probability_map[
        best_validation_model_name
    ]
)

In [ ]:
# ================================================================
# 26.16 — BEST-MODEL DECILE PERFORMANCE
# ================================================================

best_model_decile_performance = (
    create_decile_performance(
        y_true=y_validation,
        probabilities=best_validation_probabilities
    )
)


display(
    best_model_decile_performance
)

In [ ]:
# ================================================================
# VISUAL 26.5 — BEST-MODEL DECILE PERFORMANCE
# ================================================================

plt.figure(
    figsize=(11, 6)
)


sns.barplot(
    data=best_model_decile_performance,
    x="score_decile",
    y="actual_repeat_rate_pct"
)


plt.axhline(
    y=y_validation.mean() * 100,
    linestyle="--",
    label=(
        f"Validation average "
        f"({y_validation.mean() * 100:.1f}%)"
    )
)


plt.title(
    f"Actual Repeat Rate by Score Decile — "
    f"{best_validation_model_name}"
)

plt.xlabel(
    "Predicted Probability Decile — 1 = Highest"
)

plt.ylabel(
    "Actual Repeat Rate (%)"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 26.6 — CALIBRATION COMPARISON
# ================================================================

plt.figure(
    figsize=(9, 7)
)


for label, probabilities in [
    (
        "Logistic Regression",
        validation_logistic_probabilities
    ),
    (
        "Random Forest",
        validation_rf_probabilities
    ),
    (
        "Gradient Boosting",
        validation_gb_probabilities
    ),
    (
        "HistGradientBoosting",
        validation_hist_probabilities
    ),
]:

    fraction_positive, mean_prediction = (
        calibration_curve(
            y_validation,
            probabilities,
            n_bins=10,
            strategy="quantile"
        )
    )

    plt.plot(
        mean_prediction,
        fraction_positive,
        marker="o",
        label=label
    )


plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect Calibration"
)


plt.title(
    "Validation Probability Calibration by Model"
)

plt.xlabel(
    "Mean Predicted Probability"
)

plt.ylabel(
    "Observed Repeat Rate"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 26.18 — MODEL-SELECTION FRAMEWORK
# ================================================================

model_selection_framework = pd.DataFrame(
    {
        "Criterion": [
            "PR-AUC",
            "ROC-AUC",
            "Lift @ 10%",
            "Lift @ 20%",
            "Brier Score",
            "Decile separation",
            "Calibration",
            "Interpretability",
            "Complexity",
        ],

        "Importance": [
            "Primary",
            "Secondary",
            "Primary",
            "Primary",
            "Primary",
            "Primary",
            "Primary",
            "Secondary",
            "Secondary",
        ],

        "Decision Question": [
            "Does the model identify repeaters better?",
            "Does it rank customers effectively overall?",
            "How strong is the highest-priority customer group?",
            "Does ranking remain useful at broader targeting levels?",
            "Are probabilities reasonably accurate?",
            "Do high-score customers actually outperform low-score customers?",
            "Can predicted probabilities be trusted?",
            "Can model drivers be explained?",
            "Is added complexity justified by performance improvement?",
        ],
    }
)


display(
    model_selection_framework
)

In [ ]:
# ================================================================
# 26.19 — STEP 26 VALIDATION CHECKS
# ================================================================

validation_probability_sets = {
    "Random Forest":
        validation_rf_probabilities,

    "Gradient Boosting":
        validation_gb_probabilities,

    "HistGradientBoosting":
        validation_hist_probabilities,
}


for model_name, probabilities in (
    validation_probability_sets.items()
):

    assert (
        len(probabilities)
        ==
        len(y_validation)
    ), (
        f"{model_name} prediction length mismatch."
    )


    assert np.all(
        (
            probabilities >= 0
        )
        &
        (
            probabilities <= 1
        )
    ), (
        f"{model_name} generated invalid probabilities."
    )


print(
    "✓ Step 26 model probabilities validated."
)

print(
    "✓ Models evaluated only on validation data."
)

print(
    "✓ Final temporal test set remains untouched."
)

### Step 26 Summary — Non-Linear Model Comparison and Model Selection

Step 26 tested whether more flexible, non-linear machine-learning models could improve prediction of 90-day repeat purchasing beyond the logistic-regression benchmark established in Step 25.

Random Forest, Gradient Boosting and HistGradientBoosting were trained on the same historical training population and evaluated on the same later validation population. The final temporal test set remained untouched throughout this stage.

### Logistic Regression Remained the Strongest Validation Model

Logistic regression produced the strongest overall validation performance across the metrics most relevant to this use case.

Its results were:

- **PR-AUC: 0.407**
- **ROC-AUC: 0.593**
- **Brier score: 0.225**
- **Lift at the top 10%: 1.43×**
- **Lift at the top 20%: 1.33×**
- **Capture at the top 10%: 14.36%**
- **Capture at the top 20%: 26.67%**

These results were stronger than all three non-linear alternatives.

Gradient Boosting was the best-performing tree-based model, but its validation performance was lower:

- **PR-AUC: 0.382**
- **ROC-AUC: 0.564**
- **Lift at the top 10%: 1.28×**
- **Lift at the top 20%: 1.23×**

Random Forest and HistGradientBoosting produced weaker discrimination and lower lift again.

This means that additional model complexity did not improve generalisation to later customers.

### The Logistic Model Provides the Best Customer Ranking

The central commercial objective is not simply to classify customers correctly at an arbitrary probability threshold.

The more important question is whether the model can rank newly acquired customers so that high-scored customers are materially more likely to repeat than the overall customer population.

Logistic regression performed best on this task.

The highest-scored 10% of customers repeated at approximately **1.43 times** the overall validation repeat rate.

The highest-scored 20% captured approximately **26.7% of all actual repeat purchasers**.

This indicates that the model has meaningful value as a customer-prioritisation tool, even though its discrimination is moderate rather than exceptionally strong.

### The Naive Baseline Confirms That the Lift Is Genuine

The corrected training-prevalence baseline has:

- **ROC-AUC: 0.50**
- **Lift at the top 10%: 1.00×**
- **Lift at the top 20%: 1.00×**

This is the expected result from a model that assigns the same probability to every customer and therefore has no ranking ability.

Against that benchmark, logistic regression's **1.43× top-decile lift** represents genuine customer-level ranking signal.

### Threshold Metrics Should Not Drive Model Selection

Gradient Boosting produced stronger recall and F1 at the default 0.50 threshold.

However, this does not make it the preferred model.

A 0.50 threshold is arbitrary and does not reflect the actual business decision.

The intended use case is to rank customers by repeat-purchase propensity and prioritise intervention capacity.

For this purpose, logistic regression performs better on:

- PR-AUC;
- ROC-AUC;
- top-decile lift;
- broader lift;
- repeater capture;
- and Brier score.

The final operating threshold should therefore be selected later using commercial capacity and intervention economics rather than the default 0.50 cut-off.

### Model Complexity Did Not Add Predictive Value

The results suggest that the available first-order features may contain relatively broad predictive relationships rather than strong complex interactions.

The non-linear models were capable of fitting more complicated patterns, but those patterns did not generalise as effectively to the later validation population.

Given the modest sample size and temporal validation design, the simpler model appears to generalise better.

This is important because a more complex model should only be selected when its predictive improvement is sufficient to justify:

- reduced interpretability;
- increased implementation complexity;
- and greater monitoring requirements.

That condition was not met.

### Model Selection Decision

Based on the validation evidence, **Logistic Regression remains the preferred candidate model**.

It currently provides the best balance of:

- predictive discrimination;
- commercial lift;
- probability error;
- interpretability;
- and implementation simplicity.

The tree-based models are therefore not selected for further development at this stage.

### Current Limitation

The selected logistic model should still be regarded as a **moderately predictive model**, not a final production-ready system.

The model can meaningfully separate higher- and lower-propensity customers, but its ranking is not perfectly monotonic across all customer groups and its predicted probabilities show evidence of miscalibration.

The next stage should therefore focus on improving and stress-testing the logistic specification before the reserved temporal test set is opened.

The next questions are:

> **Which first-order characteristics are driving the prediction?**

> **Are those effects stable and interpretable?**

> **Can calibration be improved without weakening ranking performance?**

> **Does the final selected specification still perform when evaluated once on the untouched future test population?**

## Step 27 — Logistic Model Diagnostics, Interpretation and Calibration

Step 26 established Logistic Regression as the strongest validation model for predicting 90-day repeat purchase.

Step 27 now evaluates whether that model is sufficiently stable, interpretable and well calibrated to proceed toward final testing.

This stage does not open the reserved temporal test population.

The analysis focuses on five questions:

1. Are the numerical predictors excessively skewed or redundant?
2. Which first-order characteristics are most strongly associated with repeat purchase?
3. Are the model's relationships stable across later customer cohorts?
4. Can logistic regularisation be improved using historical training data only?
5. Can predicted probabilities be better calibrated without sacrificing customer ranking?

The objective is to lock a final candidate specification before the temporal test set is evaluated once.

## Step 27 — Logistic Model Diagnostics, Interpretation, Stability and Calibration

Step 26 established Logistic Regression as the strongest validation model for predicting 90-day repeat purchase.

Step 27 does not introduce a new model family.

Instead, it subjects the selected logistic model to deeper diagnostic testing before the reserved temporal test set is opened.

The objectives are to determine:

- which first-order characteristics contribute most strongly to predicted repeat propensity;
- whether important predictors are excessively redundant or correlated;
- whether heavily skewed first-order variables would benefit from transformation;
- whether a simpler logistic specification performs as well as the full model;
- whether predictive relationships remain reasonably stable through time;
- whether the training and validation populations exhibit material feature drift;
- whether the model's predicted probabilities are appropriately calibrated;
- and whether calibration can improve probability accuracy without reducing customer-ranking value.

The current Logistic Regression model remains the benchmark throughout this stage.

The final temporal test population remains untouched.

### Interpretation Principle

Logistic-regression coefficients represent predictive associations conditional on the other variables in the model.

They should not be interpreted as causal effects.

Because numerical predictors were standardised before modelling, numerical odds ratios represent the change in predicted odds associated with approximately one training-standard-deviation increase in the corresponding feature.

### Decision Principle

A revised logistic specification will only replace the current model if it demonstrates comparable or stronger validation ranking performance while improving simplicity, stability, interpretability or probability calibration.

In [ ]:
# ================================================================
# 27.2 — IMPORT DIAGNOSTIC LIBRARIES
# ================================================================

from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit
from sklearn.calibration import (
    CalibratedClassifierCV,
    calibration_curve,
)

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
)

import warnings

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)

In [ ]:
# ================================================================
# OPTIONAL — VIF SUPPORT
# ================================================================

try:

    from statsmodels.stats.outliers_influence import (
        variance_inflation_factor
    )

    STATSMODELS_AVAILABLE = True

except ImportError:

    STATSMODELS_AVAILABLE = False


print(
    f"Statsmodels available for VIF: "
    f"{STATSMODELS_AVAILABLE}"
)

In [ ]:
# ================================================================
# 27.3 — VALIDATE REQUIRED OBJECTS
# ================================================================

required_objects = [
    "train_data",
    "validation_data",
    "X_train",
    "y_train",
    "X_validation",
    "y_validation",
    "numeric_features",
    "categorical_features",
    "model_features",
    "logistic_model",
    "validation_logistic_probabilities",
    "evaluate_binary_model",
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:

    raise RuntimeError(
        "The following required objects are missing. "
        "Rerun the relevant Step 24/25 cells first: "
        f"{missing_objects}"
    )


print(
    "✓ Required Step 24–26 modelling objects are available."
)

In [ ]:
# ================================================================
# 27.4 — CURRENT CHAMPION BENCHMARK
# ================================================================

current_champion_results = (
    evaluate_binary_model(
        model_name="Current Logistic Regression",
        y_true=y_validation,
        probabilities=validation_logistic_probabilities,
    )
)


display(
    current_champion_results
)

In [ ]:
# ================================================================
# 27.5 — EXTRACT MODEL COEFFICIENTS
# ================================================================

fitted_preprocessor = (
    logistic_model
    .named_steps[
        "preprocessing"
    ]
)


fitted_classifier = (
    logistic_model
    .named_steps[
        "model"
    ]
)


transformed_feature_names = (
    fitted_preprocessor
    .get_feature_names_out()
)


logistic_coefficients = (
    fitted_classifier
    .coef_[0]
)


assert (
    len(transformed_feature_names)
    ==
    len(logistic_coefficients)
)


coefficient_diagnostics = pd.DataFrame(
    {
        "feature":
            transformed_feature_names,

        "coefficient":
            logistic_coefficients,
    }
)


coefficient_diagnostics[
    "odds_ratio"
] = np.exp(
    coefficient_diagnostics[
        "coefficient"
    ]
)


coefficient_diagnostics[
    "absolute_coefficient"
] = (
    coefficient_diagnostics[
        "coefficient"
    ]
    .abs()
)


coefficient_diagnostics[
    "direction"
] = np.where(
    coefficient_diagnostics[
        "coefficient"
    ] > 0,
    "Higher repeat propensity",
    np.where(
        coefficient_diagnostics[
            "coefficient"
        ] < 0,
        "Lower repeat propensity",
        "Neutral"
    )
)


coefficient_diagnostics = (
    coefficient_diagnostics
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    coefficient_diagnostics.head(30)
)

In [ ]:
# ================================================================
# 27.6 — CLEAN FEATURE LABELS
# ================================================================

coefficient_diagnostics[
    "clean_feature"
] = (
    coefficient_diagnostics[
        "feature"
    ]
    .str.replace(
        "numeric__",
        "",
        regex=False
    )
    .str.replace(
        "categorical__",
        "",
        regex=False
    )
)


display(
    coefficient_diagnostics[
        [
            "clean_feature",
            "coefficient",
            "odds_ratio",
            "direction",
        ]
    ]
    .head(25)
)

In [ ]:
# ================================================================
# VISUAL 27.1 — STRONGEST LOGISTIC ASSOCIATIONS
# ================================================================

top_coefficient_plot = (
    coefficient_diagnostics
    .head(20)
    .sort_values(
        "coefficient"
    )
)


plt.figure(
    figsize=(11, 9)
)


sns.barplot(
    data=top_coefficient_plot,
    y="clean_feature",
    x="coefficient"
)


plt.axvline(
    0,
    linewidth=1
)


plt.title(
    "Strongest Predictive Associations with 90-Day Repeat Purchase"
)

plt.xlabel(
    "Standardised Logistic Coefficient"
)

plt.ylabel(
    ""
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 27.7 — NUMERIC VS CATEGORICAL COEFFICIENTS
# ================================================================

numeric_coefficient_table = (
    coefficient_diagnostics.loc[
        coefficient_diagnostics[
            "feature"
        ]
        .str.startswith(
            "numeric__"
        )
    ]
    .copy()
)


categorical_coefficient_table = (
    coefficient_diagnostics.loc[
        coefficient_diagnostics[
            "feature"
        ]
        .str.startswith(
            "categorical__"
        )
    ]
    .copy()
)


print(
    "NUMERIC EFFECTS"
)

display(
    numeric_coefficient_table[
        [
            "clean_feature",
            "coefficient",
            "odds_ratio",
            "direction",
        ]
    ]
)


print(
    "\nCATEGORICAL / REGION EFFECTS"
)

display(
    categorical_coefficient_table[
        [
            "clean_feature",
            "coefficient",
            "odds_ratio",
            "direction",
        ]
    ]
    .head(30)
)

In [ ]:
# ================================================================
# 27.8 — NUMERIC FEATURE CORRELATION
# ================================================================

train_numeric_for_diagnostics = (
    X_train[
        numeric_features
    ]
    .copy()
)


numeric_correlation_matrix = (
    train_numeric_for_diagnostics
    .corr(
        method="spearman"
    )
)


display(
    numeric_correlation_matrix.round(3)
)

In [ ]:
# ================================================================
# VISUAL 27.2 — NUMERIC FEATURE CORRELATION HEATMAP
# ================================================================

plt.figure(
    figsize=(14, 11)
)


sns.heatmap(
    numeric_correlation_matrix,
    cmap="coolwarm",
    center=0,
    annot=False
)


plt.title(
    "Spearman Correlation — First-Order Predictors"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 27.9 — HIGH-CORRELATION PAIRS
# ================================================================

correlation_pairs = []


for i, feature_a in enumerate(
    numeric_correlation_matrix.columns
):

    for j, feature_b in enumerate(
        numeric_correlation_matrix.columns
    ):

        if j <= i:
            continue


        correlation_value = (
            numeric_correlation_matrix
            .loc[
                feature_a,
                feature_b
            ]
        )


        correlation_pairs.append(
            {
                "feature_1":
                    feature_a,

                "feature_2":
                    feature_b,

                "spearman_correlation":
                    correlation_value,

                "absolute_correlation":
                    abs(
                        correlation_value
                    ),
            }
        )


high_correlation_pairs = (
    pd.DataFrame(
        correlation_pairs
    )
    .sort_values(
        "absolute_correlation",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    high_correlation_pairs.head(25)
)

In [ ]:
display(
    high_correlation_pairs.loc[
        high_correlation_pairs[
            "absolute_correlation"
        ] >= 0.80
    ]
)

In [ ]:
# ================================================================
# 27.10 — VARIANCE INFLATION FACTOR
# ================================================================

if STATSMODELS_AVAILABLE:

    vif_imputer = SimpleImputer(
        strategy="median"
    )


    vif_matrix = (
        vif_imputer
        .fit_transform(
            X_train[
                numeric_features
            ]
        )
    )


    vif_table = pd.DataFrame(
        {
            "feature":
                numeric_features,

            "VIF":
                [
                    variance_inflation_factor(
                        vif_matrix,
                        i
                    )
                    for i in range(
                        vif_matrix.shape[1]
                    )
                ],
        }
    )


    vif_table = (
        vif_table
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .sort_values(
            "VIF",
            ascending=False,
            na_position="first"
        )
        .reset_index(
            drop=True
        )
    )


    display(
        vif_table
    )

else:

    print(
        "Statsmodels is unavailable. "
        "Correlation diagnostics remain available."
    )

In [ ]:
# ================================================================
# 27.11 — NUMERIC FEATURE SKEWNESS
# ================================================================

feature_skewness = (
    X_train[
        numeric_features
    ]
    .skew(
        numeric_only=True
    )
    .sort_values(
        key=lambda x: x.abs(),
        ascending=False
    )
    .rename(
        "skewness"
    )
    .reset_index()
    .rename(
        columns={
            "index":
                "feature"
        }
    )
)


feature_skewness[
    "absolute_skewness"
] = (
    feature_skewness[
        "skewness"
    ]
    .abs()
)


display(
    feature_skewness
)

In [ ]:
# ================================================================
# 27.12 — NUMERIC QUANTILE DIAGNOSTICS
# ================================================================

numeric_quantiles = (
    X_train[
        numeric_features
    ]
    .quantile(
        [
            0.00,
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
            1.00,
        ]
    )
    .T
)


numeric_quantiles.columns = [
    "min",
    "p01",
    "p05",
    "p25",
    "median",
    "p75",
    "p95",
    "p99",
    "max",
]


display(
    numeric_quantiles
)

In [ ]:
OneHotEncoder(drop="first")

In [ ]:
# ================================================================
# 27.13 — INTERPRETABLE DROP-FIRST LOGISTIC MODEL
# ================================================================

interpretable_categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        ),
    ]
)


interpretable_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features,
        ),

        (
            "categorical",
            interpretable_categorical_transformer,
            categorical_features,
        ),
    ]
)


interpretable_logistic_model = Pipeline(
    steps=[
        (
            "preprocessing",
            interpretable_preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=3000,
                random_state=42
            )
        ),
    ]
)

In [ ]:
interpretable_logistic_model.fit(
    X_train,
    y_train
)

In [ ]:
validation_interpretable_probabilities = (
    interpretable_logistic_model
    .predict_proba(
        X_validation
    )[:, 1]
)

In [ ]:
interpretable_results = (
    evaluate_binary_model(
        model_name="Drop-First Interpretable Logistic",
        y_true=y_validation,
        probabilities=validation_interpretable_probabilities,
    )
)


display(
    interpretable_results
)

In [ ]:
# ================================================================
# 27.14 — CORE FEATURE SPECIFICATION
# ================================================================

core_numeric_features = [
    "first_order_value",
    "first_order_units",
    "first_order_product_count",
    "first_order_average_unit_price",
    "first_order_line_count",
    "first_order_month_number",
    "first_order_weekday",
    "first_order_hour",
    "first_order_is_weekend",
]


core_categorical_features = [
    "first_order_region",
]


core_model_features = (
    core_numeric_features
    +
    core_categorical_features
)

In [ ]:
core_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            core_numeric_features,
        ),

        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),

                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            drop="first"
                        )
                    ),
                ]
            ),
            core_categorical_features,
        ),
    ]
)


core_logistic_model = Pipeline(
    steps=[
        (
            "preprocessing",
            core_preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=3000,
                random_state=42
            )
        ),
    ]
)

In [ ]:
core_logistic_model.fit(
    X_train[
        core_model_features
    ],
    y_train
)

In [ ]:
validation_core_probabilities = (
    core_logistic_model
    .predict_proba(
        X_validation[
            core_model_features
        ]
    )[:, 1]
)

In [ ]:
core_logistic_results = (
    evaluate_binary_model(
        model_name="Core Logistic Regression",
        y_true=y_validation,
        probabilities=validation_core_probabilities,
    )
)


display(
    core_logistic_results
)

In [ ]:
# ================================================================
# 27.15 — LOG-TRANSFORMED CORE FEATURES
# ================================================================

log_transform_features = [
    "first_order_value",
    "first_order_units",
    "first_order_product_count",
    "first_order_average_unit_price",
    "first_order_line_count",
]

In [ ]:
def create_log_core_features(
    data
):

    transformed = (
        data[
            core_model_features
        ]
        .copy()
    )


    for feature in log_transform_features:

        transformed[
            f"log1p_{feature}"
        ] = np.log1p(
            transformed[
                feature
            ]
            .clip(
                lower=0
            )
        )


    transformed = transformed.drop(
        columns=log_transform_features
    )


    return transformed

In [ ]:
X_train_log_core = (
    create_log_core_features(
        X_train
    )
)


X_validation_log_core = (
    create_log_core_features(
        X_validation
    )
)

In [ ]:
log_core_numeric_features = [
    f"log1p_{feature}"
    for feature in log_transform_features
] + [
    "first_order_month_number",
    "first_order_weekday",
    "first_order_hour",
    "first_order_is_weekend",
]


log_core_categorical_features = [
    "first_order_region"
]

In [ ]:
log_core_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            log_core_numeric_features,
        ),

        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),

                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            drop="first"
                        )
                    ),
                ]
            ),
            log_core_categorical_features,
        ),
    ]
)


log_core_logistic_model = Pipeline(
    steps=[
        (
            "preprocessing",
            log_core_preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=3000,
                random_state=42
            )
        ),
    ]
)

In [ ]:
log_core_logistic_model.fit(
    X_train_log_core,
    y_train
)

In [ ]:
validation_log_core_probabilities = (
    log_core_logistic_model
    .predict_proba(
        X_validation_log_core
    )[:, 1]
)

In [ ]:
log_core_results = (
    evaluate_binary_model(
        model_name="Log-Transformed Core Logistic",
        y_true=y_validation,
        probabilities=validation_log_core_probabilities,
    )
)


display(
    log_core_results
)

In [ ]:
# ================================================================
# 27.16 — LOGISTIC SPECIFICATION COMPARISON
# ================================================================

logistic_specification_comparison = (
    pd.concat(
        [
            current_champion_results,
            interpretable_results,
            core_logistic_results,
            log_core_results,
        ],
        ignore_index=True
    )
)


logistic_specification_comparison = (
    logistic_specification_comparison
    .sort_values(
        [
            "PR-AUC",
            "Lift @ 10%"
        ],
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    logistic_specification_comparison
)

In [ ]:
# ================================================================
# VISUAL 27.3 — PR-AUC ACROSS LOGISTIC SPECIFICATIONS
# ================================================================

plt.figure(
    figsize=(11, 6)
)


sns.barplot(
    data=logistic_specification_comparison,
    y="Model",
    x="PR-AUC"
)


plt.title(
    "Validation PR-AUC Across Logistic Specifications"
)

plt.xlabel(
    "PR-AUC"
)

plt.ylabel(
    ""
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 27.17 — NUMERIC FEATURE DRIFT
# ================================================================

feature_drift_records = []


for feature in numeric_features:

    train_values = (
        pd.to_numeric(
            X_train[
                feature
            ],
            errors="coerce"
        )
    )


    validation_values = (
        pd.to_numeric(
            X_validation[
                feature
            ],
            errors="coerce"
        )
    )


    train_mean = (
        train_values.mean()
    )

    validation_mean = (
        validation_values.mean()
    )


    train_std = (
        train_values.std()
    )

    validation_std = (
        validation_values.std()
    )


    pooled_std = np.sqrt(
        (
            train_std ** 2
            +
            validation_std ** 2
        )
        / 2
    )


    if (
        pd.isna(
            pooled_std
        )
        or
        pooled_std == 0
    ):

        smd = np.nan

    else:

        smd = (
            validation_mean
            -
            train_mean
        ) / pooled_std


    feature_drift_records.append(
        {
            "feature":
                feature,

            "train_mean":
                train_mean,

            "validation_mean":
                validation_mean,

            "standardised_mean_difference":
                smd,

            "absolute_smd":
                abs(smd)
                if not pd.isna(smd)
                else np.nan,
        }
    )


numeric_feature_drift = (
    pd.DataFrame(
        feature_drift_records
    )
    .sort_values(
        "absolute_smd",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    numeric_feature_drift
)

In [ ]:
# ================================================================
# VISUAL 27.4 — NUMERIC FEATURE DRIFT
# ================================================================

drift_plot = (
    numeric_feature_drift
    .sort_values(
        "absolute_smd"
    )
)


plt.figure(
    figsize=(11, 8)
)


sns.barplot(
    data=drift_plot,
    y="feature",
    x="absolute_smd"
)


plt.axvline(
    0.10,
    linestyle="--",
    label="0.10"
)


plt.axvline(
    0.25,
    linestyle="--",
    label="0.25"
)


plt.title(
    "Training-to-Validation Numeric Feature Drift"
)

plt.xlabel(
    "Absolute Standardised Mean Difference"
)

plt.ylabel(
    ""
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 27.18 — CATEGORICAL REGION DRIFT
# ================================================================

train_region_share = (
    X_train[
        "first_order_region"
    ]
    .fillna(
        "Missing"
    )
    .value_counts(
        normalize=True
    )
    .rename(
        "train_share"
    )
)


validation_region_share = (
    X_validation[
        "first_order_region"
    ]
    .fillna(
        "Missing"
    )
    .value_counts(
        normalize=True
    )
    .rename(
        "validation_share"
    )
)


region_drift = (
    pd.concat(
        [
            train_region_share,
            validation_region_share,
        ],
        axis=1
    )
    .fillna(0)
    .reset_index()
    .rename(
        columns={
            "index":
                "region"
        }
    )
)


region_drift[
    "share_change_pp"
] = (
    region_drift[
        "validation_share"
    ]
    -
    region_drift[
        "train_share"
    ]
) * 100


region_drift[
    "absolute_change_pp"
] = (
    region_drift[
        "share_change_pp"
    ]
    .abs()
)


region_drift = (
    region_drift
    .sort_values(
        "absolute_change_pp",
        ascending=False
    )
)


display(
    region_drift.head(20)
)

In [ ]:
# ================================================================
# 27.19 — UNSEEN VALIDATION CATEGORIES
# ================================================================

train_regions = set(
    X_train[
        "first_order_region"
    ]
    .dropna()
    .astype(str)
)


validation_regions = set(
    X_validation[
        "first_order_region"
    ]
    .dropna()
    .astype(str)
)


unseen_validation_regions = (
    validation_regions
    -
    train_regions
)


print(
    "Validation regions not observed in training:"
)

print(
    unseen_validation_regions
)

In [ ]:
handle_unknown="ignore"

In [ ]:
# ================================================================
# 27.20 — TEMPORAL CROSS-VALIDATION
# ================================================================

train_ordered = (
    train_data
    .sort_values(
        "first_order_date"
    )
    .reset_index(
        drop=True
    )
)


X_train_ordered = (
    train_ordered[
        model_features
    ]
    .copy()
)


y_train_ordered = (
    train_ordered[
        target_column
    ]
    .copy()
)


time_cv = TimeSeriesSplit(
    n_splits=5
)

In [ ]:
temporal_cv_records = []


for fold_number, (
    fold_train_index,
    fold_validation_index
) in enumerate(
    time_cv.split(
        X_train_ordered
    ),
    start=1
):

    X_fold_train = (
        X_train_ordered
        .iloc[
            fold_train_index
        ]
    )


    y_fold_train = (
        y_train_ordered
        .iloc[
            fold_train_index
        ]
    )


    X_fold_validation = (
        X_train_ordered
        .iloc[
            fold_validation_index
        ]
    )


    y_fold_validation = (
        y_train_ordered
        .iloc[
            fold_validation_index
        ]
    )


    fold_model = clone(
        logistic_model
    )


    fold_model.fit(
        X_fold_train,
        y_fold_train
    )


    fold_probabilities = (
        fold_model
        .predict_proba(
            X_fold_validation
        )[:, 1]
    )


    temporal_cv_records.append(
        {
            "fold":
                fold_number,

            "train_rows":
                len(
                    fold_train_index
                ),

            "validation_rows":
                len(
                    fold_validation_index
                ),

            "validation_repeat_rate":
                y_fold_validation.mean(),

            "PR_AUC":
                average_precision_score(
                    y_fold_validation,
                    fold_probabilities
                ),

            "ROC_AUC":
                roc_auc_score(
                    y_fold_validation,
                    fold_probabilities
                ),

            "Brier_Score":
                brier_score_loss(
                    y_fold_validation,
                    fold_probabilities
                ),

            "Lift_10":
                lift_at_fraction(
                    y_fold_validation,
                    fold_probabilities,
                    0.10
                ),

            "Lift_20":
                lift_at_fraction(
                    y_fold_validation,
                    fold_probabilities,
                    0.20
                ),
        }
    )


temporal_cv_results = pd.DataFrame(
    temporal_cv_records
)


display(
    temporal_cv_results
)

In [ ]:
# ================================================================
# 27.21 — TEMPORAL CV SUMMARY
# ================================================================

temporal_cv_summary = pd.DataFrame(
    {
        "metric": [
            "PR_AUC",
            "ROC_AUC",
            "Brier_Score",
            "Lift_10",
            "Lift_20",
        ],

        "mean": [
            temporal_cv_results[
                "PR_AUC"
            ].mean(),

            temporal_cv_results[
                "ROC_AUC"
            ].mean(),

            temporal_cv_results[
                "Brier_Score"
            ].mean(),

            temporal_cv_results[
                "Lift_10"
            ].mean(),

            temporal_cv_results[
                "Lift_20"
            ].mean(),
        ],

        "std": [
            temporal_cv_results[
                "PR_AUC"
            ].std(),

            temporal_cv_results[
                "ROC_AUC"
            ].std(),

            temporal_cv_results[
                "Brier_Score"
            ].std(),

            temporal_cv_results[
                "Lift_10"
            ].std(),

            temporal_cv_results[
                "Lift_20"
            ].std(),
        ],

        "minimum": [
            temporal_cv_results[
                "PR_AUC"
            ].min(),

            temporal_cv_results[
                "ROC_AUC"
            ].min(),

            temporal_cv_results[
                "Brier_Score"
            ].min(),

            temporal_cv_results[
                "Lift_10"
            ].min(),

            temporal_cv_results[
                "Lift_20"
            ].min(),
        ],

        "maximum": [
            temporal_cv_results[
                "PR_AUC"
            ].max(),

            temporal_cv_results[
                "ROC_AUC"
            ].max(),

            temporal_cv_results[
                "Brier_Score"
            ].max(),

            temporal_cv_results[
                "Lift_10"
            ].max(),

            temporal_cv_results[
                "Lift_20"
            ].max(),
        ],
    }
)


display(
    temporal_cv_summary
)

In [ ]:
# ================================================================
# VISUAL 27.5 — PR-AUC BY TEMPORAL FOLD
# ================================================================

plt.figure(
    figsize=(9, 5)
)


sns.lineplot(
    data=temporal_cv_results,
    x="fold",
    y="PR_AUC",
    marker="o"
)


plt.title(
    "PR-AUC Across Expanding Temporal Validation Folds"
)

plt.xlabel(
    "Temporal Fold"
)

plt.ylabel(
    "PR-AUC"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 27.6 — LIFT@10 BY TEMPORAL FOLD
# ================================================================

plt.figure(
    figsize=(9, 5)
)


sns.lineplot(
    data=temporal_cv_results,
    x="fold",
    y="Lift_10",
    marker="o"
)


plt.axhline(
    1,
    linestyle="--"
)


plt.title(
    "Top-Decile Lift Across Temporal Validation Folds"
)

plt.xlabel(
    "Temporal Fold"
)

plt.ylabel(
    "Lift @ 10%"
)

plt.tight_layout()
plt.show()

#### 27.22 — Calibration: why we are doing it
Our Step 25–26 analysis suggested:
> Ranking is useful, but probabilities are too high in several deciles.

We should therefore test calibration.
Importantly, calibration must not use the final test set.
We will calibrate using temporal cross-validation inside the training population, then evaluate the calibrated model on the existing validation population.

In [ ]:
# ================================================================
# 27.23 — TEMPORAL CALIBRATION CROSS-VALIDATION
# ================================================================

calibration_cv = TimeSeriesSplit(
    n_splits=5
)

#### 27.24 — Sigmoid / Platt calibration

In [ ]:
# ================================================================
# 27.24 — CALIBRATION HELPER
# ================================================================

def build_calibrated_classifier(
    estimator,
    method,
    cv
):

    try:

        return CalibratedClassifierCV(
            estimator=estimator,
            method=method,
            cv=cv
        )

    except TypeError:

        return CalibratedClassifierCV(
            base_estimator=estimator,
            method=method,
            cv=cv
        )

In [ ]:
sigmoid_calibrated_model = (
    build_calibrated_classifier(
        estimator=clone(
            logistic_model
        ),
        method="sigmoid",
        cv=calibration_cv
    )
)

In [ ]:
sigmoid_calibrated_model.fit(
    X_train_ordered,
    y_train_ordered
)

In [ ]:
validation_sigmoid_probabilities = (
    sigmoid_calibrated_model
    .predict_proba(
        X_validation
    )[:, 1]
)

In [ ]:
sigmoid_calibration_results = (
    evaluate_binary_model(
        model_name="Logistic + Sigmoid Calibration",
        y_true=y_validation,
        probabilities=validation_sigmoid_probabilities,
    )
)


display(
    sigmoid_calibration_results
)

#### 27.25 — Isotonic calibration

In [ ]:
# ================================================================
# 27.25 — ISOTONIC CALIBRATION
# ================================================================

isotonic_calibrated_model = (
    build_calibrated_classifier(
        estimator=clone(
            logistic_model
        ),
        method="isotonic",
        cv=TimeSeriesSplit(
            n_splits=5
        )
    )
)

In [ ]:
isotonic_calibrated_model.fit(
    X_train_ordered,
    y_train_ordered
)

In [ ]:
validation_isotonic_probabilities = (
    isotonic_calibrated_model
    .predict_proba(
        X_validation
    )[:, 1]
)

In [ ]:
isotonic_calibration_results = (
    evaluate_binary_model(
        model_name="Logistic + Isotonic Calibration",
        y_true=y_validation,
        probabilities=validation_isotonic_probabilities,
    )
)


display(
    isotonic_calibration_results
)

In [ ]:
# ================================================================
# 27.26 — CALIBRATION COMPARISON
# ================================================================

calibration_comparison = (
    pd.concat(
        [
            current_champion_results,
            sigmoid_calibration_results,
            isotonic_calibration_results,
        ],
        ignore_index=True
    )
)


calibration_comparison = (
    calibration_comparison
    .sort_values(
        "Brier Score"
    )
    .reset_index(
        drop=True
    )
)


display(
    calibration_comparison
)

In [ ]:
# ================================================================
# VISUAL 27.7 — CALIBRATION COMPARISON
# ================================================================

plt.figure(
    figsize=(9, 7)
)


calibration_models = [
    (
        "Original Logistic",
        validation_logistic_probabilities
    ),

    (
        "Sigmoid Calibrated",
        validation_sigmoid_probabilities
    ),

    (
        "Isotonic Calibrated",
        validation_isotonic_probabilities
    ),
]


for label, probabilities in calibration_models:

    observed_rate, predicted_rate = (
        calibration_curve(
            y_validation,
            probabilities,
            n_bins=10,
            strategy="quantile"
        )
    )


    plt.plot(
        predicted_rate,
        observed_rate,
        marker="o",
        label=label
    )


plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect Calibration"
)


plt.title(
    "Calibration of 90-Day Repeat Probabilities"
)

plt.xlabel(
    "Mean Predicted Probability"
)

plt.ylabel(
    "Observed Repeat Rate"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 27.27 — DECILE CALIBRATION TABLE
# ================================================================

def calibration_decile_table(
    y_true,
    probabilities,
    model_name
):

    table = create_decile_performance(
        y_true=y_true,
        probabilities=probabilities,
        n_deciles=10
    )


    table[
        "calibration_error_pp"
    ] = (
        table[
            "average_predicted_probability_pct"
        ]
        -
        table[
            "actual_repeat_rate_pct"
        ]
    )


    table[
        "absolute_calibration_error_pp"
    ] = (
        table[
            "calibration_error_pp"
        ]
        .abs()
    )


    table[
        "model"
    ] = model_name


    return table

In [ ]:
original_calibration_deciles = (
    calibration_decile_table(
        y_validation,
        validation_logistic_probabilities,
        "Original Logistic"
    )
)


sigmoid_calibration_deciles = (
    calibration_decile_table(
        y_validation,
        validation_sigmoid_probabilities,
        "Sigmoid"
    )
)


isotonic_calibration_deciles = (
    calibration_decile_table(
        y_validation,
        validation_isotonic_probabilities,
        "Isotonic"
    )
)

In [ ]:
calibration_error_summary = pd.DataFrame(
    {
        "Model": [
            "Original Logistic",
            "Sigmoid",
            "Isotonic",
        ],

        "Mean Absolute Decile Calibration Error (pp)": [
            original_calibration_deciles[
                "absolute_calibration_error_pp"
            ].mean(),

            sigmoid_calibration_deciles[
                "absolute_calibration_error_pp"
            ].mean(),

            isotonic_calibration_deciles[
                "absolute_calibration_error_pp"
            ].mean(),
        ],
    }
)


display(
    calibration_error_summary
)

In [ ]:
# ================================================================
# 27.28 — RANKING VS CALIBRATION DECISION TABLE
# ================================================================

ranking_calibration_decision = (
    calibration_comparison[
        [
            "Model",
            "PR-AUC",
            "ROC-AUC",
            "Brier Score",
            "Lift @ 10%",
            "Lift @ 20%",
            "Capture @ 10%",
            "Capture @ 20%",
        ]
    ]
    .copy()
)


display(
    ranking_calibration_decision
)

In [ ]:
# ================================================================
# 27.29 — CALIBRATION DECISION SUPPORT
# ================================================================

original_pr_auc = (
    current_champion_results[
        "PR-AUC"
    ]
    .iloc[0]
)


original_lift10 = (
    current_champion_results[
        "Lift @ 10%"
    ]
    .iloc[0]
)


calibration_decision_support = (
    calibration_comparison
    .copy()
)


calibration_decision_support[
    "PR_AUC_change_vs_original"
] = (
    calibration_decision_support[
        "PR-AUC"
    ]
    -
    original_pr_auc
)


calibration_decision_support[
    "Lift10_change_vs_original"
] = (
    calibration_decision_support[
        "Lift @ 10%"
    ]
    -
    original_lift10
)


display(
    calibration_decision_support
)

In [ ]:
# ================================================================
# 27.30 — STEP 27 DIAGNOSTIC REGISTER
# ================================================================

step27_diagnostic_register = pd.DataFrame(
    {
        "Diagnostic Area": [
            "Predictor importance",
            "Feature correlation",
            "Multicollinearity",
            "Feature skewness",
            "Model simplification",
            "Feature drift",
            "Temporal stability",
            "Probability calibration",
        ],

        "Question": [
            "Which first-order characteristics drive repeat propensity?",
            "Are predictors carrying overlapping information?",
            "Are individual coefficients potentially unstable?",
            "Would nonlinear transformations improve the specification?",
            "Can a smaller model generalise as well as the full model?",
            "Do later customers materially differ from training customers?",
            "Does performance persist across historical time windows?",
            "Can probability accuracy improve without sacrificing ranking?",
        ],

        "Primary Output": [
            "coefficient_diagnostics",
            "high_correlation_pairs",
            "vif_table if available",
            "feature_skewness",
            "logistic_specification_comparison",
            "numeric_feature_drift / region_drift",
            "temporal_cv_results",
            "calibration_comparison",
        ],
    }
)


display(
    step27_diagnostic_register
)

In [ ]:
# ================================================================
# 27.31 — STEP 27 VALIDATION
# ================================================================

assert len(
    validation_sigmoid_probabilities
) == len(
    y_validation
)


assert len(
    validation_isotonic_probabilities
) == len(
    y_validation
)


assert np.all(
    (
        validation_sigmoid_probabilities >= 0
    )
    &
    (
        validation_sigmoid_probabilities <= 1
    )
)


assert np.all(
    (
        validation_isotonic_probabilities >= 0
    )
    &
    (
        validation_isotonic_probabilities <= 1
    )
)


assert (
    len(
        temporal_cv_results
    )
    == 5
)


assert (
    "X_test"
    in globals()
)


print(
    "✓ Step 27 diagnostics complete."
)

print(
    "✓ Logistic specifications compared."
)

print(
    "✓ Temporal stability assessed."
)

print(
    "✓ Probability calibration assessed."
)

print(
    "✓ Final temporal test set remains untouched."
)

### Step 27 Summary — Logistic Diagnostics, Stability and Calibration

Step 27 subjected the selected logistic-regression model to deeper diagnostic testing before opening the reserved temporal test population.

The analysis examined predictor interpretation, alternative logistic specifications, population drift, temporal model stability and probability calibration.

### First-Order Basket Behaviour Contains Interpretable Predictive Signal

Among the strongest numerical predictors, first-order product breadth and first-order monetary value were positively associated with 90-day repeat purchasing.

The standardised coefficient for first-order product count was approximately **0.410**, corresponding to an odds ratio of approximately **1.51**.

The coefficient for first-order order value was approximately **0.385**, corresponding to an odds ratio of approximately **1.47**.

This means that, conditional on the other predictors in the model, customers beginning with broader or higher-value baskets tended to have higher subsequent repeat-purchase odds.

These relationships are predictive associations rather than evidence that increasing basket breadth or monetary value would itself cause a customer to return.

Some related basket measures, including total units and invoice-line count, showed negative conditional coefficients. Because several first-order basket characteristics measure overlapping aspects of the same transaction, individual coefficients should be interpreted cautiously and in the context of multicollinearity.

### Geographic Effects Require Cautious Interpretation

Several geographic categories displayed relatively large positive or negative coefficients.

However, geographic coefficients should not be interpreted as causal market effects.

Some markets may contain relatively few observations, and the original one-hot specification did not use a single explicit reference category.

An alternative drop-first categorical specification was therefore tested to improve interpretability.

### The Interpretable Logistic Specification Preserved Performance

The **Drop-First Interpretable Logistic Regression** produced the strongest overall balance of predictive performance and interpretability.

Its validation results were approximately:

- **PR-AUC: 0.408**
- **ROC-AUC: 0.592**
- **Brier score: 0.225**
- **top-10% lift: 1.43×**
- **top-20% lift: 1.33×**

Performance was effectively equivalent to, and marginally better on some measures than, the original logistic specification.

Because the drop-first model maintains predictive quality while producing cleaner categorical interpretation, it is preferred as the current model specification.

### Most Predictive Signal Exists in a Relatively Small Feature Set

A simplified Core Logistic Regression performed almost as well as the full model:

- **PR-AUC: 0.406**
- **ROC-AUC: 0.591**
- **top-10% lift: 1.43×**
- **top-20% lift: 1.41×**

This indicates that most of the useful predictive signal is contained within a relatively small number of first-order variables.

The additional engineered features contribute limited incremental predictive value.

The simplified specification therefore remains an important challenger because it may provide operational advantages in deployment and monitoring.

### Log Transformation Did Not Improve Customer Ranking

A log-transformed core specification slightly reduced Brier score but weakened PR-AUC and top-decile lift.

The probability-error improvement was therefore not sufficient to justify the reduction in customer-ranking performance.

The log-transformed model is not selected.

### Most Customer Characteristics Remained Stable Across Time

Training-to-validation feature-drift analysis showed that most first-order economic characteristics were relatively stable.

First-order value, units per product and average line quantity displayed particularly small differences between the training and validation populations.

The principal exception was acquisition month.

The standardised mean difference for first-order month number was approximately **0.78**, reflecting the intentionally chronological train-validation design and the fact that later acquisition months dominate the validation population.

This also identifies a feature-engineering limitation: calendar month is currently represented as a simple integer despite being cyclical.

A final diagnostic specification should therefore replace raw month number with cyclical month features before final test evaluation.

### Predictive Performance Varies Across Historical Periods

Expanding temporal cross-validation showed that model performance was not perfectly stable through time.

PR-AUC varied materially across historical folds, and top-decile lift ranged from below 1.0 in one early period to approximately 1.4 in stronger later periods.

This indicates that the relationship between first-order characteristics and repeat purchasing changes to some extent across customer cohorts.

The model should therefore be described as having **moderate but temporally variable predictive signal** rather than assumed to perform uniformly through time.

This finding reinforces the importance of the untouched temporal test population.

### Calibration Did Not Provide a Sufficient Improvement

Both sigmoid and isotonic probability calibration were evaluated.

Sigmoid calibration slightly improved Brier score and reduced average decile calibration error, but the improvement was small while PR-AUC, ROC-AUC and broader customer-ranking lift declined.

Isotonic calibration performed worse on both ranking and calibration-error measures.

The raw logistic model is therefore retained without additional calibration.

At this stage, its output should primarily be interpreted as a **relative propensity score** rather than a perfectly calibrated individual probability.

### Step 27 Model Decision

The preferred model remains a logistic-regression approach.

The **Drop-First Interpretable Logistic Regression** is selected as the current champion because it retains the strongest predictive performance while improving categorical interpretability.

The Core Logistic Regression remains a strong challenger because it achieves nearly identical performance using fewer predictors.

No calibration method is selected.

Before opening the reserved test set, one final specification refinement is warranted: calendar month should be represented cyclically rather than as a linear integer.

After that refinement, the model specification should be frozen and evaluated once against the untouched future test population.



## Step 28 — Final Feature Specification, Cyclical Time Encoding and Champion Freeze

Step 27 identified the final important feature-engineering issue before opening the reserved temporal test population.

Calendar month was represented as an integer from 1 to 12.

This representation incorrectly treats calendar time as linear. For example, December and January are adjacent months in reality, but their numerical values are far apart.

A similar issue exists for weekday, where Sunday and Monday are adjacent despite appearing at opposite ends of a conventional integer encoding.

Step 28 therefore performs one final controlled feature-engineering experiment.

The objective is to compare the current selected logistic-regression specification against versions that replace linear month and weekday variables with cyclical sine and cosine representations.

No new model families are introduced.

The models compared are:

1. the current Drop-First Interpretable Logistic Regression;
2. a full Logistic Regression using cyclical month and weekday features; and
3. a simplified Core Logistic Regression using the same cyclical calendar representation.

The final specification will be selected using validation performance only.

The reserved temporal test set will remain untouched.

After this comparison, feature engineering and model selection will be frozen. The selected specification will then proceed to a single final evaluation on the future test population.

In [ ]:
# ================================================================
# 28.2 — TEST-SET GOVERNANCE CHECK
# ================================================================

print(
    f"Train rows:      {len(X_train):,}"
)

print(
    f"Validation rows: {len(X_validation):,}"
)

print(
    f"Test rows:       {len(X_test):,}"
)


print(
    "\nStep 28 uses TRAIN and VALIDATION only."
)

print(
    "The final TEST set remains reserved."
)

In [ ]:
TEST_SET_OPENED = False

assert TEST_SET_OPENED is False

print(
    "✓ Final temporal test set remains unopened."
)

In [ ]:
# ================================================================
# 28.4 — CYCLICAL FEATURE ENGINEERING FUNCTION
# ================================================================

def add_cyclical_calendar_features(
    data
):
    """
    Replace linear calendar representations with
    cyclical sine/cosine features.

    Month is encoded over a 12-month cycle.
    Weekday is encoded over a 7-day cycle.

    Original input data is not modified.
    """

    transformed = data.copy()


    # ------------------------------------------------------------
    # Month
    # ------------------------------------------------------------

    transformed[
        "first_order_month_sin"
    ] = np.sin(
        2
        *
        np.pi
        *
        transformed[
            "first_order_month_number"
        ]
        /
        12
    )


    transformed[
        "first_order_month_cos"
    ] = np.cos(
        2
        *
        np.pi
        *
        transformed[
            "first_order_month_number"
        ]
        /
        12
    )


    # ------------------------------------------------------------
    # Weekday
    # Existing encoding:
    # Monday = 0 ... Sunday = 6
    # ------------------------------------------------------------

    transformed[
        "first_order_weekday_sin"
    ] = np.sin(
        2
        *
        np.pi
        *
        transformed[
            "first_order_weekday"
        ]
        /
        7
    )


    transformed[
        "first_order_weekday_cos"
    ] = np.cos(
        2
        *
        np.pi
        *
        transformed[
            "first_order_weekday"
        ]
        /
        7
    )


    return transformed

In [ ]:
# ================================================================
# 28.5 — CREATE CYCLICAL TRAIN / VALIDATION DATA
# ================================================================

X_train_cyclical = (
    add_cyclical_calendar_features(
        X_train
    )
)


X_validation_cyclical = (
    add_cyclical_calendar_features(
        X_validation
    )
)

In [ ]:
# ================================================================
# 28.6 — VALIDATE CYCLICAL FEATURES
# ================================================================

cyclical_columns = [
    "first_order_month_sin",
    "first_order_month_cos",
    "first_order_weekday_sin",
    "first_order_weekday_cos",
]


for column in cyclical_columns:

    assert (
        column
        in
        X_train_cyclical.columns
    )


    assert (
        X_train_cyclical[
            column
        ]
        .between(
            -1,
            1
        )
        .all()
    )


print(
    "✓ Cyclical features successfully created."
)

In [ ]:
# ================================================================
# 28.7 — MONTH ENCODING LOOKUP
# ================================================================

month_encoding_lookup = pd.DataFrame(
    {
        "month": np.arange(
            1,
            13
        )
    }
)


month_encoding_lookup[
    "month_sin"
] = np.sin(
    2
    *
    np.pi
    *
    month_encoding_lookup[
        "month"
    ]
    /
    12
)


month_encoding_lookup[
    "month_cos"
] = np.cos(
    2
    *
    np.pi
    *
    month_encoding_lookup[
        "month"
    ]
    /
    12
)


display(
    month_encoding_lookup.round(4)
)

In [ ]:
# ================================================================
# VISUAL 28.1 — CYCLICAL MONTH ENCODING
# ================================================================

plt.figure(
    figsize=(8, 7)
)


plt.scatter(
    month_encoding_lookup[
        "month_cos"
    ],
    month_encoding_lookup[
        "month_sin"
    ]
)


for _, row in (
    month_encoding_lookup
    .iterrows()
):

    plt.text(
        row[
            "month_cos"
        ],
        row[
            "month_sin"
        ],
        str(
            int(
                row[
                    "month"
                ]
            )
        )
    )


plt.title(
    "Cyclical Representation of Calendar Month"
)

plt.xlabel(
    "Month Cosine"
)

plt.ylabel(
    "Month Sine"
)

plt.axhline(
    0,
    linewidth=0.5
)

plt.axvline(
    0,
    linewidth=0.5
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 28.8 — FULL CYCLICAL FEATURE SET
# ================================================================

full_cyclical_numeric_features = [
    feature
    for feature in numeric_features
    if feature
    not in [
        "first_order_month_number",
        "first_order_weekday",
    ]
]

In [ ]:
full_cyclical_numeric_features += [
    "first_order_month_sin",
    "first_order_month_cos",
    "first_order_weekday_sin",
    "first_order_weekday_cos",
]

In [ ]:
full_cyclical_categorical_features = (
    categorical_features.copy()
)

In [ ]:
full_cyclical_features = (
    full_cyclical_numeric_features
    +
    full_cyclical_categorical_features
)


print(
    f"Full cyclical numeric features: "
    f"{len(full_cyclical_numeric_features)}"
)

print(
    f"Categorical features: "
    f"{len(full_cyclical_categorical_features)}"
)

print(
    f"Total predictors before one-hot expansion: "
    f"{len(full_cyclical_features)}"
)

In [ ]:
# ================================================================
# 28.9 — FULL CYCLICAL LOGISTIC PIPELINE
# ================================================================

full_cyclical_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median"
                        )
                    ),

                    (
                        "scaler",
                        StandardScaler()
                    ),
                ]
            ),
            full_cyclical_numeric_features,
        ),

        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),

                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            drop="first"
                        )
                    ),
                ]
            ),
            full_cyclical_categorical_features,
        ),
    ]
)

In [ ]:
full_cyclical_logistic_model = Pipeline(
    steps=[
        (
            "preprocessing",
            full_cyclical_preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=3000,
                random_state=42
            )
        ),
    ]
)

In [ ]:
# ================================================================
# 28.10 — TRAIN FULL CYCLICAL LOGISTIC
# ================================================================

full_cyclical_logistic_model.fit(
    X_train_cyclical[
        full_cyclical_features
    ],
    y_train
)


print(
    "✓ Full cyclical logistic model trained."
)

In [ ]:
# ================================================================
# 28.11 — FULL CYCLICAL VALIDATION PERFORMANCE
# ================================================================

validation_full_cyclical_probabilities = (
    full_cyclical_logistic_model
    .predict_proba(
        X_validation_cyclical[
            full_cyclical_features
        ]
    )[:, 1]
)


full_cyclical_results = (
    evaluate_binary_model(
        model_name="Full Cyclical Logistic",
        y_true=y_validation,
        probabilities=validation_full_cyclical_probabilities,
    )
)


display(
    full_cyclical_results
)

In [ ]:
# ================================================================
# 28.12 — CORE CYCLICAL FEATURE SET
# ================================================================

core_cyclical_numeric_features = [
    "first_order_value",
    "first_order_units",
    "first_order_product_count",
    "first_order_average_unit_price",
    "first_order_line_count",
    "first_order_hour",
    "first_order_is_weekend",

    "first_order_month_sin",
    "first_order_month_cos",

    "first_order_weekday_sin",
    "first_order_weekday_cos",
]


core_cyclical_categorical_features = [
    "first_order_region",
]


core_cyclical_features = (
    core_cyclical_numeric_features
    +
    core_cyclical_categorical_features
)

In [ ]:
# ================================================================
# 28.13 — CORE CYCLICAL LOGISTIC PIPELINE
# ================================================================

core_cyclical_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median"
                        )
                    ),

                    (
                        "scaler",
                        StandardScaler()
                    ),
                ]
            ),
            core_cyclical_numeric_features,
        ),

        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),

                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            drop="first"
                        )
                    ),
                ]
            ),
            core_cyclical_categorical_features,
        ),
    ]
)

In [ ]:
core_cyclical_logistic_model = Pipeline(
    steps=[
        (
            "preprocessing",
            core_cyclical_preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=3000,
                random_state=42
            )
        ),
    ]
)

In [ ]:
# ================================================================
# 28.14 — TRAIN CORE CYCLICAL LOGISTIC
# ================================================================

core_cyclical_logistic_model.fit(
    X_train_cyclical[
        core_cyclical_features
    ],
    y_train
)


print(
    "✓ Core cyclical logistic model trained."
)

In [ ]:
# ================================================================
# 28.15 — CORE CYCLICAL VALIDATION PERFORMANCE
# ================================================================

validation_core_cyclical_probabilities = (
    core_cyclical_logistic_model
    .predict_proba(
        X_validation_cyclical[
            core_cyclical_features
        ]
    )[:, 1]
)


core_cyclical_results = (
    evaluate_binary_model(
        model_name="Core Cyclical Logistic",
        y_true=y_validation,
        probabilities=validation_core_cyclical_probabilities,
    )
)


display(
    core_cyclical_results
)

In [ ]:
# ================================================================
# 28.16 — FINAL SPECIFICATION COMPARISON
# ================================================================

final_specification_comparison = (
    pd.concat(
        [
            interpretable_results,
            full_cyclical_results,
            core_cyclical_results,
        ],
        ignore_index=True
    )
)


final_specification_comparison = (
    final_specification_comparison
    .sort_values(
        [
            "PR-AUC",
            "Lift @ 10%",
            "Lift @ 20%"
        ],
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    final_specification_comparison
)

In [ ]:
# ================================================================
# 28.17 — PERFORMANCE CHANGE VS CURRENT CHAMPION
# ================================================================

champion_pr_auc = (
    interpretable_results[
        "PR-AUC"
    ]
    .iloc[0]
)


champion_roc_auc = (
    interpretable_results[
        "ROC-AUC"
    ]
    .iloc[0]
)


champion_brier = (
    interpretable_results[
        "Brier Score"
    ]
    .iloc[0]
)


champion_lift10 = (
    interpretable_results[
        "Lift @ 10%"
    ]
    .iloc[0]
)


champion_lift20 = (
    interpretable_results[
        "Lift @ 20%"
    ]
    .iloc[0]
)

In [ ]:
final_specification_deltas = (
    final_specification_comparison[
        [
            "Model",
            "PR-AUC",
            "ROC-AUC",
            "Brier Score",
            "Lift @ 10%",
            "Lift @ 20%",
            "Capture @ 10%",
            "Capture @ 20%",
        ]
    ]
    .copy()
)


final_specification_deltas[
    "PR_AUC_delta"
] = (
    final_specification_deltas[
        "PR-AUC"
    ]
    -
    champion_pr_auc
)


final_specification_deltas[
    "ROC_AUC_delta"
] = (
    final_specification_deltas[
        "ROC-AUC"
    ]
    -
    champion_roc_auc
)


final_specification_deltas[
    "Brier_delta"
] = (
    final_specification_deltas[
        "Brier Score"
    ]
    -
    champion_brier
)


final_specification_deltas[
    "Lift10_delta"
] = (
    final_specification_deltas[
        "Lift @ 10%"
    ]
    -
    champion_lift10
)


final_specification_deltas[
    "Lift20_delta"
] = (
    final_specification_deltas[
        "Lift @ 20%"
    ]
    -
    champion_lift20
)


display(
    final_specification_deltas
)

In [ ]:
# ================================================================
# VISUAL 28.2 — FINAL PR-AUC COMPARISON
# ================================================================

plt.figure(
    figsize=(10, 6)
)


sns.barplot(
    data=final_specification_comparison,
    y="Model",
    x="PR-AUC"
)


plt.axvline(
    champion_pr_auc,
    linestyle="--",
    label=(
        f"Current champion "
        f"({champion_pr_auc:.4f})"
    )
)


plt.title(
    "Final Logistic Specification Comparison — PR-AUC"
)

plt.xlabel(
    "Validation PR-AUC"
)

plt.ylabel(
    ""
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 28.3 — TOP-DECILE LIFT
# ================================================================

plt.figure(
    figsize=(10, 6)
)


sns.barplot(
    data=final_specification_comparison,
    y="Model",
    x="Lift @ 10%"
)


plt.axvline(
    1,
    linestyle="--",
    label="No Lift"
)


plt.title(
    "Final Logistic Specification Comparison — Top-Decile Lift"
)

plt.xlabel(
    "Lift @ Top 10%"
)

plt.ylabel(
    ""
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 28.18 — DECILE PERFORMANCE FOR FINAL CANDIDATES
# ================================================================

current_champion_deciles = (
    create_decile_performance(
        y_true=y_validation,
        probabilities=validation_interpretable_probabilities
    )
)


full_cyclical_deciles = (
    create_decile_performance(
        y_true=y_validation,
        probabilities=validation_full_cyclical_probabilities
    )
)


core_cyclical_deciles = (
    create_decile_performance(
        y_true=y_validation,
        probabilities=validation_core_cyclical_probabilities
    )
)

In [ ]:
current_champion_deciles[
    "Model"
] = "Drop-First Logistic"


full_cyclical_deciles[
    "Model"
] = "Full Cyclical Logistic"


core_cyclical_deciles[
    "Model"
] = "Core Cyclical Logistic"

In [ ]:
final_decile_comparison = pd.concat(
    [
        current_champion_deciles,
        full_cyclical_deciles,
        core_cyclical_deciles,
    ],
    ignore_index=True
)


display(
    final_decile_comparison
)

In [ ]:
# ================================================================
# VISUAL 28.4 — DECILE SEPARATION COMPARISON
# ================================================================

plt.figure(
    figsize=(12, 7)
)


sns.lineplot(
    data=final_decile_comparison,
    x="score_decile",
    y="actual_repeat_rate_pct",
    hue="Model",
    marker="o"
)


plt.axhline(
    y=y_validation.mean() * 100,
    linestyle="--",
    label="Validation Average"
)


plt.title(
    "Observed Repeat Rate by Model Score Decile"
)

plt.xlabel(
    "Score Decile — 1 = Highest Predicted Propensity"
)

plt.ylabel(
    "Observed 90-Day Repeat Rate (%)"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 28.19 — RAW CALIBRATION COMPARISON
# ================================================================

final_calibration_records = []


for model_name, probabilities in [
    (
        "Drop-First Logistic",
        validation_interpretable_probabilities
    ),

    (
        "Full Cyclical Logistic",
        validation_full_cyclical_probabilities
    ),

    (
        "Core Cyclical Logistic",
        validation_core_cyclical_probabilities
    ),
]:

    decile_table = (
        calibration_decile_table(
            y_true=y_validation,
            probabilities=probabilities,
            model_name=model_name
        )
    )


    final_calibration_records.append(
        {
            "Model":
                model_name,

            "Mean Absolute Decile Calibration Error (pp)":
                decile_table[
                    "absolute_calibration_error_pp"
                ]
                .mean(),
        }
    )


final_calibration_comparison = pd.DataFrame(
    final_calibration_records
)


display(
    final_calibration_comparison
)

In [ ]:
# ================================================================
# 28.20 — CYCLICAL MODEL COEFFICIENTS
# ================================================================

cyclical_preprocessor_fitted = (
    full_cyclical_logistic_model
    .named_steps[
        "preprocessing"
    ]
)


cyclical_classifier_fitted = (
    full_cyclical_logistic_model
    .named_steps[
        "model"
    ]
)


cyclical_feature_names = (
    cyclical_preprocessor_fitted
    .get_feature_names_out()
)


cyclical_coefficients = (
    cyclical_classifier_fitted
    .coef_[0]
)


cyclical_coefficient_table = pd.DataFrame(
    {
        "feature":
            cyclical_feature_names,

        "coefficient":
            cyclical_coefficients,
    }
)


cyclical_coefficient_table[
    "odds_ratio"
] = np.exp(
    cyclical_coefficient_table[
        "coefficient"
    ]
)


cyclical_coefficient_table[
    "absolute_coefficient"
] = (
    cyclical_coefficient_table[
        "coefficient"
    ]
    .abs()
)


cyclical_coefficient_table = (
    cyclical_coefficient_table
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    cyclical_coefficient_table.head(25)
)

In [ ]:
# ================================================================
# 28.21 — TIME EFFECTS
# ================================================================

cyclical_time_effects = (
    cyclical_coefficient_table.loc[
        cyclical_coefficient_table[
            "feature"
        ]
        .str.contains(
            "month_sin|month_cos|weekday_sin|weekday_cos",
            regex=True
        )
    ]
    .copy()
)


display(
    cyclical_time_effects
)

In [ ]:
# ================================================================
# 28.22 — PRACTICAL EQUIVALENCE THRESHOLD
# ================================================================

PR_AUC_EQUIVALENCE_TOLERANCE = 0.002

In [ ]:
# ================================================================
# 28.23 — FINAL SELECTION SCORECARD
# ================================================================

final_selection_scorecard = (
    final_specification_comparison[
        [
            "Model",
            "PR-AUC",
            "ROC-AUC",
            "Brier Score",
            "Lift @ 10%",
            "Lift @ 20%",
            "Capture @ 10%",
            "Capture @ 20%",
        ]
    ]
    .copy()
)

In [ ]:
complexity_map = {
    "Drop-First Interpretable Logistic":
        "Moderate",

    "Full Cyclical Logistic":
        "Moderate",

    "Core Cyclical Logistic":
        "Lower",
}


final_selection_scorecard[
    "Complexity"
] = (
    final_selection_scorecard[
        "Model"
    ]
    .map(
        complexity_map
    )
)

In [ ]:
final_selection_scorecard = (
    final_selection_scorecard
    .merge(
        final_calibration_comparison,
        on="Model",
        how="left"
    )
)

In [ ]:
final_calibration_comparison[
    "Model"
] = (
    final_calibration_comparison[
        "Model"
    ]
    .replace(
        {
            "Drop-First Logistic":
                "Drop-First Interpretable Logistic"
        }
    )
)

In [ ]:
final_selection_scorecard = (
    final_specification_comparison[
        [
            "Model",
            "PR-AUC",
            "ROC-AUC",
            "Brier Score",
            "Lift @ 10%",
            "Lift @ 20%",
            "Capture @ 10%",
            "Capture @ 20%",
        ]
    ]
    .copy()
)


final_selection_scorecard[
    "Complexity"
] = (
    final_selection_scorecard[
        "Model"
    ]
    .map(
        complexity_map
    )
)


final_selection_scorecard = (
    final_selection_scorecard
    .merge(
        final_calibration_comparison,
        on="Model",
        how="left"
    )
)


display(
    final_selection_scorecard
)

In [ ]:
# ================================================================
# 28.24 — STATISTICAL LEADER
# ================================================================

best_pr_auc = (
    final_selection_scorecard[
        "PR-AUC"
    ]
    .max()
)


final_selection_scorecard[
    "Within_PR_AUC_Equivalence_Band"
] = (
    final_selection_scorecard[
        "PR-AUC"
    ]
    >=
    (
        best_pr_auc
        -
        PR_AUC_EQUIVALENCE_TOLERANCE
    )
)


display(
    final_selection_scorecard
)

In [ ]:
# ================================================================
# 28.25 — PRACTICALLY COMPETITIVE MODELS
# ================================================================

competitive_models = (
    final_selection_scorecard.loc[
        final_selection_scorecard[
            "Within_PR_AUC_Equivalence_Band"
        ]
    ]
    .copy()
)


competitive_models = (
    competitive_models
    .sort_values(
        [
            "Lift @ 10%",
            "Lift @ 20%",
            "Brier Score",
        ],
        ascending=[
            False,
            False,
            True,
        ]
    )
)


display(
    competitive_models
)

In [ ]:
# ================================================================
# 28.26 — CHAMPION RECOMMENDATION
# ================================================================

recommended_champion_row = (
    competitive_models
    .iloc[0]
)


recommended_champion_name = (
    recommended_champion_row[
        "Model"
    ]
)


print(
    "Recommended model based on "
    "validation evidence:"
)

print(
    recommended_champion_name
)

In [ ]:
# ================================================================
# 28.27 — MODEL OBJECT REGISTER
# ================================================================

final_model_registry = {
    "Drop-First Interpretable Logistic":
        interpretable_logistic_model,

    "Full Cyclical Logistic":
        full_cyclical_logistic_model,

    "Core Cyclical Logistic":
        core_cyclical_logistic_model,
}

In [ ]:
final_probability_registry = {
    "Drop-First Interpretable Logistic":
        validation_interpretable_probabilities,

    "Full Cyclical Logistic":
        validation_full_cyclical_probabilities,

    "Core Cyclical Logistic":
        validation_core_cyclical_probabilities,
}

In [ ]:
# ================================================================
# 28.28 — PROVISIONAL CHAMPION FREEZE
# ================================================================

frozen_model_name = (
    recommended_champion_name
)


frozen_model = (
    final_model_registry[
        frozen_model_name
    ]
)


frozen_validation_probabilities = (
    final_probability_registry[
        frozen_model_name
    ]
)


print(
    f"Frozen candidate model: "
    f"{frozen_model_name}"
)

In [ ]:
# ================================================================
# 28.29 — FROZEN FEATURE SPECIFICATION
# ================================================================

if (
    frozen_model_name
    ==
    "Drop-First Interpretable Logistic"
):

    frozen_feature_type = (
        "Original calendar encoding"
    )

    frozen_features = (
        model_features.copy()
    )


elif (
    frozen_model_name
    ==
    "Full Cyclical Logistic"
):

    frozen_feature_type = (
        "Full cyclical encoding"
    )

    frozen_features = (
        full_cyclical_features.copy()
    )


elif (
    frozen_model_name
    ==
    "Core Cyclical Logistic"
):

    frozen_feature_type = (
        "Core cyclical encoding"
    )

    frozen_features = (
        core_cyclical_features.copy()
    )


else:

    raise ValueError(
        "Unknown frozen model."
    )

In [ ]:
print(
    f"Frozen feature type: "
    f"{frozen_feature_type}"
)

print(
    f"Number of pre-encoded predictors: "
    f"{len(frozen_features)}"
)


display(
    pd.DataFrame(
        {
            "Frozen Features":
                frozen_features
        }
    )
)

In [ ]:
# ================================================================
# 28.30 — FROZEN MODEL DECILE PERFORMANCE
# ================================================================

frozen_model_decile_performance = (
    create_decile_performance(
        y_true=y_validation,
        probabilities=frozen_validation_probabilities,
    )
)


display(
    frozen_model_decile_performance
)

In [ ]:
# ================================================================
# 28.31 — FROZEN MODEL VALIDATION REGISTER
# ================================================================

frozen_validation_results = (
    evaluate_binary_model(
        model_name=frozen_model_name,
        y_true=y_validation,
        probabilities=frozen_validation_probabilities,
    )
)


display(
    frozen_validation_results
)

In [ ]:
# ================================================================
# 28.32 — MODEL FREEZE REGISTER
# ================================================================

model_freeze_register = pd.DataFrame(
    {
        "Item": [
            "Prediction target",
            "Prediction point",
            "Prediction horizon",
            "Selected algorithm family",
            "Frozen specification",
            "Calibration layer",
            "Primary selection metric",
            "Secondary commercial metric",
            "Final test status",
        ],

        "Decision": [
            "Second purchase within 90 days",
            "Immediately after first observable order",
            "90 days",
            "Logistic Regression",
            frozen_model_name,
            "None",
            "PR-AUC",
            "Lift @ 10% and Lift @ 20%",
            "UNOPENED",
        ],
    }
)


display(
    model_freeze_register
)

In [ ]:
# ================================================================
# 28.33 — ANTI-TUNING DECLARATION
# ================================================================

MODEL_SPECIFICATION_FROZEN = True

TEST_SET_OPENED = False


assert MODEL_SPECIFICATION_FROZEN
assert not TEST_SET_OPENED


print(
    "✓ Model specification frozen."
)

print(
    "✓ No additional validation-driven feature engineering "
    "should occur after this point."
)

print(
    "✓ Final temporal test set remains unopened."
)

In [ ]:
# ================================================================
# 28.34 — STEP 28 VALIDATION
# ================================================================

assert len(
    validation_full_cyclical_probabilities
) == len(
    y_validation
)


assert len(
    validation_core_cyclical_probabilities
) == len(
    y_validation
)


assert np.all(
    (
        validation_full_cyclical_probabilities
        >= 0
    )
    &
    (
        validation_full_cyclical_probabilities
        <= 1
    )
)


assert np.all(
    (
        validation_core_cyclical_probabilities
        >= 0
    )
    &
    (
        validation_core_cyclical_probabilities
        <= 1
    )
)


assert frozen_model_name in (
    final_model_registry
)


assert MODEL_SPECIFICATION_FROZEN


assert not TEST_SET_OPENED


print(
    "✓ Step 28 complete."
)

print(
    "✓ Cyclical calendar representations evaluated."
)

print(
    "✓ Final logistic candidates compared."
)

print(
    f"✓ Frozen candidate: {frozen_model_name}"
)

print(
    "✓ Final test population remains untouched."
)

### Step 28 Summary — Final Feature Specification and Champion Freeze

Step 28 performed the final validation-driven feature-engineering experiment before opening the reserved temporal test population.

The objective was to determine whether replacing linear month and weekday variables with cyclical sine and cosine representations would improve prediction of 90-day repeat purchasing.

Three final logistic-regression specifications were compared:

1. Drop-First Interpretable Logistic Regression;
2. Full Cyclical Logistic Regression; and
3. Core Cyclical Logistic Regression.

### Cyclical Encoding Did Not Improve Overall Model Quality

The existing Drop-First Interpretable Logistic Regression remained the strongest overall validation model.

Its performance was approximately:

- **PR-AUC: 0.408**
- **ROC-AUC: 0.592**
- **Brier score: 0.225**
- **top-10% lift: 1.43×**
- **top-20% lift: 1.33×**
- **top-10% repeater capture: 14.36%**
- **top-20% repeater capture: 26.67%**

The cyclical specifications produced stronger concentration among the highest-ranked 10% of customers.

The Full Cyclical Logistic model, for example, achieved approximately **1.69× top-decile lift**, while the Core Cyclical Logistic achieved approximately **1.64×**.

However, those gains were accompanied by materially weaker overall discrimination.

The Full Cyclical model's PR-AUC declined to approximately **0.398** and ROC-AUC to approximately **0.571**.

The Core Cyclical model produced approximately **0.399 PR-AUC** and **0.574 ROC-AUC**.

Neither model therefore remained within the predetermined PR-AUC practical-equivalence band of the current champion.

### Probability Quality Also Deteriorated

The cyclical specifications substantially weakened probability accuracy.

The current Drop-First Logistic model produced a Brier score of approximately **0.225**, compared with approximately **0.249** for both cyclical alternatives.

Average absolute decile calibration error was approximately:

- **9.74 percentage points — Drop-First Logistic**
- **17.83 percentage points — Full Cyclical Logistic**
- **17.92 percentage points — Core Cyclical Logistic**

The cyclical models therefore almost doubled the average calibration error.

### Higher Top-Decile Lift Was Not Sufficient to Replace the Champion

Although cyclical encoding improved top-decile lift, the purpose of the model is broader repeat-propensity ranking rather than optimisation of a single customer segment.

Selecting a cyclical model would improve performance within the highest-ranked 10% while sacrificing broader ranking performance and probability quality.

That trade-off was not considered sufficient to justify changing the model.

### Calendar Drift Was Investigated Rather Than Ignored

Step 27 identified substantial train-to-validation drift in first-order calendar month.

Because month was encoded as a linear integer, Step 28 explicitly tested a more theoretically appropriate cyclical representation.

The experiment demonstrated that the mathematically more natural representation did not improve out-of-time validation performance.

The original calendar specification is therefore retained based on empirical evidence rather than convenience.

### Final Champion

The **Drop-First Interpretable Logistic Regression** is formally frozen as the final candidate model.

The frozen modelling specification is:

- **target:** second observable purchase within 90 days;
- **prediction point:** immediately after first observable order;
- **algorithm:** Logistic Regression;
- **categorical encoding:** drop-first one-hot encoding;
- **numeric preprocessing:** median imputation and standardisation;
- **calibration layer:** none;
- **primary model-selection metric:** PR-AUC;
- **commercial ranking metrics:** Lift@10% and Lift@20%;
- **validation strategy:** chronological;
- **final temporal test status:** unopened.

### Model-Governance Decision

No further feature engineering, algorithm selection, calibration selection or validation-driven tuning should occur after this point.

The model specification is now frozen.

The next stage will open the reserved future test population once and evaluate whether the predictive performance observed during development generalises to previously untouched customers.




## Step 29 — Final Untouched Temporal Test Evaluation

Step 29 performs the final out-of-time evaluation of the frozen 90-day repeat-purchase propensity model.

All model-selection decisions were completed before this stage.

The frozen specification is the **Drop-First Interpretable Logistic Regression**, selected using historical training and validation populations only.

The final temporal test population has not previously been used for:

- feature selection;
- algorithm selection;
- model comparison;
- calibration selection;
- hyperparameter tuning;
- threshold optimisation;
- or model interpretation.

Before scoring the test population, the frozen specification will be refitted using the combined Train + Validation development data.

This does not constitute additional model selection. The algorithm, features, preprocessing and hyperparameters remain unchanged.

The refitted model will then be evaluated once on the later, untouched test population.

### Primary Evaluation Questions

The final evaluation asks:

1. Does the model retain predictive discrimination on future customers?
2. Does top-ranked customer lift remain commercially useful?
3. Does probability quality remain acceptable?
4. How much performance deteriorates relative to validation?
5. Is the model sufficiently stable to justify continued use as a customer-prioritisation tool?

### Governance Principle

The test set is an evaluation dataset, not another development dataset.

If test performance is weak, the model will not be retuned against the test population.

The appropriate conclusion would instead be that the current modelling specification did not generalise sufficiently.

In [ ]:
# ================================================================
# 29.2 — CONFIRM MODEL FREEZE
# ================================================================

assert (
    MODEL_SPECIFICATION_FROZEN
    is True
), (
    "Model specification must be frozen before test evaluation."
)


assert (
    frozen_model_name
    ==
    "Drop-First Interpretable Logistic"
), (
    f"Unexpected frozen model: {frozen_model_name}"
)


print(
    f"Frozen model: {frozen_model_name}"
)

print(
    f"Frozen feature type: {frozen_feature_type}"
)

print(
    f"Number of pre-encoded features: "
    f"{len(frozen_features)}"
)

In [ ]:
# ================================================================
# 29.3 — FINAL SPECIFICATION RECORD
# ================================================================

final_model_specification = pd.DataFrame(
    {
        "Component": [
            "Business target",
            "Prediction point",
            "Prediction horizon",
            "Algorithm",
            "Numerical imputation",
            "Numerical scaling",
            "Categorical imputation",
            "Categorical encoding",
            "Calibration",
            "Primary selection metric",
            "Commercial ranking metrics",
            "Training design",
        ],

        "Final Decision": [
            "Second observable purchase within 90 days",
            "Immediately after first observable order",
            "90 days",
            "Logistic Regression",
            "Median",
            "StandardScaler",
            "Most frequent",
            "Drop-first one-hot encoding",
            "None",
            "PR-AUC",
            "Lift@10% and Lift@20%",
            "Chronological",
        ],
    }
)


display(
    final_model_specification
)

In [ ]:
# ================================================================
# 29.4 — PRE-SPECIFIED FINAL DECISION FRAMEWORK
# ================================================================

final_decision_framework = pd.DataFrame(
    {
        "Decision": [
            "GO",
            "CONDITIONAL GO",
            "NO-GO",
        ],

        "Interpretation": [
            (
                "Model preserves useful future-customer ranking with "
                "limited degradation from validation."
            ),

            (
                "Model retains some useful ranking signal but performance "
                "is materially weaker or unstable; use only for controlled "
                "pilots and monitoring."
            ),

            (
                "Model does not demonstrate sufficient out-of-time "
                "generalisation for operational use."
            ),
        ],
    }
)


display(
    final_decision_framework
)

In [ ]:
# ================================================================
# 29.4B — QUANTITATIVE DECISION GUIDES
# ================================================================

decision_thresholds = {
    "minimum_roc_auc_for_signal":
        0.55,

    "minimum_lift10_for_useful_ranking":
        1.20,

    "minimum_lift20_for_useful_ranking":
        1.10,

    "maximum_relative_pr_auc_degradation_for_go":
        0.15,

    "maximum_relative_pr_auc_degradation_for_conditional":
        0.30,
}


display(
    pd.DataFrame(
        {
            "Criterion":
                list(
                    decision_thresholds.keys()
                ),

            "Threshold":
                list(
                    decision_thresholds.values()
                ),
        }
    )
)

In [ ]:
# ================================================================
# 29.5 — COMBINE TRAIN + VALIDATION DEVELOPMENT DATA
# ================================================================

development_data = (
    pd.concat(
        [
            train_data,
            validation_data,
        ],
        ignore_index=True
    )
    .sort_values(
        "first_order_date"
    )
    .reset_index(
        drop=True
    )
)


print(
    f"Train customers: "
    f"{len(train_data):,}"
)

print(
    f"Validation customers: "
    f"{len(validation_data):,}"
)

print(
    f"Combined development customers: "
    f"{len(development_data):,}"
)

In [ ]:
assert (
    len(development_data)
    ==
    len(train_data)
    +
    len(validation_data)
)

In [ ]:
# ================================================================
# 29.6 — FINAL DEVELOPMENT MATRICES
# ================================================================

X_development = (
    development_data[
        model_features
    ]
    .copy()
)


y_development = (
    development_data[
        target_column
    ]
    .copy()
)


print(
    f"Development rows: "
    f"{len(X_development):,}"
)

print(
    f"Development repeat rate: "
    f"{y_development.mean() * 100:.2f}%"
)

In [ ]:
# ================================================================
# 29.7 — CREATE FINAL REFIT MODEL
# ================================================================

final_refit_model = clone(
    interpretable_logistic_model
)


print(
    "✓ Frozen model specification cloned."
)

In [ ]:
# ================================================================
# 29.8 — FINAL HISTORICAL REFIT
# ================================================================

final_refit_model.fit(
    X_development,
    y_development
)


print(
    "✓ Frozen logistic specification refitted "
    "using all development data."
)

In [ ]:
# ================================================================
# 29.9 — TEMPORAL SEPARATION CHECK
# ================================================================

development_end_date = (
    development_data[
        "first_order_date"
    ]
    .max()
)


test_start_date = (
    test_data[
        "first_order_date"
    ]
    .min()
)


print(
    f"Development period ends: "
    f"{development_end_date}"
)

print(
    f"Test period starts: "
    f"{test_start_date}"
)


assert (
    development_end_date
    <
    test_start_date
), (
    "Development and test periods overlap."
)


print(
    "✓ Development and test periods are chronologically separated."
)

In [ ]:
# ================================================================
# 29.10 — OPEN FINAL TEMPORAL TEST SET
# ================================================================

TEST_SET_OPENED = True


assert TEST_SET_OPENED is True


print(
    "FINAL TEST SET IS NOW OPEN."
)

print(
    "No further model tuning is permitted."
)

In [ ]:
# ================================================================
# 29.11 — FINAL TEST MATRICES
# ================================================================

X_test_final = (
    test_data[
        model_features
    ]
    .copy()
)


y_test_final = (
    test_data[
        target_column
    ]
    .copy()
)


print(
    f"Test customers: "
    f"{len(X_test_final):,}"
)

print(
    f"Observed test repeat rate: "
    f"{y_test_final.mean() * 100:.2f}%"
)

In [ ]:
# ================================================================
# 29.12 — FINAL TEST PREDICTIONS
# ================================================================

test_probabilities = (
    final_refit_model
    .predict_proba(
        X_test_final
    )[:, 1]
)


assert (
    len(
        test_probabilities
    )
    ==
    len(
        y_test_final
    )
)


assert np.all(
    (
        test_probabilities
        >= 0
    )
    &
    (
        test_probabilities
        <= 1
    )
)


print(
    "✓ Final test probabilities generated."
)

In [ ]:
# ================================================================
# 29.13 — FINAL TEST PERFORMANCE
# ================================================================

final_test_results = (
    evaluate_binary_model(
        model_name="Frozen Logistic — Final Test",
        y_true=y_test_final,
        probabilities=test_probabilities,
    )
)


display(
    final_test_results
)

In [ ]:
# ================================================================
# 29.14 — PRE-TEST VALIDATION BENCHMARK
# ================================================================

pretest_validation_benchmark = (
    interpretable_results
    .copy()
)


pretest_validation_benchmark[
    "Evaluation Population"
] = (
    "Validation"
)


display(
    pretest_validation_benchmark
)

In [ ]:
# ================================================================
# 29.15 — VALIDATION VS TEST COMPARISON
# ================================================================

validation_vs_test = pd.DataFrame(
    {
        "Metric": [
            "PR-AUC",
            "ROC-AUC",
            "Brier Score",
            "Lift @ 10%",
            "Lift @ 20%",
            "Capture @ 10%",
            "Capture @ 20%",
        ],

        "Validation": [
            interpretable_results[
                "PR-AUC"
            ].iloc[0],

            interpretable_results[
                "ROC-AUC"
            ].iloc[0],

            interpretable_results[
                "Brier Score"
            ].iloc[0],

            interpretable_results[
                "Lift @ 10%"
            ].iloc[0],

            interpretable_results[
                "Lift @ 20%"
            ].iloc[0],

            interpretable_results[
                "Capture @ 10%"
            ].iloc[0],

            interpretable_results[
                "Capture @ 20%"
            ].iloc[0],
        ],

        "Test": [
            final_test_results[
                "PR-AUC"
            ].iloc[0],

            final_test_results[
                "ROC-AUC"
            ].iloc[0],

            final_test_results[
                "Brier Score"
            ].iloc[0],

            final_test_results[
                "Lift @ 10%"
            ].iloc[0],

            final_test_results[
                "Lift @ 20%"
            ].iloc[0],

            final_test_results[
                "Capture @ 10%"
            ].iloc[0],

            final_test_results[
                "Capture @ 20%"
            ].iloc[0],
        ],
    }
)

In [ ]:
validation_vs_test[
    "Absolute Change"
] = (
    validation_vs_test[
        "Test"
    ]
    -
    validation_vs_test[
        "Validation"
    ]
)

In [ ]:
display(
    validation_vs_test
)

In [ ]:
# ================================================================
# 29.16 — RELATIVE VALIDATION-TO-TEST CHANGE
# ================================================================

validation_vs_test[
    "Relative Change (%)"
] = np.where(
    validation_vs_test[
        "Validation"
    ]
    .ne(0),

    (
        validation_vs_test[
            "Test"
        ]
        -
        validation_vs_test[
            "Validation"
        ]
    )
    /
    validation_vs_test[
        "Validation"
    ]
    * 100,

    np.nan
)


display(
    validation_vs_test
)

In [ ]:
# ================================================================
# VISUAL 29.1 — VALIDATION VS TEST
# ================================================================

higher_is_better_metrics = (
    validation_vs_test.loc[
        validation_vs_test[
            "Metric"
        ].isin(
            [
                "PR-AUC",
                "ROC-AUC",
                "Lift @ 10%",
                "Lift @ 20%",
            ]
        )
    ]
    .copy()
)


validation_test_plot = (
    higher_is_better_metrics
    .melt(
        id_vars="Metric",
        value_vars=[
            "Validation",
            "Test"
        ],
        var_name="Dataset",
        value_name="Score"
    )
)


plt.figure(
    figsize=(11, 6)
)


sns.barplot(
    data=validation_test_plot,
    x="Metric",
    y="Score",
    hue="Dataset"
)


plt.title(
    "Validation vs Final Temporal Test Performance"
)

plt.xlabel(
    ""
)

plt.ylabel(
    "Metric Value"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 29.17 — FINAL TEST DECILE PERFORMANCE
# ================================================================

final_test_decile_performance = (
    create_decile_performance(
        y_true=y_test_final,
        probabilities=test_probabilities,
        n_deciles=10
    )
)


display(
    final_test_decile_performance
)

In [ ]:
# ================================================================
# VISUAL 29.2 — FINAL TEST DECILE SEPARATION
# ================================================================

plt.figure(
    figsize=(11, 6)
)


sns.barplot(
    data=final_test_decile_performance,
    x="score_decile",
    y="actual_repeat_rate_pct"
)


plt.axhline(
    y=y_test_final.mean() * 100,
    linestyle="--",
    label=(
        f"Overall test repeat rate "
        f"({y_test_final.mean() * 100:.1f}%)"
    )
)


plt.title(
    "Final Test — Actual 90-Day Repeat Rate by Score Decile"
)

plt.xlabel(
    "Score Decile — 1 = Highest Predicted Propensity"
)

plt.ylabel(
    "Actual Repeat Rate (%)"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 29.18 — FINAL TEST CUMULATIVE GAINS
# ================================================================

test_scored = pd.DataFrame(
    {
        "actual":
            np.asarray(
                y_test_final
            ),

        "probability":
            test_probabilities,
    }
)


test_scored = (
    test_scored
    .sort_values(
        "probability",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


test_scored[
    "cumulative_customers"
] = np.arange(
    1,
    len(test_scored) + 1
)


test_scored[
    "cumulative_customer_pct"
] = (
    test_scored[
        "cumulative_customers"
    ]
    /
    len(test_scored)
    * 100
)


test_scored[
    "cumulative_repeaters"
] = (
    test_scored[
        "actual"
    ]
    .cumsum()
)


total_test_repeaters = (
    test_scored[
        "actual"
    ]
    .sum()
)


test_scored[
    "cumulative_repeater_capture_pct"
] = (
    test_scored[
        "cumulative_repeaters"
    ]
    /
    total_test_repeaters
    * 100
)

In [ ]:
# ================================================================
# VISUAL 29.3 — FINAL TEST CUMULATIVE GAINS
# ================================================================

plt.figure(
    figsize=(8, 6)
)


plt.plot(
    test_scored[
        "cumulative_customer_pct"
    ],
    test_scored[
        "cumulative_repeater_capture_pct"
    ],
    label="Frozen Logistic Model"
)


plt.plot(
    [0, 100],
    [0, 100],
    linestyle="--",
    label="Random Targeting"
)


plt.title(
    "Final Test — Cumulative Repeat-Purchaser Capture"
)

plt.xlabel(
    "Customers Targeted (%)"
)

plt.ylabel(
    "Repeat Purchasers Captured (%)"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 29.4 — FINAL TEST PRECISION-RECALL
# ================================================================

test_precision, test_recall, _ = (
    precision_recall_curve(
        y_test_final,
        test_probabilities
    )
)


plt.figure(
    figsize=(8, 6)
)


plt.plot(
    test_recall,
    test_precision,
    label="Frozen Logistic Model"
)


plt.axhline(
    y=y_test_final.mean(),
    linestyle="--",
    label=(
        f"Test prevalence "
        f"({y_test_final.mean():.3f})"
    )
)


plt.title(
    "Final Test — Precision-Recall Curve"
)

plt.xlabel(
    "Recall"
)

plt.ylabel(
    "Precision"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 29.5 — FINAL TEST ROC CURVE
# ================================================================

test_fpr, test_tpr, _ = (
    roc_curve(
        y_test_final,
        test_probabilities
    )
)


plt.figure(
    figsize=(8, 6)
)


plt.plot(
    test_fpr,
    test_tpr,
    label=(
        f"ROC-AUC = "
        f"{roc_auc_score(y_test_final, test_probabilities):.3f}"
    )
)


plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Ranking"
)


plt.title(
    "Final Test — ROC Curve"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 29.21 — FINAL TEST CALIBRATION
# ================================================================

test_observed_rate, test_predicted_rate = (
    calibration_curve(
        y_test_final,
        test_probabilities,
        n_bins=10,
        strategy="quantile"
    )
)


test_calibration_table = pd.DataFrame(
    {
        "mean_predicted_probability":
            test_predicted_rate,

        "observed_repeat_rate":
            test_observed_rate,
    }
)


test_calibration_table[
    "calibration_error"
] = (
    test_calibration_table[
        "mean_predicted_probability"
    ]
    -
    test_calibration_table[
        "observed_repeat_rate"
    ]
)


test_calibration_table[
    "absolute_calibration_error"
] = (
    test_calibration_table[
        "calibration_error"
    ]
    .abs()
)


display(
    test_calibration_table
)

In [ ]:
# ================================================================
# 29.22 — FINAL TEST CALIBRATION ERROR
# ================================================================

test_mean_absolute_calibration_error_pp = (
    test_calibration_table[
        "absolute_calibration_error"
    ]
    .mean()
    * 100
)


print(
    "Mean absolute test calibration error: "
    f"{test_mean_absolute_calibration_error_pp:.2f} pp"
)

In [ ]:
# ================================================================
# VISUAL 29.6 — FINAL TEST CALIBRATION CURVE
# ================================================================

plt.figure(
    figsize=(8, 6)
)


plt.plot(
    test_predicted_rate,
    test_observed_rate,
    marker="o",
    label="Frozen Logistic"
)


plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect Calibration"
)


plt.title(
    "Final Test — Probability Calibration"
)

plt.xlabel(
    "Mean Predicted Probability"
)

plt.ylabel(
    "Observed Repeat Rate"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 29.23 — BOOTSTRAP METRIC CONFIDENCE INTERVALS
# ================================================================

def bootstrap_binary_metrics(
    y_true,
    probabilities,
    n_bootstrap=2000,
    random_state=42
):
    """
    Bootstrap confidence intervals for final test metrics.

    Resamples customers with replacement.
    Bootstrap samples containing only one target class
    are skipped.
    """

    y_true = np.asarray(
        y_true
    )

    probabilities = np.asarray(
        probabilities
    )


    rng = np.random.default_rng(
        random_state
    )


    n = len(
        y_true
    )


    records = []


    for _ in range(
        n_bootstrap
    ):

        sample_indices = (
            rng.integers(
                0,
                n,
                size=n
            )
        )


        y_sample = (
            y_true[
                sample_indices
            ]
        )


        probability_sample = (
            probabilities[
                sample_indices
            ]
        )


        if (
            np.unique(
                y_sample
            ).size
            < 2
        ):
            continue


        records.append(
            {
                "PR-AUC":
                    average_precision_score(
                        y_sample,
                        probability_sample
                    ),

                "ROC-AUC":
                    roc_auc_score(
                        y_sample,
                        probability_sample
                    ),

                "Brier Score":
                    brier_score_loss(
                        y_sample,
                        probability_sample
                    ),

                "Lift @ 10%":
                    lift_at_fraction(
                        y_sample,
                        probability_sample,
                        0.10
                    ),

                "Lift @ 20%":
                    lift_at_fraction(
                        y_sample,
                        probability_sample,
                        0.20
                    ),
            }
        )


    return pd.DataFrame(
        records
    )

In [ ]:
test_bootstrap_metrics = (
    bootstrap_binary_metrics(
        y_true=y_test_final,
        probabilities=test_probabilities,
        n_bootstrap=2000,
        random_state=42
    )
)


print(
    f"Valid bootstrap samples: "
    f"{len(test_bootstrap_metrics):,}"
)

In [ ]:
# ================================================================
# 29.24 — BOOTSTRAP 95% CONFIDENCE INTERVALS
# ================================================================

bootstrap_summary_records = []


for metric in [
    "PR-AUC",
    "ROC-AUC",
    "Brier Score",
    "Lift @ 10%",
    "Lift @ 20%",
]:

    bootstrap_summary_records.append(
        {
            "Metric":
                metric,

            "Estimate":
                final_test_results[
                    metric
                ]
                .iloc[0],

            "Bootstrap Mean":
                test_bootstrap_metrics[
                    metric
                ]
                .mean(),

            "CI 2.5%":
                test_bootstrap_metrics[
                    metric
                ]
                .quantile(
                    0.025
                ),

            "CI 97.5%":
                test_bootstrap_metrics[
                    metric
                ]
                .quantile(
                    0.975
                ),
        }
    )


test_bootstrap_summary = pd.DataFrame(
    bootstrap_summary_records
)


display(
    test_bootstrap_summary
)

In [ ]:
# ================================================================
# 29.25 — DEVELOPMENT → TEST FEATURE DRIFT
# ================================================================

test_drift_records = []


for feature in numeric_features:

    development_values = pd.to_numeric(
        X_development[
            feature
        ],
        errors="coerce"
    )


    test_values = pd.to_numeric(
        X_test_final[
            feature
        ],
        errors="coerce"
    )


    development_mean = (
        development_values.mean()
    )


    test_mean = (
        test_values.mean()
    )


    development_std = (
        development_values.std()
    )


    test_std = (
        test_values.std()
    )


    pooled_std = np.sqrt(
        (
            development_std ** 2
            +
            test_std ** 2
        )
        / 2
    )


    if (
        pd.isna(
            pooled_std
        )
        or
        pooled_std == 0
    ):

        smd = np.nan

    else:

        smd = (
            test_mean
            -
            development_mean
        ) / pooled_std


    test_drift_records.append(
        {
            "feature":
                feature,

            "development_mean":
                development_mean,

            "test_mean":
                test_mean,

            "SMD":
                smd,

            "absolute_SMD":
                abs(
                    smd
                )
                if not pd.isna(
                    smd
                )
                else np.nan,
        }
    )


development_test_feature_drift = (
    pd.DataFrame(
        test_drift_records
    )
    .sort_values(
        "absolute_SMD",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    development_test_feature_drift
)

In [ ]:
# ================================================================
# VISUAL 29.7 — DEVELOPMENT → TEST FEATURE DRIFT
# ================================================================

plt.figure(
    figsize=(11, 8)
)


sns.barplot(
    data=(
        development_test_feature_drift
        .sort_values(
            "absolute_SMD"
        )
    ),
    y="feature",
    x="absolute_SMD"
)


plt.axvline(
    0.10,
    linestyle="--",
    label="0.10"
)


plt.axvline(
    0.25,
    linestyle="--",
    label="0.25"
)


plt.title(
    "Development-to-Test Numeric Feature Drift"
)

plt.xlabel(
    "Absolute Standardised Mean Difference"
)

plt.ylabel(
    ""
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 29.26 — FINAL DECISION METRICS
# ================================================================

validation_pr_auc = (
    interpretable_results[
        "PR-AUC"
    ]
    .iloc[0]
)


test_pr_auc = (
    final_test_results[
        "PR-AUC"
    ]
    .iloc[0]
)


test_roc_auc = (
    final_test_results[
        "ROC-AUC"
    ]
    .iloc[0]
)


test_lift10 = (
    final_test_results[
        "Lift @ 10%"
    ]
    .iloc[0]
)


test_lift20 = (
    final_test_results[
        "Lift @ 20%"
    ]
    .iloc[0]
)


relative_pr_auc_degradation = (
    (
        validation_pr_auc
        -
        test_pr_auc
    )
    /
    validation_pr_auc
)

In [ ]:
print(
    f"Validation PR-AUC: "
    f"{validation_pr_auc:.4f}"
)

print(
    f"Test PR-AUC: "
    f"{test_pr_auc:.4f}"
)

print(
    f"Relative PR-AUC degradation: "
    f"{relative_pr_auc_degradation * 100:.2f}%"
)

print(
    f"Test ROC-AUC: "
    f"{test_roc_auc:.4f}"
)

print(
    f"Test Lift@10%: "
    f"{test_lift10:.4f}"
)

print(
    f"Test Lift@20%: "
    f"{test_lift20:.4f}"
)

In [ ]:
# ================================================================
# 29.27 — FINAL MODEL DECISION SUPPORT
# ================================================================

has_basic_discrimination = (
    test_roc_auc
    >=
    decision_thresholds[
        "minimum_roc_auc_for_signal"
    ]
)


has_useful_top10_lift = (
    test_lift10
    >=
    decision_thresholds[
        "minimum_lift10_for_useful_ranking"
    ]
)


has_useful_top20_lift = (
    test_lift20
    >=
    decision_thresholds[
        "minimum_lift20_for_useful_ranking"
    ]
)


limited_pr_auc_degradation = (
    relative_pr_auc_degradation
    <=
    decision_thresholds[
        "maximum_relative_pr_auc_degradation_for_go"
    ]
)


moderate_pr_auc_degradation = (
    relative_pr_auc_degradation
    <=
    decision_thresholds[
        "maximum_relative_pr_auc_degradation_for_conditional"
    ]
)

In [ ]:
if (
    has_basic_discrimination
    and
    has_useful_top10_lift
    and
    has_useful_top20_lift
    and
    limited_pr_auc_degradation
):

    final_model_decision = (
        "GO"
    )


elif (
    has_basic_discrimination
    and
    (
        has_useful_top10_lift
        or
        has_useful_top20_lift
    )
    and
    moderate_pr_auc_degradation
):

    final_model_decision = (
        "CONDITIONAL GO"
    )


else:

    final_model_decision = (
        "NO-GO"
    )


print(
    f"FINAL MODEL DECISION: "
    f"{final_model_decision}"
)

In [ ]:
# ================================================================
# 29.28 — FINAL DECISION EVIDENCE
# ================================================================

final_decision_evidence = pd.DataFrame(
    {
        "Criterion": [
            "ROC-AUC >= minimum signal threshold",
            "Lift@10% >= useful targeting threshold",
            "Lift@20% >= useful targeting threshold",
            "PR-AUC degradation acceptable for GO",
            "PR-AUC degradation acceptable for Conditional Go",
        ],

        "Observed": [
            test_roc_auc,
            test_lift10,
            test_lift20,
            relative_pr_auc_degradation,
            relative_pr_auc_degradation,
        ],

        "Threshold": [
            decision_thresholds[
                "minimum_roc_auc_for_signal"
            ],

            decision_thresholds[
                "minimum_lift10_for_useful_ranking"
            ],

            decision_thresholds[
                "minimum_lift20_for_useful_ranking"
            ],

            decision_thresholds[
                "maximum_relative_pr_auc_degradation_for_go"
            ],

            decision_thresholds[
                "maximum_relative_pr_auc_degradation_for_conditional"
            ],
        ],

        "Passed": [
            has_basic_discrimination,
            has_useful_top10_lift,
            has_useful_top20_lift,
            limited_pr_auc_degradation,
            moderate_pr_auc_degradation,
        ],
    }
)


display(
    final_decision_evidence
)

### Interpretation of the Final Model Decision

**GO** means that the frozen model demonstrates sufficiently consistent out-of-time ranking performance to justify proceeding to a controlled operational pilot.

It does not imply that the model has proven causal business value.

**CONDITIONAL GO** means that the model retains useful signal but exhibits sufficient degradation or instability that it should only be used within a monitored pilot with conservative expectations.

**NO-GO** means that the frozen model did not generalise sufficiently to future customers and should not be operationalised in its current form.

Regardless of the statistical decision, the model should initially support prioritisation rather than fully automated customer treatment.

A campaign or intervention experiment would still be required to determine whether acting on the scores causes an improvement in repeat purchasing.

In [ ]:
# ================================================================
# 29.30 — TEST DECILE CALIBRATION
# ================================================================

final_test_decile_calibration = (
    final_test_decile_performance
    .copy()
)


final_test_decile_calibration[
    "calibration_error_pp"
] = (
    final_test_decile_calibration[
        "average_predicted_probability_pct"
    ]
    -
    final_test_decile_calibration[
        "actual_repeat_rate_pct"
    ]
)


final_test_decile_calibration[
    "absolute_calibration_error_pp"
] = (
    final_test_decile_calibration[
        "calibration_error_pp"
    ]
    .abs()
)


display(
    final_test_decile_calibration
)

In [ ]:
# ================================================================
# 29.31 — FINAL MODEL CARD
# ================================================================

final_model_card = pd.DataFrame(
    {
        "Item": [
            "Model",
            "Objective",
            "Prediction unit",
            "Target",
            "Horizon",
            "Final training population",
            "Final evaluation population",
            "Validation strategy",
            "Primary metric",
            "Primary commercial metrics",
            "Calibration",
            "Recommended use",
            "Not recommended for",
            "Final decision",
        ],

        "Value": [
            "Drop-First Interpretable Logistic Regression",

            (
                "Predict 90-day repeat purchase immediately "
                "after first observable order"
            ),

            "One observable newly acquired customer",

            "Second purchase within 90 days",

            "90 days",

            (
                "Combined chronological Train + Validation "
                "development population"
            ),

            "Previously untouched later temporal Test population",

            "Chronological out-of-time validation",

            "PR-AUC",

            "Lift@10%, Lift@20%, repeater capture",

            "No post-hoc calibration layer",

            (
                "Customer prioritisation and controlled "
                "retention experimentation"
            ),

            (
                "Causal treatment decisions, guaranteed "
                "individual probabilities, or permanent churn inference"
            ),

            final_model_decision,
        ],
    }
)


display(
    final_model_card
)

In [ ]:
# ================================================================
# 29.32 — TEST GOVERNANCE LOCK
# ================================================================

TEST_SET_CONSUMED = True


assert TEST_SET_OPENED
assert TEST_SET_CONSUMED
assert MODEL_SPECIFICATION_FROZEN


print(
    "✓ Final test population has now been consumed."
)

print(
    "✓ It must not be reused for model tuning."
)

print(
    "✓ Any future redevelopment requires a new future holdout "
    "or subsequent production data."
)

In [ ]:
# ================================================================
# 29.33 — STEP 29 COMPLETION CHECKS
# ================================================================

assert (
    len(
        test_probabilities
    )
    ==
    len(
        y_test_final
    )
)


assert (
    final_test_results[
        "PR-AUC"
    ]
    .notna()
    .all()
)


assert (
    final_test_results[
        "ROC-AUC"
    ]
    .notna()
    .all()
)


assert (
    final_test_results[
        "Brier Score"
    ]
    .notna()
    .all()
)


assert (
    final_model_decision
    in [
        "GO",
        "CONDITIONAL GO",
        "NO-GO",
    ]
)


print(
    "✓ Step 29 complete."
)

print(
    "✓ Frozen specification refitted on development data."
)

print(
    "✓ Untouched temporal test set evaluated."
)

print(
    "✓ Bootstrap uncertainty estimated."
)

print(
    "✓ Final model decision generated."
)

print(
    "✓ Test set permanently retired from model development."
)

In [ ]:
# ================================================================
# 29.34 — PREVALENCE-ADJUSTED PR-AUC DIAGNOSTIC
# ================================================================

validation_prevalence = (
    y_validation.mean()
)

test_prevalence = (
    y_test_final.mean()
)


validation_pr_auc = (
    interpretable_results[
        "PR-AUC"
    ]
    .iloc[0]
)


test_pr_auc = (
    final_test_results[
        "PR-AUC"
    ]
    .iloc[0]
)

In [ ]:
validation_pr_auc_lift_over_prevalence = (
    validation_pr_auc
    /
    validation_prevalence
)


test_pr_auc_lift_over_prevalence = (
    test_pr_auc
    /
    test_prevalence
)

In [ ]:
validation_normalised_pr_auc = (
    (
        validation_pr_auc
        -
        validation_prevalence
    )
    /
    (
        1
        -
        validation_prevalence
    )
)


test_normalised_pr_auc = (
    (
        test_pr_auc
        -
        test_prevalence
    )
    /
    (
        1
        -
        test_prevalence
    )
)

In [ ]:
prevalence_adjusted_pr_auc = pd.DataFrame(
    {
        "Dataset": [
            "Validation",
            "Test",
        ],

        "Repeat Prevalence": [
            validation_prevalence,
            test_prevalence,
        ],

        "PR-AUC": [
            validation_pr_auc,
            test_pr_auc,
        ],

        "PR-AUC / Prevalence": [
            validation_pr_auc_lift_over_prevalence,
            test_pr_auc_lift_over_prevalence,
        ],

        "Normalised PR-AUC": [
            validation_normalised_pr_auc,
            test_normalised_pr_auc,
        ],
    }
)


display(
    prevalence_adjusted_pr_auc
)

### Step 29 Summary — Final Untouched Temporal Test Evaluation

Step 29 performed the first and only evaluation of the frozen repeat-purchase model on the previously untouched future test population.

Before evaluation, the frozen Drop-First Interpretable Logistic Regression specification was refitted using the combined Train and Validation development population.

No features, algorithms, calibration methods or hyperparameters were changed after the model specification was frozen.

The test population was therefore used strictly as an out-of-time final evaluation dataset.

### The Model Retained Predictive Signal, but Ranking Performance Weakened

The final test model achieved approximately:

- **PR-AUC: 0.503**
- **ROC-AUC: 0.560**
- **Brier score: 0.248**
- **top-10% lift: 1.03×**
- **top-20% lift: 1.11×**
- **top-10% repeater capture: 10.37%**
- **top-20% repeater capture: 22.26%**

ROC-AUC remained above random performance, confirming that the model retained some ability to differentiate future repeaters from non-repeaters.

However, discrimination was weaker than during validation.

ROC-AUC declined from approximately **0.592 during validation to 0.560 in the final test period**.

More importantly, the strong top-of-ranking performance observed during model development did not fully generalise.

Top-decile lift declined from approximately **1.43× to 1.03×**, while top-20% lift declined from approximately **1.33× to 1.11×**.

The model therefore retained broad predictive signal but lost much of its earlier ability to concentrate repeat purchasers within the very highest-scored group.

### Raw PR-AUC Improvement Must Be Interpreted Carefully

PR-AUC increased from approximately **0.408 during validation to 0.503 on the final test population**.

This should not be interpreted as evidence that ranking performance improved.

The underlying repeat-purchase prevalence increased substantially in the future test population, from approximately **32.6% during validation to approximately 45.9% in the test period**.

Because PR-AUC is sensitive to positive-class prevalence, part of the increase reflects the higher baseline frequency of repeat purchasing rather than stronger model discrimination.

ROC-AUC, lift and prevalence-adjusted PR-AUC measures therefore provide a more appropriate interpretation of the change in ranking quality.

### Top-Decile Targeting Did Not Generalise Reliably

The highest-ranked 10% of test customers achieved approximately **1.03× lift**, only slightly above the overall test repeat rate.

The bootstrap 95% confidence interval for top-decile lift was approximately **0.84× to 1.33×**, which includes random-performance lift of 1.0.

The evidence therefore does not support deploying the model as a high-confidence top-decile targeting mechanism.

The highest-ranked 20% performed somewhat better, producing approximately **1.11× lift** and capturing approximately **22.3% of all test repeat purchasers**.

However, the bootstrap 95% confidence interval for top-20% lift was approximately **0.94× to 1.27×**, which also includes 1.0.

The model's commercial targeting advantage should therefore be regarded as promising but unproven.

### The Model May Be More Useful for Identifying Lower-Propensity Customers

The final test-decile distribution was not perfectly monotonic.

Several middle and upper deciles produced similar repeat rates, limiting precise ordering among higher-propensity customers.

However, the lowest-scored decile recorded an actual repeat rate of approximately **33.3%**, materially below the approximately **45.9% overall test repeat rate**.

This suggests that the model may be more useful for identifying customers at elevated risk of failing to repeat than for identifying an exceptionally strong highest-propensity group.

This distinction should influence how the model is tested commercially.

### Probability Calibration Improved

Although ranking performance weakened, final test probability calibration improved materially.

Mean absolute decile calibration error was approximately **4.71 percentage points**, compared with approximately **9.74 percentage points during validation**.

The final model therefore produced probability levels that were more closely aligned with observed repeat rates in the future population.

This reinforces the distinction between ranking performance and probability calibration: a model may estimate broad outcome rates reasonably well while still providing only modest customer ordering.

### Test Uncertainty Requires Cautious Interpretation

Bootstrap analysis produced the following approximate 95% confidence intervals:

- **PR-AUC: 0.453–0.562**
- **ROC-AUC: 0.517–0.602**
- **Lift@10%: 0.84×–1.33×**
- **Lift@20%: 0.94×–1.27×**

The ROC-AUC interval remains above 0.50, supporting the presence of genuine but modest predictive signal.

However, both lift intervals include 1.0.

The retrospective evidence therefore does not establish a statistically robust commercial targeting advantage at the 10% or 20% targeting levels.

### Feature Drift Was Limited

Development-to-test feature drift was generally small.

The largest observed standardised mean difference was approximately **0.11 for first-order calendar month**.

Most economic and basket-related variables remained below approximately **0.08 absolute SMD**.

The deterioration in ranking performance therefore does not appear to result from severe changes in the measured customer-feature distribution.

Instead, the relationship between first-order behaviour and subsequent repeat purchasing appears to vary across customer cohorts and time periods.

This is consistent with the temporal-instability findings identified during model development.

### Final Model Decision — CONDITIONAL GO

The predetermined evaluation framework produced a final decision of:

> **CONDITIONAL GO**

The model retained ROC-AUC above the minimum predictive-signal threshold and top-20% lift marginally above the predetermined useful-ranking threshold.

However, top-10% lift failed the required threshold and the confidence intervals around commercial lift remain wide.

The model should therefore not be deployed as an autonomous decision engine or treated as proven evidence that high-scored customers will respond to intervention.

Instead, it is suitable for a **controlled commercial pilot**.

### Recommended Use

The model should initially be used as a propensity and risk-prioritisation layer within a controlled retention experiment.

A particularly promising use case is identifying first-time customers with relatively low predicted repeat propensity and testing whether targeted post-purchase interventions improve their second-purchase rate.

Randomised control groups should be maintained so that the business can distinguish:

- customers who would have repeated naturally;
- customers who respond because of the intervention;
- and customers for whom intervention generates no incremental benefit.

The next analytical question is therefore no longer simply:

> **Can we predict repeat purchasing?**

It becomes:

> **Can acting on these predictions causally increase repeat purchasing enough to create economic value?**

## Step 30 — Business Deployment Strategy, Controlled Experiment Design and Final Notebook Conclusions

The predictive modelling phase has established that first-order customer behaviour contains modest but genuine information about future 90-day repeat purchasing.

The final frozen Logistic Regression model retained out-of-time predictive signal, but its commercial ranking advantage weakened materially in the untouched future test population.

The final model decision was therefore **CONDITIONAL GO** rather than unrestricted deployment.

Step 30 converts that conclusion into an operational strategy.

The objective is not to deploy the model as an autonomous decision engine.

Instead, the model should be used as a customer-prioritisation layer within a controlled experiment designed to answer a more commercially important question:

> **Can intervention targeted using predicted repeat propensity causally increase 90-day repeat purchasing and incremental customer revenue?**

This final step defines:

- the recommended use of the model;
- the customer groups that should be prioritised;
- the intervention experiment;
- treatment and control design;
- commercial and statistical success metrics;
- deployment safeguards;
- model-monitoring requirements;
- known limitations;
- and the final conclusions of the repeat-purchase modelling notebook.

In [ ]:
# ================================================================
# 30.2 — FINAL ANALYTICAL CONCLUSION REGISTER
# ================================================================

final_analytical_conclusions = pd.DataFrame(
    [
        {
            "Conclusion":
                "First-order customer behaviour contains predictive signal.",

            "Evidence":
                "Final temporal test ROC-AUC approximately 0.560 with "
                "bootstrap interval above 0.50.",

            "Strength":
                "Supported",

            "Operational Meaning":
                "The score contains more information than random ranking."
        },

        {
            "Conclusion":
                "The model strongly identifies the highest-propensity 10%.",

            "Evidence":
                "Final test Lift@10% approximately 1.03 and bootstrap "
                "confidence interval includes 1.0.",

            "Strength":
                "Not supported",

            "Operational Meaning":
                "Do not use a top-10%-only targeting strategy."
        },

        {
            "Conclusion":
                "The highest-ranked 20% retains some positive ranking value.",

            "Evidence":
                "Final test Lift@20% approximately 1.11.",

            "Strength":
                "Partially supported",

            "Operational Meaning":
                "Potential targeting value exists but must be proven experimentally."
        },

        {
            "Conclusion":
                "The model identifies a lower-propensity customer segment.",

            "Evidence":
                "Lowest test decile repeated materially below the overall "
                "test repeat rate.",

            "Strength":
                "Supported",

            "Operational Meaning":
                "Low-score customers are credible candidates for retention intervention."
        },

        {
            "Conclusion":
                "Model probabilities are perfectly calibrated.",

            "Evidence":
                "Final test calibration improved, but residual calibration "
                "error remains.",

            "Strength":
                "Not supported",

            "Operational Meaning":
                "Treat outputs primarily as propensity scores rather than exact individual probabilities."
        },

        {
            "Conclusion":
                "Model performance is stable across all periods.",

            "Evidence":
                "Temporal validation and final testing showed meaningful "
                "variation in discrimination and lift.",

            "Strength":
                "Not supported",

            "Operational Meaning":
                "Performance monitoring is mandatory after deployment."
        },

        {
            "Conclusion":
                "The model causes customers to repeat.",

            "Evidence":
                "The model is observational and predictive only.",

            "Strength":
                "Unsupported causal claim",

            "Operational Meaning":
                "A randomised experiment is required before claiming business impact."
        },
    ]
)


display(
    final_analytical_conclusions
)

### Recommended Commercial Use Case

The recommended initial use of the repeat-purchase model is **retention-risk prioritisation**.

Rather than allocating intervention budget to customers who already appear highly likely to repeat naturally, the business should use the model to identify customers whose first-order characteristics are associated with relatively low repeat propensity.

These customers become candidates for controlled post-purchase intervention.

The objective is not to maximise the observed repeat rate of the treatment population.

The objective is to maximise **incremental repeat purchasing caused by the intervention**.

This distinction is critical.

A customer who has a 70% natural repeat probability but receives a promotion and repeats does not automatically represent campaign success.

The correct question is:

> Would this customer have repeated anyway?

Only a randomised control group can estimate that counterfactual.

In [ ]:
# ================================================================
# 30.4 — SCORE DEVELOPMENT POPULATION
# ================================================================

development_probabilities = (
    final_refit_model
    .predict_proba(
        X_development
    )[:, 1]
)


development_scored = (
    development_data[
        [
            "customer",
            "first_order_date",
            target_column,
        ]
    ]
    .copy()
)


development_scored[
    "repeat_propensity_score"
] = (
    development_probabilities
)

In [ ]:
# ================================================================
# 30.5 — DEVELOPMENT PROPENSITY THRESHOLDS
# ================================================================

propensity_thresholds = (
    development_scored[
        "repeat_propensity_score"
    ]
    .quantile(
        [
            0.25,
            0.50,
            0.75,
        ]
    )
)


low_threshold = (
    propensity_thresholds.loc[
        0.25
    ]
)

median_threshold = (
    propensity_thresholds.loc[
        0.50
    ]
)

high_threshold = (
    propensity_thresholds.loc[
        0.75
    ]
)


print(
    f"25th percentile: {low_threshold:.4f}"
)

print(
    f"50th percentile: {median_threshold:.4f}"
)

print(
    f"75th percentile: {high_threshold:.4f}"
)

In [ ]:
# ================================================================
# 30.6 — PROPENSITY SEGMENT FUNCTION
# ================================================================

def assign_propensity_segment(
    probability,
    q25,
    q50,
    q75
):

    if probability <= q25:

        return "Low Propensity"

    elif probability <= q50:

        return "Lower-Middle Propensity"

    elif probability <= q75:

        return "Upper-Middle Propensity"

    else:

        return "High Propensity"

In [ ]:
development_scored[
    "propensity_segment"
] = (
    development_scored[
        "repeat_propensity_score"
    ]
    .apply(
        lambda x:
        assign_propensity_segment(
            x,
            low_threshold,
            median_threshold,
            high_threshold
        )
    )
)

In [ ]:
# ================================================================
# 30.7 — DEVELOPMENT SEGMENT SUMMARY
# ================================================================

development_segment_summary = (
    development_scored
    .groupby(
        "propensity_segment",
        as_index=False
    )
    .agg(
        customers=(
            "customer",
            "size"
        ),

        actual_repeat_rate=(
            target_column,
            "mean"
        ),

        average_score=(
            "repeat_propensity_score",
            "mean"
        ),
    )
)


development_segment_summary[
    "actual_repeat_rate_pct"
] = (
    development_segment_summary[
        "actual_repeat_rate"
    ]
    * 100
)


development_segment_summary[
    "average_score_pct"
] = (
    development_segment_summary[
        "average_score"
    ]
    * 100
)


display(
    development_segment_summary
)

In [ ]:
# ================================================================
# 30.8 — PILOT ELIGIBILITY DEFINITION
# ================================================================

pilot_eligible_segments = [
    "Low Propensity",
    "Lower-Middle Propensity",
]


development_scored[
    "pilot_eligible"
] = (
    development_scored[
        "propensity_segment"
    ]
    .isin(
        pilot_eligible_segments
    )
)

### Pilot Experimental Design

The recommended pilot uses a randomised controlled design within the model-selected intervention population.

Immediately after a customer's first observable purchase:

1. the frozen propensity model generates a repeat-purchase score;
2. the customer is assigned to a fixed propensity segment;
3. customers in the intervention-eligible segments enter the experiment;
4. eligible customers are randomly assigned to Treatment or Control;
5. Treatment customers receive the defined retention intervention;
6. Control customers continue under business-as-usual treatment;
7. both groups are observed for 90 days.

Randomisation should occur **after model scoring and eligibility selection**.

This design allows the business to answer:

> Among customers selected by the model, does intervention causally increase repeat purchasing?

In [ ]:
# ================================================================
# 30.10 — EXPERIMENT ARM REGISTER
# ================================================================

experiment_arm_register = pd.DataFrame(
    [
        {
            "Arm":
                "Treatment",

            "Eligibility":
                "Model-selected low/lower-middle propensity customers",

            "Experience":
                "Defined post-purchase retention intervention",

            "Primary Observation":
                "Repeat purchase within 90 days"
        },

        {
            "Arm":
                "Control",

            "Eligibility":
                "Same model-selected population",

            "Experience":
                "Business as usual",

            "Primary Observation":
                "Repeat purchase within 90 days"
        },
    ]
)


display(
    experiment_arm_register
)

### Intervention Discipline

The predictive model determines **who may warrant intervention**.

It does not determine **which intervention will cause repeat purchase**.

The first controlled experiment should therefore use a clearly specified treatment.

The intervention should have:

- a defined start time after first purchase;
- consistent content;
- consistent contact frequency;
- known delivery cost;
- known incentive cost, if any;
- and a measurable exposure record.

Multiple changing treatments should not be combined into the first experiment because doing so would make the causal result difficult to interpret.

In [ ]:
# ================================================================
# 30.12 — EXPERIMENT PRIMARY OUTCOME
# ================================================================

primary_experiment_outcome = (
    "repeat_within_90d"
)

In [ ]:
# ================================================================
# 30.13 — EXPERIMENT OUTCOME REGISTER
# ================================================================

experiment_outcomes = pd.DataFrame(
    [
        {
            "Outcome":
                "90-day repeat purchase rate",

            "Role":
                "Primary",

            "Definition":
                "Customer completes another qualifying merchandise order within 90 days"
        },

        {
            "Outcome":
                "Days to second purchase",

            "Role":
                "Secondary",

            "Definition":
                "Elapsed days from first to second qualifying order"
        },

        {
            "Outcome":
                "90-day customer revenue",

            "Role":
                "Secondary",

            "Definition":
                "Net merchandise revenue generated during the 90-day follow-up window"
        },

        {
            "Outcome":
                "90-day order count",

            "Role":
                "Secondary",

            "Definition":
                "Number of qualifying orders during the follow-up period"
        },

        {
            "Outcome":
                "Incremental gross margin",

            "Role":
                "Commercial",

            "Definition":
                "Treatment-driven incremental margin where cost data are available"
        },

        {
            "Outcome":
                "Intervention cost",

            "Role":
                "Commercial",

            "Definition":
                "Communication, discount, incentive and operating cost"
        },

        {
            "Outcome":
                "Incremental ROI",

            "Role":
                "Commercial",

            "Definition":
                "Incremental commercial contribution relative to intervention cost"
        },
    ]
)


display(
    experiment_outcomes
)

In [ ]:
# ================================================================
# 30.14 — EXPERIMENT EFFECT FUNCTIONS
# ================================================================

def calculate_experiment_effects(
    treatment_rate,
    control_rate
):

    absolute_uplift = (
        treatment_rate
        -
        control_rate
    )


    if control_rate > 0:

        relative_uplift = (
            absolute_uplift
            /
            control_rate
        )

    else:

        relative_uplift = np.nan


    if absolute_uplift > 0:

        number_needed_to_treat = (
            1
            /
            absolute_uplift
        )

    else:

        number_needed_to_treat = np.nan


    return {
        "Treatment Repeat Rate":
            treatment_rate,

        "Control Repeat Rate":
            control_rate,

        "Absolute Uplift":
            absolute_uplift,

        "Relative Uplift":
            relative_uplift,

        "Number Needed to Treat":
            number_needed_to_treat,
    }

In [ ]:
illustrative_effect = (
    calculate_experiment_effects(
        treatment_rate=0.40,
        control_rate=0.35
    )
)


display(
    pd.DataFrame(
        [illustrative_effect]
    )
)

In [ ]:
%pip install statsmodels

In [ ]:
# ================================================================
# 30.16 — TWO-PROPORTION EXPERIMENT SAMPLE SIZE
# ================================================================

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

In [ ]:
def required_sample_size_per_arm(
    baseline_rate,
    target_rate,
    alpha=0.05,
    power=0.80
):
    """
    Approximate required sample size per experimental arm
    for comparing two proportions.
    """

    effect_size = (
        proportion_effectsize(
            baseline_rate,
            target_rate
        )
    )


    analysis = NormalIndPower()


    required_n = (
        analysis.solve_power(
            effect_size=effect_size,
            alpha=alpha,
            power=power,
            ratio=1.0,
            alternative="two-sided"
        )
    )


    return int(
        np.ceil(
            required_n
        )
    )

In [ ]:
# ================================================================
# 30.17 — PILOT SAMPLE-SIZE SCENARIOS
# ================================================================

illustrative_control_rate = 0.40


target_rates = [
    0.42,
    0.45,
    0.47,
    0.50,
]


sample_size_scenarios = []


for target_rate in target_rates:

    required_per_arm = (
        required_sample_size_per_arm(
            baseline_rate=illustrative_control_rate,
            target_rate=target_rate,
            alpha=0.05,
            power=0.80
        )
    )


    sample_size_scenarios.append(
        {
            "Assumed Control Repeat Rate":
                illustrative_control_rate,

            "Target Treatment Repeat Rate":
                target_rate,

            "Absolute Uplift (pp)":
                (
                    target_rate
                    -
                    illustrative_control_rate
                )
                * 100,

            "Required Customers per Arm":
                required_per_arm,

            "Approximate Total Sample":
                required_per_arm
                * 2,
        }
    )


sample_size_scenarios = pd.DataFrame(
    sample_size_scenarios
)


display(
    sample_size_scenarios
)

In [ ]:
# ================================================================
# 30.18 — COMMERCIAL BREAK-EVEN FUNCTION
# ================================================================

def campaign_economics(
    customers_treated,
    absolute_repeat_uplift,
    contribution_per_incremental_repeater,
    cost_per_treated_customer
):

    incremental_repeaters = (
        customers_treated
        *
        absolute_repeat_uplift
    )


    incremental_contribution = (
        incremental_repeaters
        *
        contribution_per_incremental_repeater
    )


    campaign_cost = (
        customers_treated
        *
        cost_per_treated_customer
    )


    incremental_profit = (
        incremental_contribution
        -
        campaign_cost
    )


    if campaign_cost > 0:

        roi = (
            incremental_profit
            /
            campaign_cost
        )

    else:

        roi = np.nan


    return pd.DataFrame(
        {
            "Customers Treated": [
                customers_treated
            ],

            "Incremental Repeaters": [
                incremental_repeaters
            ],

            "Incremental Contribution": [
                incremental_contribution
            ],

            "Campaign Cost": [
                campaign_cost
            ],

            "Incremental Profit": [
                incremental_profit
            ],

            "ROI": [
                roi
            ],
        }
    )

In [ ]:
# ================================================================
# 30.19 — BREAK-EVEN UPLIFT
# ================================================================

def break_even_repeat_uplift(
    cost_per_customer,
    contribution_per_incremental_repeater
):

    if (
        contribution_per_incremental_repeater
        <= 0
    ):
        return np.nan


    return (
        cost_per_customer
        /
        contribution_per_incremental_repeater
    )

In [ ]:
# ================================================================
# 30.20 — REPRODUCIBLE RANDOM ASSIGNMENT
# ================================================================

def assign_experiment_arm(
    data,
    random_state=42
):

    assigned = (
        data
        .sample(
            frac=1,
            random_state=random_state
        )
        .reset_index(
            drop=True
        )
        .copy()
    )


    assigned[
        "experiment_arm"
    ] = np.where(
        np.arange(
            len(
                assigned
            )
        )
        % 2
        == 0,
        "Treatment",
        "Control"
    )


    return assigned

In [ ]:
# ================================================================
# 30.21 — RANDOMISATION BALANCE CHECK
# ================================================================

def experiment_balance_summary(
    experiment_data,
    numeric_columns
):

    balance_rows = []


    for feature in numeric_columns:

        treatment_mean = (
            experiment_data.loc[
                experiment_data[
                    "experiment_arm"
                ] == "Treatment",
                feature
            ]
            .mean()
        )


        control_mean = (
            experiment_data.loc[
                experiment_data[
                    "experiment_arm"
                ] == "Control",
                feature
            ]
            .mean()
        )


        pooled_std = np.sqrt(
            (
                experiment_data.loc[
                    experiment_data[
                        "experiment_arm"
                    ] == "Treatment",
                    feature
                ]
                .var()
                +
                experiment_data.loc[
                    experiment_data[
                        "experiment_arm"
                    ] == "Control",
                    feature
                ]
                .var()
            )
            /
            2
        )


        if (
            pooled_std == 0
            or
            pd.isna(
                pooled_std
            )
        ):

            smd = np.nan

        else:

            smd = (
                treatment_mean
                -
                control_mean
            ) / pooled_std


        balance_rows.append(
            {
                "Feature":
                    feature,

                "Treatment Mean":
                    treatment_mean,

                "Control Mean":
                    control_mean,

                "SMD":
                    smd,

                "Absolute SMD":
                    abs(smd)
                    if not pd.isna(smd)
                    else np.nan,
            }
        )


    return pd.DataFrame(
        balance_rows
    )

In [ ]:
# ================================================================
# 30.22 — EXPERIMENT OUTCOME ANALYSIS
# ================================================================

def analyse_repeat_experiment(
    experiment_data,
    outcome_column="repeat_within_90d"
):

    arm_summary = (
        experiment_data
        .groupby(
            "experiment_arm",
            as_index=False
        )
        .agg(
            customers=(
                outcome_column,
                "size"
            ),

            repeaters=(
                outcome_column,
                "sum"
            ),

            repeat_rate=(
                outcome_column,
                "mean"
            ),
        )
    )


    treatment_rate = (
        arm_summary.loc[
            arm_summary[
                "experiment_arm"
            ] == "Treatment",
            "repeat_rate"
        ]
        .iloc[0]
    )


    control_rate = (
        arm_summary.loc[
            arm_summary[
                "experiment_arm"
            ] == "Control",
            "repeat_rate"
        ]
        .iloc[0]
    )


    effects = (
        calculate_experiment_effects(
            treatment_rate=treatment_rate,
            control_rate=control_rate
        )
    )


    return (
        arm_summary,
        pd.DataFrame(
            [effects]
        )
    )

### Recommended Production Scoring Flow

A future operational workflow should follow this sequence:

**1. First order completed**

A qualifying first customer order is recorded.

**2. Feature generation**

Only information available from that first order is used to construct the frozen model features.

**3. Model scoring**

The frozen Logistic Regression model generates the customer's repeat-propensity score.

**4. Propensity segmentation**

The score is mapped to fixed thresholds established from the model-development population.

**5. Experiment eligibility**

Customers within the predetermined intervention-eligible propensity groups enter the experiment.

**6. Randomisation**

Eligible customers are randomly assigned to Treatment or Control.

**7. Intervention**

Treatment customers receive the defined post-purchase retention intervention.

**8. Observation**

Customer behaviour is observed for the full 90-day outcome window.

**9. Causal evaluation**

Treatment and Control repeat rates, revenue and commercial contribution are compared.

**10. Deployment decision**

Only if the intervention demonstrates positive incremental commercial value should the model-supported strategy progress beyond pilot stage.

In [ ]:
# ================================================================
# 30.24 — MODEL MONITORING REGISTER
# ================================================================

model_monitoring_register = pd.DataFrame(
    [
        {
            "Monitoring Area":
                "Score distribution",

            "Metric":
                "Mean, median and percentile propensity scores",

            "Purpose":
                "Detect changes in customer scoring behaviour"
        },

        {
            "Monitoring Area":
                "Feature drift",

            "Metric":
                "SMD / PSI or comparable drift measures",

            "Purpose":
                "Detect changes in incoming customer characteristics"
        },

        {
            "Monitoring Area":
                "Outcome prevalence",

            "Metric":
                "Observed 90-day repeat rate",

            "Purpose":
                "Detect structural changes in customer behaviour"
        },

        {
            "Monitoring Area":
                "Discrimination",

            "Metric":
                "ROC-AUC and PR-AUC",

            "Purpose":
                "Check whether model ranking continues to work"
        },

        {
            "Monitoring Area":
                "Commercial ranking",

            "Metric":
                "Lift@20% and segment-level repeat rates",

            "Purpose":
                "Monitor operational usefulness"
        },

        {
            "Monitoring Area":
                "Calibration",

            "Metric":
                "Observed vs predicted rates",

            "Purpose":
                "Monitor probability accuracy"
        },

        {
            "Monitoring Area":
                "Experiment effectiveness",

            "Metric":
                "Treatment-control uplift",

            "Purpose":
                "Measure causal intervention impact"
        },

        {
            "Monitoring Area":
                "Economics",

            "Metric":
                "Incremental margin and ROI",

            "Purpose":
                "Verify that intervention remains financially worthwhile"
        },
    ]
)


display(
    model_monitoring_register
)

In [ ]:
# ================================================================
# 30.25 — MONITORING CADENCE
# ================================================================

monitoring_cadence = pd.DataFrame(
    [
        {
            "Frequency":
                "Weekly",

            "Review":
                "Scoring volume, missing fields, pipeline failures"
        },

        {
            "Frequency":
                "Monthly",

            "Review":
                "Score distribution, feature drift, customer mix"
        },

        {
            "Frequency":
                "Quarterly / once outcomes mature",

            "Review":
                "ROC-AUC, PR-AUC, lift, calibration"
        },

        {
            "Frequency":
                "After each experiment",

            "Review":
                "Treatment uplift, incremental revenue and ROI"
        },

        {
            "Frequency":
                "Before model redevelopment",

            "Review":
                "Full temporal backtest and new untouched holdout"
        },
    ]
)


display(
    monitoring_cadence
)

In [ ]:
# ================================================================
# 30.26 — FINAL MODEL LIMITATIONS
# ================================================================

model_limitations = pd.DataFrame(
    [
        {
            "Limitation":
                "Moderate discrimination",

            "Implication":
                "The model cannot precisely identify individual repeaters."
        },

        {
            "Limitation":
                "Top-decile lift did not generalise",

            "Implication":
                "A top-10%-only targeting strategy is not supported."
        },

        {
            "Limitation":
                "Temporal performance variation",

            "Implication":
                "Predictive relationships change across customer cohorts."
        },

        {
            "Limitation":
                "Observational predictors",

            "Implication":
                "Feature importance and coefficients are not causal effects."
        },

        {
            "Limitation":
                "Customer identification gaps in the source data",

            "Implication":
                "The model applies only to identified customers."
        },

        {
            "Limitation":
                "First-order-only feature design",

            "Implication":
                "Potentially useful CRM, demographic, marketing and behavioural variables are unavailable."
        },

        {
            "Limitation":
                "Historical retail dataset",

            "Implication":
                "Performance may differ materially in another business or future operating environment."
        },

        {
            "Limitation":
                "No treatment-effect modelling",

            "Implication":
                "The model predicts natural repeat propensity, not responsiveness to intervention."
        },

        {
            "Limitation":
                "No direct margin information",

            "Implication":
                "Commercial value cannot be established from revenue alone."
        },

        {
            "Limitation":
                "Limited annual history",

            "Implication":
                "Long-run seasonal and structural stability cannot be fully established."
        },
    ]
)


display(
    model_limitations
)

### Propensity Is Not Uplift

The model developed in this notebook is a **propensity model**.

It estimates which customers are naturally more or less likely to make another purchase.

It does not estimate which customers are most likely to change behaviour because of an intervention.

These are different questions.

A customer with low repeat propensity may:

- respond strongly to intervention;
- remain unlikely to return regardless of intervention;
- or even react negatively to excessive marketing.

Similarly, a customer with high natural repeat propensity may require no intervention at all.

If sufficient randomised experiment data become available, a future analytical stage could investigate **uplift modelling** or heterogeneous treatment-effect modelling.

Such a model would attempt to predict:

> **Which customers are most likely to make an additional purchase because they were treated?**

That would be a more direct optimisation model for retention spending.

In [ ]:
# ================================================================
# 30.28 — DEPLOYMENT MATURITY ROADMAP
# ================================================================

deployment_roadmap = pd.DataFrame(
    [
        {
            "Stage":
                1,

            "Phase":
                "Offline model development",

            "Status":
                "Complete",

            "Objective":
                "Establish predictive signal"
        },

        {
            "Stage":
                2,

            "Phase":
                "Untouched temporal validation",

            "Status":
                "Complete",

            "Objective":
                "Test future generalisation"
        },

        {
            "Stage":
                3,

            "Phase":
                "Controlled pilot",

            "Status":
                "Next",

            "Objective":
                "Measure causal intervention uplift"
        },

        {
            "Stage":
                4,

            "Phase":
                "Commercial economics",

            "Status":
                "Future",

            "Objective":
                "Establish incremental profit and ROI"
        },

        {
            "Stage":
                5,

            "Phase":
                "Monitored production deployment",

            "Status":
                "Future",

            "Objective":
                "Operationalise proven strategy"
        },

        {
            "Stage":
                6,

            "Phase":
                "Uplift modelling",

            "Status":
                "Future",

            "Objective":
                "Predict treatment responsiveness"
        },
    ]
)


display(
    deployment_roadmap
)

In [ ]:
# ================================================================
# 30.29 — FINAL NOTEBOOK MODEL CARD
# ================================================================

notebook2_final_model_card = pd.DataFrame(
    {
        "Field": [
            "Business problem",
            "Model target",
            "Prediction point",
            "Eligibility",
            "Algorithm",
            "Validation design",
            "Final ROC-AUC",
            "Final PR-AUC",
            "Final Lift@10%",
            "Final Lift@20%",
            "Final calibration error",
            "Final decision",
            "Recommended use",
            "Prohibited interpretation",
            "Next analytical stage",
        ],

        "Final Position": [
            "Predict 90-day customer repeat purchase",

            "Second observable qualifying order within 90 days",

            "Immediately after first observable order",

            "Identified observable acquisitions with complete 90-day follow-up",

            "Drop-First Interpretable Logistic Regression",

            "Chronological Train → Validation → untouched future Test",

            "Approximately 0.560",

            "Approximately 0.503",

            "Approximately 1.03×",

            "Approximately 1.11×",

            "Approximately 4.71 percentage points",

            "CONDITIONAL GO",

            (
                "Prioritisation layer for controlled retention experiments, "
                "especially lower-propensity customers"
            ),

            (
                "Not causal, not guaranteed individual probability, "
                "not permanent-churn classification"
            ),

            "Randomised retention experiment / uplift measurement",
        ],
    }
)


display(
    notebook2_final_model_card
)

In [ ]:
# ================================================================
# 30.30 — NOTEBOOK 2 FINAL VALIDATION
# ================================================================

assert (
    MODEL_SPECIFICATION_FROZEN
    is True
)


assert (
    TEST_SET_OPENED
    is True
)


assert (
    TEST_SET_CONSUMED
    is True
)


assert (
    final_model_decision
    ==
    "CONDITIONAL GO"
)


assert (
    frozen_model_name
    ==
    "Drop-First Interpretable Logistic"
)


print(
    "✓ Predictive modelling complete."
)

print(
    "✓ Frozen model evaluated on untouched temporal holdout."
)

print(
    "✓ Final decision: CONDITIONAL GO."
)

print(
    "✓ Model is suitable for controlled experimentation, "
    "not unrestricted deployment."
)

print(
    "✓ Test population permanently retired from model development."
)

# Final Notebook Conclusion

## 90-Day Repeat Purchase Propensity Model

This notebook converted the customer-retention findings from the exploratory analysis into a formally defined machine-learning problem.

The objective was to predict, immediately after a customer's first observable qualifying purchase, whether that customer would make a second purchase within 90 days.

A leakage-safe modelling population was constructed using only information available at the first order.

Customers without a complete 90-day future observation window were excluded from binary target construction, and customers first observed during the opening month of the dataset were excluded because their earlier purchase history could not be observed reliably.

### Model Development

Several predictive approaches were evaluated.

A naive prevalence baseline provided no customer-level discrimination.

A single-feature basket-breadth model produced modest predictive improvement.

The multivariable Logistic Regression model improved further and outperformed Random Forest, Gradient Boosting and HistGradientBoosting on the principal validation metrics.

Additional diagnostics tested:

- simplified feature specifications;
- log transformations;
- categorical reference encoding;
- calendar feature encoding;
- temporal stability;
- population drift;
- and probability calibration.

The **Drop-First Interpretable Logistic Regression** was ultimately selected and frozen because it provided the strongest overall balance of discrimination, lift, probability quality, interpretability and implementation simplicity.

### Final Out-of-Time Performance

The frozen specification was refitted on the complete Train + Validation development population and evaluated once on the previously untouched future test population.

Final test performance was approximately:

- **PR-AUC: 0.503**
- **ROC-AUC: 0.560**
- **Brier score: 0.248**
- **Lift@10%: 1.03×**
- **Lift@20%: 1.11×**
- **Capture@10%: 10.37%**
- **Capture@20%: 22.26%**

The final test demonstrated genuine but modest predictive discrimination.

ROC-AUC remained above random performance, but the stronger top-decile lift observed during model development did not persist in the future holdout population.

Bootstrap uncertainty also showed that the confidence intervals for top-10% and top-20% lift included 1.0.

The retrospective evidence therefore does not justify unrestricted operational deployment.

### Final Decision — CONDITIONAL GO

The model received a final decision of:

> **CONDITIONAL GO**

This means that the model contains sufficient future-customer signal to justify a controlled commercial experiment, but not enough evidence to justify automated treatment decisions or claims of proven commercial value.

The model should initially be used as a customer-prioritisation mechanism rather than an autonomous decision engine.

### Recommended Commercial Strategy

The most promising initial application is not simply targeting the customers with the highest predicted probability of natural repeat purchase.

The final test suggests that the model may be more useful in identifying customers with relatively low repeat propensity.

These customers can be placed into a controlled retention experiment.

Within that model-selected population, eligible customers should be randomly assigned to:

- **Treatment — targeted post-purchase intervention**
- **Control — business as usual**

The primary outcome should remain repeat purchase within 90 days.

Secondary outcomes should include:

- time to second purchase;
- 90-day revenue;
- order count;
- incremental gross margin;
- intervention cost;
- and incremental ROI.

### Prediction Is Not Causation

The model predicts natural customer behaviour.

It does not identify which customers will change behaviour because of intervention.

Therefore, the next commercial question is not:

> **Who will repeat?**

It is:

> **Whose repeat behaviour can we profitably change?**

That requires randomised experimentation.

If sufficient treatment-control data are accumulated, the natural next evolution would be uplift modelling or heterogeneous treatment-effect modelling, where the prediction target becomes incremental response to treatment rather than natural repeat propensity.

### Final Analytical Position

The modelling exercise succeeded in demonstrating that first-order purchase behaviour contains measurable information about subsequent repeat purchasing.

However, the final temporal holdout also demonstrated the importance of honest out-of-time evaluation.

A model that appeared commercially stronger during validation became materially more modest when evaluated against genuinely future customers.

This is not a modelling failure.

It is precisely what an untouched temporal holdout is designed to reveal.

The model should therefore move forward cautiously as an experimental decision-support tool, with future deployment determined by measured incremental business value rather than retrospective predictive performance alone.

In [ ]:
# ================================================================
# EXPORT AUDIT — REPEAT PURCHASE NOTEBOOK
# ================================================================

repeat_export_candidates = []


for name, obj in list(
    globals().items()
):

    if not isinstance(
        obj,
        pd.DataFrame
    ):
        continue


    columns_lower = {
        str(column).lower()
        for column in obj.columns
    }


    has_customer = any(
        term in columns_lower
        for term in [
            "customer",
            "customer_id",
        ]
    )


    probability_like_columns = [
        column
        for column in obj.columns
        if any(
            term
            in str(column).lower()
            for term in [
                "prob",
                "prediction",
                "score",
                "pred",
            ]
        )
    ]


    if (
        has_customer
        or
        probability_like_columns
    ):

        repeat_export_candidates.append(
            {
                "DataFrame":
                    name,

                "Rows":
                    len(obj),

                "Columns":
                    list(
                        obj.columns
                    ),

                "Probability / Score Columns":
                    probability_like_columns,
            }
        )


repeat_export_candidates = pd.DataFrame(
    repeat_export_candidates
)


display(
    repeat_export_candidates
)

In [ ]:
# ================================================================
# EXPORT AUDIT 2 — REPEAT SCORE TABLES
# ================================================================

repeat_score_tables = []


for name, obj in list(globals().items()):

    if not isinstance(
        obj,
        pd.DataFrame
    ):
        continue


    columns = list(
        obj.columns
    )


    columns_lower = [
        str(column).lower()
        for column in columns
    ]


    has_customer = (
        "customer"
        in
        columns_lower
    )


    has_repeat_score = any(
        (
            "repeat"
            in column
            and
            (
                "score"
                in column
                or
                "prob"
                in column
                or
                "pred"
                in column
            )
        )
        for column in columns_lower
    )


    if (
        has_customer
        and
        has_repeat_score
    ):

        repeat_score_tables.append(
            {
                "DataFrame":
                    name,

                "Rows":
                    len(obj),

                "Columns":
                    columns,
            }
        )


repeat_score_tables = pd.DataFrame(
    repeat_score_tables
)


display(
    repeat_score_tables
)

In [ ]:
# ================================================================
# EXPORT AUDIT 3 — DEVELOPMENT SCORED STRUCTURE
# ================================================================

print(
    "development_scored shape:",
    development_scored.shape
)

print(
    "\nColumns:"
)

for column in development_scored.columns:

    print(
        " -",
        column
    )


display(
    development_scored.head(
        10
    )
)

In [ ]:
# ================================================================
# EXPORT — REPEAT PURCHASE MODEL OUTPUT
# ================================================================

from pathlib import Path

MODEL_OUTPUT_DIR = Path("model_outputs")

MODEL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


repeat_purchase_scores = (
    development_scored[
        [
            "customer",
            "first_order_date",
            "repeat_propensity_score",
            "propensity_segment",
        ]
    ]
    .copy()
)


# ------------------------------------------------
# Canonical column names for Notebook 8
# ------------------------------------------------

repeat_purchase_scores = (
    repeat_purchase_scores
    .rename(
        columns={
            "first_order_date":
                "repeat_score_date",

            "repeat_propensity_score":
                "repeat_probability",

            "propensity_segment":
                "repeat_propensity_segment",
        }
    )
)


# ------------------------------------------------
# Explicit model provenance
# ------------------------------------------------

repeat_purchase_scores[
    "repeat_model_scope"
] = (
    "90-day repeat propensity after first observed order"
)


repeat_purchase_scores[
    "repeat_score_provenance"
] = (
    "historical development scoring population"
)


# ------------------------------------------------
# Customer ID standardisation
# ------------------------------------------------

repeat_purchase_scores[
    "customer"
] = (
    pd.to_numeric(
        repeat_purchase_scores[
            "customer"
        ],
        errors="coerce"
    )
    .astype("Int64")
    .astype("string")
)


repeat_purchase_scores[
    "repeat_score_date"
] = pd.to_datetime(
    repeat_purchase_scores[
        "repeat_score_date"
    ]
)


# ------------------------------------------------
# Safety checks
# ------------------------------------------------

assert (
    repeat_purchase_scores[
        "customer"
    ]
    .notna()
    .all()
)


assert (
    repeat_purchase_scores[
        "repeat_probability"
    ]
    .between(
        0,
        1
    )
    .all()
)


assert (
    repeat_purchase_scores[
        "customer"
    ]
    .is_unique
), (
    "Expected one Repeat Purchase score per customer."
)


assert (
    repeat_purchase_scores[
        "repeat_score_date"
    ]
    .notna()
    .all()
)


# ------------------------------------------------
# Save persistent output
# ------------------------------------------------

repeat_output_path = (
    MODEL_OUTPUT_DIR
    /
    "repeat_purchase_scores.parquet"
)


repeat_purchase_scores.to_parquet(
    repeat_output_path,
    index=False
)


print(
    f"✓ Repeat Purchase scores exported:"
)

print(
    repeat_output_path.resolve()
)

print(
    f"Rows: {len(repeat_purchase_scores):,}"
)

print(
    f"Customers: "
    f"{repeat_purchase_scores['customer'].nunique():,}"
)

print(
    f"Score date range: "
    f"{repeat_purchase_scores['repeat_score_date'].min()} "
    f"to "
    f"{repeat_purchase_scores['repeat_score_date'].max()}"
)


display(
    repeat_purchase_scores.head(
        10
    )
)